In [2]:
import pandas as pd
import requests
from io import StringIO

url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"

headers = {
    "User-Agent": "Mozilla/5.0"
}

response = requests.get(url, headers=headers, timeout=30)
response.raise_for_status()

tables = pd.read_html(StringIO(response.text))

sp500 = tables[0]

print("Total S&P 500 rows:", len(sp500))
print(sp500.columns.tolist())

sp500.head()

Total S&P 500 rows: 503
['Symbol', 'Security', 'GICS Sector', 'GICS Sub-Industry', 'Headquarters Location', 'Date added', 'CIK', 'Founded']


,Symbol,Security,GICS Sector,GICS Sub-Industry,Headquarters Location,Date added,CIK,Founded
0,MMM,3M,Industrials,Industrial Conglomerates,"Saint Paul, Minnesota",1957-03-04,66740,1902
1,AOS,A. O. Smith,Industrials,Building Products,"Milwaukee, Wisconsin",2017-07-26,91142,1916
2,ABT,Abbott Laboratories,Health Care,Health Care Equipment,"North Chicago, Illinois",1957-03-04,1800,1888
3,ABBV,AbbVie,Health Care,Biotechnology,"North Chicago, Illinois",2012-12-31,1551152,2013 (1888)
4,ACN,Accenture,Information Technology,IT Consulting & Other Services,"Dublin, Ireland",2011-07-06,1467373,1989


In [3]:
target_sectors = [
    "Information Technology",
    "Communication Services"
]

company_universe = sp500[
    sp500["GICS Sector"].isin(target_sectors)
].copy()

company_universe = company_universe[
    [
        "Symbol",
        "Security",
        "CIK",
        "GICS Sector",
        "GICS Sub-Industry"
    ]
]

company_universe = company_universe.rename(
    columns={
        "Symbol": "ticker",
        "Security": "company",
        "CIK": "cik",
        "GICS Sector": "sector",
        "GICS Sub-Industry": "industry"
    }
)

company_universe = company_universe.reset_index(drop=True)

print("Companies in initial universe:", len(company_universe))

company_universe.head(10)

Companies in initial universe: 97


,ticker,company,cik,sector,industry
0,ACN,Accenture,1467373,Information Technology,IT Consulting & Other Services
1,ADBE,Adobe Inc.,796343,Information Technology,Application Software
2,AMD,Advanced Micro Devices,2488,Information Technology,Semiconductors
3,AKAM,Akamai Technologies,1086222,Information Technology,Internet Services & Infrastructure
4,GOOGL,Alphabet Inc. (Class A),1652044,Communication Services,Interactive Media & Services
5,GOOG,Alphabet Inc. (Class C),1652044,Communication Services,Interactive Media & Services
6,APH,Amphenol,820313,Information Technology,Electronic Components
7,ADI,Analog Devices,6281,Information Technology,Semiconductors
8,AAPL,Apple Inc.,320193,Information Technology,"Technology Hardware, Storage & Peripherals"
9,AMAT,Applied Materials,6951,Information Technology,Semiconductor Materials & Equipment


In [4]:
print("Number of companies:", len(company_universe))
print()

print("Companies by sector:")
print(company_universe["sector"].value_counts())
print()

print("Missing values:")
print(company_universe.isna().sum())
print()

print("Duplicate tickers:", company_universe["ticker"].duplicated().sum())

Number of companies: 97

Companies by sector:
sector
Information Technology    75
Communication Services    22
Name: count, dtype: int64

Missing values:
ticker      0
company     0
cik         0
sector      0
industry    0
dtype: int64

Duplicate tickers: 0


In [5]:
company_universe.to_csv(
    "../data/raw/company_universe_initial.csv",
    index=False
)

print("Saved company universe.")

Saved company universe.


In [1]:
import pandas as pd
import numpy as np
import requests

print("Project environment is working.")
print("Pandas version:", pd.__version__)

Project environment is working.
Pandas version: 3.0.6


In [6]:
import requests

apple_cik = "0000320193"

sec_url = (
    f"https://data.sec.gov/api/xbrl/companyfacts/"
    f"CIK{apple_cik}.json"
)

headers = {
    "User-Agent": "AI Valuation Research mihirjpai@gmail.com"
}

response = requests.get(
    sec_url,
    headers=headers,
    timeout=30
)

print("Status code:", response.status_code)

Status code: 200


In [7]:
apple_facts = response.json()

print("Company:", apple_facts["entityName"])
print("CIK:", apple_facts["cik"])

print("Top-level keys:")
print(apple_facts.keys())

Company: Apple Inc.
CIK: 320193
Top-level keys:
dict_keys(['cik', 'entityName', 'facts'])


In [8]:
us_gaap_facts = apple_facts["facts"]["us-gaap"]

print("Number of US-GAAP concepts:", len(us_gaap_facts))

print("First 20 concept names:")
print(list(us_gaap_facts.keys())[:20])

Number of US-GAAP concepts: 512
First 20 concept names:
['AccountsPayable', 'AccountsPayableCurrent', 'AccountsReceivableNetCurrent', 'AccruedIncomeTaxesCurrent', 'AccruedIncomeTaxesNoncurrent', 'AccruedLiabilities', 'AccruedLiabilitiesCurrent', 'AccruedMarketingCostsCurrent', 'AccumulatedDepreciationDepletionAndAmortizationPropertyPlantAndEquipment', 'AccumulatedOtherComprehensiveIncomeLossAvailableForSaleSecuritiesAdjustmentNetOfTax', 'AccumulatedOtherComprehensiveIncomeLossCumulativeChangesInNetGainLossFromCashFlowHedgesEffectNetOfTax', 'AccumulatedOtherComprehensiveIncomeLossForeignCurrencyTranslationAdjustmentNetOfTax', 'AccumulatedOtherComprehensiveIncomeLossNetOfTax', 'AdjustmentsToAdditionalPaidInCapitalSharebasedCompensationRequisiteServicePeriodRecognitionValue', 'AdjustmentsToAdditionalPaidInCapitalTaxEffectFromShareBasedCompensation', 'AdvertisingExpense', 'AllocatedShareBasedCompensationExpense', 'AllowanceForDoubtfulAccountsReceivableCurrent', 'AmortizationOfIntangibleAss

In [9]:
revenue_fact = us_gaap_facts["RevenueFromContractWithCustomerExcludingAssessedTax"]

print(revenue_fact.keys())
print()
print(revenue_fact["label"])
print(revenue_fact["description"])

dict_keys(['label', 'description', 'units'])

Revenue from Contract with Customer, Excluding Assessed Tax
Amount, excluding tax collected from customer, of revenue from satisfaction of performance obligation by transferring promised good or service to customer. Tax collected from customer is tax assessed by governmental authority that is both imposed on and concurrent with specific revenue-producing transaction, including, but not limited to, sales, use, value added and excise.


In [10]:
revenue_units = revenue_fact["units"]["USD"]

print("Number of reported revenue facts:", len(revenue_units))

revenue_units[-10:]

Number of reported revenue facts: 120


[{'start': '2025-03-30',
  'end': '2025-06-28',
  'val': 94036000000,
  'accn': '0000320193-25-000073',
  'fy': 2025,
  'fp': 'Q3',
  'form': '10-Q',
  'filed': '2025-08-01'},
 {'start': '2025-03-30',
  'end': '2025-06-28',
  'val': 94036000000,
  'accn': '0000320193-26-000020',
  'fy': 2026,
  'fp': 'Q3',
  'form': '10-Q',
  'filed': '2026-07-31',
  'frame': 'CY2025Q2'},
 {'start': '2024-09-29',
  'end': '2025-09-27',
  'val': 416161000000,
  'accn': '0000320193-25-000079',
  'fy': 2025,
  'fp': 'FY',
  'form': '10-K',
  'filed': '2025-10-31'},
 {'start': '2024-09-29',
  'end': '2025-09-27',
  'val': 416161000000,
  'accn': '0000320193-26-000026',
  'fy': 2026,
  'fp': 'FY',
  'form': '10-K',
  'filed': '2026-11-03',
  'frame': 'CY2025'},
 {'start': '2025-09-28',
  'end': '2025-12-27',
  'val': 143756000000,
  'accn': '0000320193-26-000006',
  'fy': 2026,
  'fp': 'Q1',
  'form': '10-Q',
  'filed': '2026-01-30',
  'frame': 'CY2025Q4'},
 {'start': '2025-09-28',
  'end': '2026-03-28',
  

In [11]:
revenue_df = pd.DataFrame(revenue_units)

revenue_df[
    ["start", "end", "val", "fy", "fp", "form", "filed"]
].tail(15)

,start,end,val,fy,fp,form,filed
105,2024-09-29,2025-03-29,219659000000,2026,Q2,10-Q,2026-05-01
106,2024-12-29,2025-03-29,95359000000,2025,Q2,10-Q,2025-05-02
107,2024-12-29,2025-03-29,95359000000,2026,Q2,10-Q,2026-05-01
108,2024-09-29,2025-06-28,313695000000,2025,Q3,10-Q,2025-08-01
109,2024-09-29,2025-06-28,313695000000,2026,Q3,10-Q,2026-07-31
110,2025-03-30,2025-06-28,94036000000,2025,Q3,10-Q,2025-08-01
111,2025-03-30,2025-06-28,94036000000,2026,Q3,10-Q,2026-07-31
112,2024-09-29,2025-09-27,416161000000,2025,FY,10-K,2025-10-31
113,2024-09-29,2025-09-27,416161000000,2026,FY,10-K,2026-11-03
114,2025-09-28,2025-12-27,143756000000,2026,Q1,10-Q,2026-01-30


In [12]:
annual_revenue = revenue_df[
    (revenue_df["form"] == "10-K") &
    (revenue_df["fp"] == "FY")
].copy()

annual_revenue[
    ["start", "end", "val", "fy", "form", "filed"]
].tail(15)

,start,end,val,fy,form,filed
60,2020-09-27,2021-09-25,365817000000,2021,10-K,2021-10-29
61,2020-09-27,2021-09-25,365817000000,2022,10-K,2022-10-28
62,2020-09-27,2021-09-25,365817000000,2023,10-K,2023-11-03
73,2021-09-26,2022-09-24,394328000000,2022,10-K,2022-10-28
74,2021-09-26,2022-09-24,394328000000,2023,10-K,2023-11-03
75,2021-09-26,2022-09-24,394328000000,2024,10-K,2024-11-01
86,2022-09-25,2023-09-30,383285000000,2023,10-K,2023-11-03
87,2022-09-25,2023-09-30,383285000000,2024,10-K,2024-11-01
88,2022-09-25,2023-09-30,383285000000,2025,10-K,2025-10-31
99,2023-10-01,2024-09-28,391035000000,2024,10-K,2024-11-01


In [13]:
annual_revenue_clean = (
    annual_revenue
    .sort_values("filed")
    .drop_duplicates(
        subset=["start", "end", "val"],
        keep="first"
    )
    .copy()
)

annual_revenue_clean[
    ["start", "end", "val", "fy", "form", "filed"]
].tail(10)

,start,end,val,fy,form,filed
46,2019-09-29,2020-09-26,274515000000,2020,10-K,2020-10-30
44,2020-03-29,2020-06-27,59685000000,2020,10-K,2020-10-30
34,2019-09-29,2019-12-28,91819000000,2020,10-K,2020-10-30
39,2019-12-29,2020-03-28,58313000000,2020,10-K,2020-10-30
60,2020-09-27,2021-09-25,365817000000,2021,10-K,2021-10-29
73,2021-09-26,2022-09-24,394328000000,2022,10-K,2022-10-28
86,2022-09-25,2023-09-30,383285000000,2023,10-K,2023-11-03
99,2023-10-01,2024-09-28,391035000000,2024,10-K,2024-11-01
112,2024-09-29,2025-09-27,416161000000,2025,10-K,2025-10-31
119,2025-09-28,2026-09-26,478834000000,2026,10-K,2026-11-03


In [14]:
annual_revenue_clean["start"] = pd.to_datetime(
    annual_revenue_clean["start"]
)

annual_revenue_clean["end"] = pd.to_datetime(
    annual_revenue_clean["end"]
)

annual_revenue_clean["days"] = (
    annual_revenue_clean["end"]
    - annual_revenue_clean["start"]
).dt.days

full_year_revenue = annual_revenue_clean[
    annual_revenue_clean["days"].between(300, 400)
].copy()

full_year_revenue[
    ["start", "end", "days", "val", "fy", "filed"]
].tail(10)

,start,end,days,val,fy,filed
0,2016-09-25,2017-09-30,370,229234000000,2019,2019-10-31
9,2017-10-01,2018-09-29,363,265595000000,2019,2019-10-31
28,2018-09-30,2019-09-28,363,260174000000,2019,2019-10-31
46,2019-09-29,2020-09-26,363,274515000000,2020,2020-10-30
60,2020-09-27,2021-09-25,363,365817000000,2021,2021-10-29
73,2021-09-26,2022-09-24,363,394328000000,2022,2022-10-28
86,2022-09-25,2023-09-30,370,383285000000,2023,2023-11-03
99,2023-10-01,2024-09-28,363,391035000000,2024,2024-11-01
112,2024-09-29,2025-09-27,363,416161000000,2025,2025-10-31
119,2025-09-28,2026-09-26,363,478834000000,2026,2026-11-03


In [15]:
apple_revenue_2021_2025 = full_year_revenue[
    full_year_revenue["fy"].between(2021, 2025)
].copy()

apple_revenue_2021_2025 = apple_revenue_2021_2025[
    ["fy", "start", "end", "val", "filed"]
].rename(
    columns={
        "fy": "fiscal_year",
        "val": "revenue"
    }
)

apple_revenue_2021_2025

,fiscal_year,start,end,revenue,filed
60,2021,2020-09-27,2021-09-25,365817000000,2021-10-29
73,2022,2021-09-26,2022-09-24,394328000000,2022-10-28
86,2023,2022-09-25,2023-09-30,383285000000,2023-11-03
99,2024,2023-10-01,2024-09-28,391035000000,2024-11-01
112,2025,2024-09-29,2025-09-27,416161000000,2025-10-31


In [16]:
print("Rows:", len(apple_revenue_2021_2025))
print()
print(apple_revenue_2021_2025[["fiscal_year", "revenue"]])

Rows: 5

     fiscal_year       revenue
60          2021  365817000000
73          2022  394328000000
86          2023  383285000000
99          2024  391035000000
112         2025  416161000000


In [189]:
def extract_annual_fact(
    us_gaap_facts,
    fact_name,
    start_year=2021,
    end_year=2025
):
    fact = us_gaap_facts[fact_name]
    units = fact["units"]["USD"]
    df = pd.DataFrame(units)

    df = df[
        (df["form"] == "10-K") &
        (df["fp"] == "FY")
    ].copy()

    df["start"] = pd.to_datetime(df["start"])
    df["end"] = pd.to_datetime(df["end"])

    df["days"] = (df["end"] - df["start"]).dt.days

    df = df[
        df["days"].between(300, 400)
    ].copy()

    # Keep the first-filed observation
    # for each actual annual reporting period
    df = (
        df
        .sort_values("filed")
        .drop_duplicates(
            subset=["end"],
            keep="first"
        )
    )

    # Earliest filing's SEC fiscal-year label
    # identifies the company's actual fiscal year
    df["fiscal_year"] = df["end"].dt.year

    early_january = (
        (df["end"].dt.month == 1) &
        (df["end"].dt.day <= 7)
    )

    df.loc[
        early_january,
        "fiscal_year"
    ] = df.loc[
        early_january,
        "fiscal_year"
    ] - 1

    df = df[
        df["fiscal_year"].between(
            start_year,
            end_year
        )
    ].copy()

    return df[
        [
            "fiscal_year",
            "start",
            "end",
            "val",
            "filed"
        ]
    ].reset_index(drop=True)

In [18]:
test_revenue = extract_annual_fact(
    us_gaap_facts,
    "RevenueFromContractWithCustomerExcludingAssessedTax"
)

test_revenue

,fy,start,end,val,filed
0,2021,2020-09-27,2021-09-25,365817000000,2021-10-29
1,2022,2021-09-26,2022-09-24,394328000000,2022-10-28
2,2023,2022-09-25,2023-09-30,383285000000,2023-11-03
3,2024,2023-10-01,2024-09-28,391035000000,2024-11-01
4,2025,2024-09-29,2025-09-27,416161000000,2025-10-31


In [19]:
rd_test = extract_annual_fact(
    us_gaap_facts,
    "ResearchAndDevelopmentExpense"
)

rd_test

,fy,start,end,val,filed
0,2021.0,2020-09-27,2021-09-25,21914000000,2021-10-29
1,2022.0,2021-09-26,2022-09-24,26251000000,2022-10-28
2,2023.0,2022-09-25,2023-09-30,29915000000,2023-11-03
3,2024.0,2023-10-01,2024-09-28,31370000000,2024-11-01
4,2025.0,2024-09-29,2025-09-27,34550000000,2025-10-31


In [20]:
print("Rows:", len(rd_test))
print()
print(rd_test[["fy", "val"]])

Rows: 5

       fy          val
0  2021.0  21914000000
1  2022.0  26251000000
2  2023.0  29915000000
3  2024.0  31370000000
4  2025.0  34550000000


In [21]:
operating_income_test = extract_annual_fact(
    us_gaap_facts,
    "OperatingIncomeLoss"
)

print("Rows:", len(operating_income_test))
print()
print(operating_income_test[["fy", "val"]])

Rows: 5

       fy           val
0  2021.0  108949000000
1  2022.0  119437000000
2  2023.0  114301000000
3  2024.0  123216000000
4  2025.0  133050000000


In [22]:
capex_candidates = [
    key for key in us_gaap_facts.keys()
    if "PropertyPlantAndEquipment" in key
    or "CapitalExpenditure" in key
    or "PaymentsToAcquire" in key
]

capex_candidates

['AccumulatedDepreciationDepletionAndAmortizationPropertyPlantAndEquipment',
 'CapitalExpendituresIncurredButNotYetPaid',
 'DeferredTaxAssetsPropertyPlantAndEquipment',
 'PaymentsToAcquireAvailableForSaleSecurities',
 'PaymentsToAcquireAvailableForSaleSecuritiesDebt',
 'PaymentsToAcquireBusinessesNetOfCashAcquired',
 'PaymentsToAcquireIntangibleAssets',
 'PaymentsToAcquireOtherInvestments',
 'PaymentsToAcquireProductiveAssets',
 'PaymentsToAcquirePropertyPlantAndEquipment',
 'PropertyPlantAndEquipmentGross',
 'PropertyPlantAndEquipmentNet',
 'DeferredTaxLiabilitiesPropertyPlantAndEquipment']

In [23]:
capex_test = extract_annual_fact(
    us_gaap_facts,
    "PaymentsToAcquirePropertyPlantAndEquipment"
)

print("Rows:", len(capex_test))
print()
print(capex_test[["fy", "val"]])

Rows: 5

     fy          val
0  2021  11085000000
1  2022  10708000000
2  2023  10959000000
3  2024   9447000000
4  2025  12715000000


In [24]:
apple_core = (
    test_revenue[["fy", "val"]]
    .rename(columns={"val": "revenue"})
    .merge(
        rd_test[["fy", "val"]].rename(columns={"val": "rd_expense"}),
        on="fy",
        how="outer"
    )
    .merge(
        operating_income_test[["fy", "val"]].rename(columns={"val": "operating_income"}),
        on="fy",
        how="outer"
    )
    .merge(
        capex_test[["fy", "val"]].rename(columns={"val": "capex"}),
        on="fy",
        how="outer"
    )
)

apple_core["fy"] = apple_core["fy"].astype(int)

apple_core

,fy,revenue,rd_expense,operating_income,capex
0,2021,365817000000,21914000000,108949000000,11085000000
1,2022,394328000000,26251000000,119437000000,10708000000
2,2023,383285000000,29915000000,114301000000,10959000000
3,2024,391035000000,31370000000,123216000000,9447000000
4,2025,416161000000,34550000000,133050000000,12715000000


In [25]:
print(apple_core)
print()
print("Missing values:")
print(apple_core.isna().sum())

     fy       revenue   rd_expense  operating_income        capex
0  2021  365817000000  21914000000      108949000000  11085000000
1  2022  394328000000  26251000000      119437000000  10708000000
2  2023  383285000000  29915000000      114301000000  10959000000
3  2024  391035000000  31370000000      123216000000   9447000000
4  2025  416161000000  34550000000      133050000000  12715000000

Missing values:
fy                  0
revenue             0
rd_expense          0
operating_income    0
capex               0
dtype: int64


In [26]:
apple_core["rd_intensity"] = (
    apple_core["rd_expense"] / apple_core["revenue"]
)

apple_core["capex_intensity"] = (
    apple_core["capex"] / apple_core["revenue"]
)

apple_core["operating_margin"] = (
    apple_core["operating_income"] / apple_core["revenue"]
)

apple_core[
    [
        "fy",
        "revenue",
        "rd_intensity",
        "capex_intensity",
        "operating_margin"
    ]
]

,fy,revenue,rd_intensity,capex_intensity,operating_margin
0,2021,365817000000,0.059904,0.030302,0.297824
1,2022,394328000000,0.066571,0.027155,0.302887
2,2023,383285000000,0.078049,0.028592,0.298214
3,2024,391035000000,0.080223,0.024159,0.315102
4,2025,416161000000,0.083021,0.030553,0.319708


In [27]:
apple_core[
    [
        "fy",
        "rd_intensity",
        "capex_intensity",
        "operating_margin"
    ]
].round(4)

,fy,rd_intensity,capex_intensity,operating_margin
0,2021,0.0599,0.0303,0.2978
1,2022,0.0666,0.0272,0.3029
2,2023,0.0780,0.0286,0.2982
3,2024,0.0802,0.0242,0.3151
4,2025,0.0830,0.0306,0.3197


In [28]:
apple_core = apple_core.sort_values("fy").copy()

apple_core["revenue_growth"] = (
    apple_core["revenue"]
    .pct_change()
)

apple_core[
    [
        "fy",
        "revenue",
        "revenue_growth"
    ]
].round(4)

,fy,revenue,revenue_growth
0,2021,365817000000,NaN
1,2022,394328000000,0.0779
2,2023,383285000000,-0.0280
3,2024,391035000000,0.0202
4,2025,416161000000,0.0643


In [29]:
assets_fact = us_gaap_facts["Assets"]

print(assets_fact["label"])
print(assets_fact["description"])
print()
print(assets_fact["units"].keys())

Assets
Sum of the carrying amounts as of the balance sheet date of all assets that are recognized. Assets are probable future economic benefits obtained or controlled by an entity as a result of past transactions or events.

dict_keys(['USD'])


In [30]:
assets_df = pd.DataFrame(
    assets_fact["units"]["USD"]
)

assets_df[
    ["end", "val", "fy", "fp", "form", "filed"]
].tail(15)

,end,val,fy,fp,form,filed
133,2024-09-28,364980000000,2025.0,Q2,10-Q,2025-05-02
134,2024-09-28,364980000000,2025.0,Q3,10-Q,2025-08-01
135,2024-09-28,364980000000,2025.0,FY,10-K,2025-10-31
136,2024-12-28,344085000000,2025.0,Q1,10-Q,2025-01-31
137,2025-03-29,331233000000,2025.0,Q2,10-Q,2025-05-02
138,2025-06-28,331495000000,2025.0,Q3,10-Q,2025-08-01
139,2025-09-27,359241000000,2025.0,FY,10-K,2025-10-31
140,2025-09-27,359241000000,2026.0,Q1,10-Q,2026-01-30
141,2025-09-27,359241000000,2026.0,Q2,10-Q,2026-05-01
142,2025-09-27,359241000000,2026.0,Q3,10-Q,2026-07-31


In [186]:
def extract_point_in_time_fact(
    us_gaap_facts,
    fact_name,
    start_year=2021,
    end_year=2025
):
    fact = us_gaap_facts[fact_name]
    units = fact["units"]["USD"]
    df = pd.DataFrame(units)

    df = df[
        (df["form"] == "10-K") &
        (df["fp"] == "FY")
    ].copy()

    df["end"] = pd.to_datetime(df["end"])

    # Keep earliest-filed observation for each exact date
    df = (
        df
        .sort_values("filed")
        .drop_duplicates(
            subset=["end"],
            keep="first"
        )
    )

    # Calendar-year mapping
    df["fiscal_year"] = df["end"].dt.year

    # Early-January fiscal year ends belong to prior year
    early_january = (
        (df["end"].dt.month == 1) &
        (df["end"].dt.day <= 7)
    )

    df.loc[
        early_january,
        "fiscal_year"
    ] = (
        df.loc[
            early_january,
            "fiscal_year"
        ] - 1
    )

    df = df[
        df["fiscal_year"].between(
            start_year,
            end_year
        )
    ].copy()

    # Some 10-Ks contain quarterly or transaction-date
    # point-in-time contexts. Keep the latest date
    # representing the fiscal year-end balance.
    df = (
        df
        .sort_values("end")
        .drop_duplicates(
            subset=["fiscal_year"],
            keep="last"
        )
    )

    return df[
        ["fiscal_year", "end", "val", "filed"]
    ].reset_index(drop=True)

In [32]:
assets_test = extract_point_in_time_fact(
    us_gaap_facts,
    "Assets"
)

print("Rows:", len(assets_test))
print()
print(assets_test[["fy", "val"]])

Rows: 5

       fy           val
0  2021.0  351002000000
1  2022.0  352755000000
2  2023.0  352583000000
3  2024.0  364980000000
4  2025.0  359241000000


In [33]:
cash_candidates = [
    key for key in us_gaap_facts.keys()
    if "CashAndCashEquivalents" in key
    or "CashCashEquivalents" in key
]

cash_candidates

['CashAndCashEquivalentsAtCarryingValue',
 'CashAndCashEquivalentsPeriodIncreaseDecrease',
 'CashCashEquivalentsRestrictedCashAndRestrictedCashEquivalents',
 'CashCashEquivalentsRestrictedCashAndRestrictedCashEquivalentsPeriodIncreaseDecreaseIncludingExchangeRateEffect',
 'FairValueAssetsMeasuredOnRecurringBasisCashAndCashEquivalents',
 'RestrictedCashAndCashEquivalentsAtCarryingValue',
 'RestrictedCashAndCashEquivalentsNoncurrent']

In [34]:
cash_test = extract_point_in_time_fact(
    us_gaap_facts,
    "CashAndCashEquivalentsAtCarryingValue"
)

print("Rows:", len(cash_test))
print()
print(cash_test[["fy", "val"]])

Rows: 5

       fy          val
0  2021.0  34940000000
1  2022.0  23646000000
2  2023.0  29965000000
3  2024.0  29943000000
4  2025.0  35934000000


In [35]:
debt_candidates = [
    key for key in us_gaap_facts.keys()
    if "Debt" in key
    or "LongTermBorrowings" in key
]

debt_candidates

['AvailableForSaleDebtSecuritiesAccumulatedGrossUnrealizedGainBeforeTax',
 'AvailableForSaleDebtSecuritiesAccumulatedGrossUnrealizedLossBeforeTax',
 'AvailableForSaleDebtSecuritiesAmortizedCostBasis',
 'AvailableForSaleSecuritiesDebtMaturitiesRollingAfterYearTenFairValue',
 'AvailableForSaleSecuritiesDebtMaturitiesRollingYearSixThroughTenFairValue',
 'AvailableForSaleSecuritiesDebtMaturitiesRollingYearTwoThroughFiveFairValue',
 'AvailableForSaleSecuritiesDebtMaturitiesSingleMaturityDate',
 'AvailableForSaleSecuritiesDebtSecurities',
 'AvailableForSaleSecuritiesDebtSecuritiesCurrent',
 'AvailableForSaleSecuritiesDebtSecuritiesNoncurrent',
 'DebtInstrumentCarryingAmount',
 'DebtInstrumentUnamortizedDiscount',
 'DebtInstrumentUnamortizedDiscountPremiumAndDebtIssuanceCostsNet',
 'DebtInstrumentUnamortizedDiscountPremiumNet',
 'DebtSecuritiesAvailableForSaleContinuousUnrealizedLossPosition12MonthsOrLonger',
 'DebtSecuritiesAvailableForSaleContinuousUnrealizedLossPosition12MonthsOrLongerAccu

In [36]:
better_debt_candidates = [
    key for key in us_gaap_facts.keys()
    if (
        "LongTermDebt" in key
        or "ShortTermBorrowings" in key
        or "LongTermBorrowings" in key
        or "DebtCurrent" in key
    )
]

better_debt_candidates

['LongTermDebt',
 'LongTermDebtCurrent',
 'LongTermDebtMaturitiesRepaymentsOfPrincipalAfterYearFive',
 'LongTermDebtMaturitiesRepaymentsOfPrincipalInNextTwelveMonths',
 'LongTermDebtMaturitiesRepaymentsOfPrincipalInYearFive',
 'LongTermDebtMaturitiesRepaymentsOfPrincipalInYearFour',
 'LongTermDebtMaturitiesRepaymentsOfPrincipalInYearThree',
 'LongTermDebtMaturitiesRepaymentsOfPrincipalInYearTwo',
 'LongTermDebtMaturitiesRepaymentsOfPrincipalRemainderOfFiscalYear',
 'LongTermDebtNoncurrent',
 'OtherShortTermBorrowings',
 'ProceedsFromIssuanceOfLongTermDebt',
 'RepaymentsOfLongTermDebt']

In [37]:
for tag in [
    "LongTermDebt",
    "LongTermDebtCurrent",
    "LongTermDebtNoncurrent"
]:
    fact = us_gaap_facts[tag]

    print("TAG:", tag)
    print("LABEL:", fact["label"])
    print("DESCRIPTION:", fact["description"][:250])
    print()

TAG: LongTermDebt
LABEL: Long-term Debt
DESCRIPTION: Amount, after unamortized (discount) premium and debt issuance costs, of long-term debt. Includes, but not limited to, notes payable, bonds payable, debentures, mortgage loans and commercial paper. Excludes capital lease obligations.

TAG: LongTermDebtCurrent
LABEL: Long-term Debt, Current Maturities
DESCRIPTION: Amount, after unamortized (discount) premium and debt issuance costs, of long-term debt, classified as current. Includes, but not limited to, notes payable, bonds payable, debentures, mortgage loans and commercial paper. Excludes capital lease obliga

TAG: LongTermDebtNoncurrent
LABEL: Long-term Debt, Excluding Current Maturities
DESCRIPTION: Amount after unamortized (discount) premium and debt issuance costs of long-term debt classified as noncurrent and excluding amounts to be repaid within one year or the normal operating cycle, if longer. Includes, but not limited to, notes payable, b



In [38]:
debt_total_test = extract_point_in_time_fact(
    us_gaap_facts,
    "LongTermDebt"
)

debt_current_test = extract_point_in_time_fact(
    us_gaap_facts,
    "LongTermDebtCurrent"
)

debt_noncurrent_test = extract_point_in_time_fact(
    us_gaap_facts,
    "LongTermDebtNoncurrent"
)

print("LongTermDebt:")
print(debt_total_test[["fy", "val"]])

print("\nLongTermDebtCurrent:")
print(debt_current_test[["fy", "val"]])

print("\nLongTermDebtNoncurrent:")
print(debt_noncurrent_test[["fy", "val"]])

LongTermDebt:
       fy           val
0  2023.0  110087000000
1  2023.0  105103000000
2  2024.0   96662000000
3  2025.0   90678000000

LongTermDebtCurrent:
     fy          val
0  2021   9613000000
1  2022  11128000000
2  2023   9822000000
3  2024  10912000000
4  2025  12350000000

LongTermDebtNoncurrent:
     fy           val
0  2021  109106000000
1  2022   98959000000
2  2023   95281000000
3  2024   85750000000
4  2025   78328000000


In [40]:
apple_debt = (
    debt_current_test[["fy", "val"]]
    .rename(columns={"val": "debt_current"})
    .merge(
        debt_noncurrent_test[["fy", "val"]]
        .rename(columns={"val": "debt_noncurrent"}),
        on="fy",
        how="outer"
    )
)

apple_debt["total_debt"] = (
    apple_debt["debt_current"]
    + apple_debt["debt_noncurrent"]
)

apple_debt["fy"] = apple_debt["fy"].astype(int)

apple_debt

,fy,debt_current,debt_noncurrent,total_debt
0,2021,9613000000,109106000000,118719000000
1,2022,11128000000,98959000000,110087000000
2,2023,9822000000,95281000000,105103000000
3,2024,10912000000,85750000000,96662000000
4,2025,12350000000,78328000000,90678000000


In [41]:
print(
    apple_debt[
        ["fy", "debt_current", "debt_noncurrent", "total_debt"]
    ]
)

print()
print("Missing values:")
print(apple_debt.isna().sum())

     fy  debt_current  debt_noncurrent    total_debt
0  2021    9613000000     109106000000  118719000000
1  2022   11128000000      98959000000  110087000000
2  2023    9822000000      95281000000  105103000000
3  2024   10912000000      85750000000   96662000000
4  2025   12350000000      78328000000   90678000000

Missing values:
fy                 0
debt_current       0
debt_noncurrent    0
total_debt         0
dtype: int64


In [42]:
apple_core = (
    apple_core
    .merge(
        assets_test[["fy", "val"]]
        .rename(columns={"val": "assets"}),
        on="fy",
        how="left"
    )
    .merge(
        cash_test[["fy", "val"]]
        .rename(columns={"val": "cash"}),
        on="fy",
        how="left"
    )
    .merge(
        apple_debt[["fy", "total_debt"]],
        on="fy",
        how="left"
    )
)

apple_core

,fy,revenue,rd_expense,operating_income,capex,rd_intensity,capex_intensity,operating_margin,revenue_growth,assets,cash,total_debt
0,2021,365817000000,21914000000,108949000000,11085000000,0.059904,0.030302,0.297824,NaN,351002000000,34940000000,118719000000
1,2022,394328000000,26251000000,119437000000,10708000000,0.066571,0.027155,0.302887,0.077938,352755000000,23646000000,110087000000
2,2023,383285000000,29915000000,114301000000,10959000000,0.078049,0.028592,0.298214,-0.028005,352583000000,29965000000,105103000000
3,2024,391035000000,31370000000,123216000000,9447000000,0.080223,0.024159,0.315102,0.020220,364980000000,29943000000,96662000000
4,2025,416161000000,34550000000,133050000000,12715000000,0.083021,0.030553,0.319708,0.064255,359241000000,35934000000,90678000000


In [43]:
apple_core["leverage"] = (
    apple_core["total_debt"] / apple_core["assets"]
)

apple_core["log_assets"] = np.log(
    apple_core["assets"]
)

print(
    apple_core[
        [
            "fy",
            "assets",
            "cash",
            "total_debt",
            "leverage",
            "log_assets"
        ]
    ].round(4)
)

print()
print("Missing values:")
print(
    apple_core[
        ["assets", "cash", "total_debt", "leverage", "log_assets"]
    ].isna().sum()
)

     fy        assets         cash    total_debt  leverage  log_assets
0  2021  351002000000  34940000000  118719000000    0.3382     26.5841
1  2022  352755000000  23646000000  110087000000    0.3121     26.5890
2  2023  352583000000  29965000000  105103000000    0.2981     26.5886
3  2024  364980000000  29943000000   96662000000    0.2648     26.6231
4  2025  359241000000  35934000000   90678000000    0.2524     26.6073

Missing values:
assets        0
cash          0
total_debt    0
leverage      0
log_assets    0
dtype: int64


In [53]:
import yfinance as yf

apple = yf.Ticker("AAPL")

apple_prices = apple.history(
    start="2021-09-20",
    end="2025-11-01",
    auto_adjust=False
)

apple_prices.head()

,Open,High,Low,Close,Adj Close,Volume,Dividends,Stock Splits
Date,,,,,,,,
2021-09-20 00:00:00-04:00,143.800003,144.839996,141.270004,142.940002,139.419540,123478900,0.0,0.0
2021-09-21 00:00:00-04:00,143.929993,144.600006,142.779999,143.429993,139.897491,75834000,0.0,0.0
2021-09-22 00:00:00-04:00,144.449997,146.429993,143.699997,145.850006,142.257874,76404300,0.0,0.0
2021-09-23 00:00:00-04:00,146.649994,147.080002,145.639999,146.830002,143.213776,64838200,0.0,0.0
2021-09-24 00:00:00-04:00,145.660004,147.470001,145.559998,146.919998,143.301483,53477900,0.0,0.0


In [45]:
print("Rows:", len(apple_prices))
print()
print(apple_prices.tail())

Rows: 1015

                                 Open        High         Low       Close  \
Date                                                                        
2025-09-29 00:00:00-04:00  254.559998  255.000000  253.009995  254.429993   
2025-09-30 00:00:00-04:00  254.860001  255.919998  253.110001  254.630005   
2025-10-01 00:00:00-04:00  255.039993  258.790009  254.929993  255.449997   
2025-10-02 00:00:00-04:00  256.579987  258.179993  254.149994  257.130005   
2025-10-03 00:00:00-04:00  254.669998  259.239990  253.949997  258.019989   

                            Adj Close    Volume  Dividends  Stock Splits  
Date                                                                      
2025-09-29 00:00:00-04:00  253.493591  40127700        0.0           0.0  
2025-09-30 00:00:00-04:00  253.692871  37704300        0.0           0.0  
2025-10-01 00:00:00-04:00  254.509857  48713900        0.0           0.0  
2025-10-02 00:00:00-04:00  256.183655  42630200        0.0           0.0 

In [46]:
apple_fiscal_dates = apple_core[["fy"]].copy()

apple_fiscal_dates["fiscal_year_end"] = pd.to_datetime([
    "2021-09-25",
    "2022-09-24",
    "2023-09-30",
    "2024-09-28",
    "2025-09-27"
])

apple_fiscal_dates

,fy,fiscal_year_end
0,2021,2021-09-25
1,2022,2022-09-24
2,2023,2023-09-30
3,2024,2024-09-28
4,2025,2025-09-27


In [54]:
price_data = apple_prices.reset_index().copy()

price_data["Date"] = pd.to_datetime(
    price_data["Date"]
).dt.tz_localize(None)

results = []

for _, row in apple_fiscal_dates.iterrows():
    fy = row["fy"]
    fiscal_date = row["fiscal_year_end"]

    eligible = price_data[
        price_data["Date"] <= fiscal_date
    ]

    closest_row = eligible.iloc[-1]

    results.append({
        "fy": fy,
        "fiscal_year_end": fiscal_date,
        "price_date": closest_row["Date"],
        "close_price": closest_row["Close"]
    })

apple_fy_prices = pd.DataFrame(results)

apple_fy_prices

,fy,fiscal_year_end,price_date,close_price
0,2021,2021-09-25,2021-09-24,146.919998
1,2022,2022-09-24,2022-09-23,150.429993
2,2023,2023-09-30,2023-09-29,171.210007
3,2024,2024-09-28,2024-09-27,227.789993
4,2025,2025-09-27,2025-09-26,255.460007


In [48]:
dei_facts = apple_facts["facts"]["dei"]

print("EntityCommonStockSharesOutstanding" in dei_facts)

print([
    key for key in dei_facts.keys()
    if "SharesOutstanding" in key
])

True
['EntityCommonStockSharesOutstanding']


In [49]:
shares_fact = dei_facts["EntityCommonStockSharesOutstanding"]

print(shares_fact["label"])
print(shares_fact["units"].keys())

Entity Common Stock, Shares Outstanding
dict_keys(['shares'])


In [50]:
shares_df = pd.DataFrame(
    shares_fact["units"]["shares"]
)

shares_df[
    ["end", "val", "fy", "fp", "form", "filed"]
].tail(15)

,end,val,fy,fp,form,filed
56,2023-04-21,15728702000,2023,Q2,10-Q,2023-05-05
57,2023-07-21,15634232000,2023,Q3,10-Q,2023-08-04
58,2023-10-20,15552752000,2023,FY,10-K,2023-11-03
59,2024-01-19,15441881000,2024,Q1,10-Q,2024-02-02
60,2024-04-19,15334082000,2024,Q2,10-Q,2024-05-03
61,2024-07-19,15204137000,2024,Q3,10-Q,2024-08-02
62,2024-10-18,15115823000,2024,FY,10-K,2024-11-01
63,2025-01-17,15022073000,2025,Q1,10-Q,2025-01-31
64,2025-04-18,14935826000,2025,Q2,10-Q,2025-05-02
65,2025-07-18,14840390000,2025,Q3,10-Q,2025-08-01


In [51]:
annual_shares = shares_df[
    (shares_df["form"] == "10-K") &
    (shares_df["fp"] == "FY") &
    (shares_df["fy"].between(2021, 2025))
].copy()

annual_shares = annual_shares[
    ["fy", "end", "val", "filed"]
].rename(
    columns={
        "end": "shares_date",
        "val": "shares_outstanding"
    }
)

annual_shares["fy"] = annual_shares["fy"].astype(int)

annual_shares

,fy,shares_date,shares_outstanding,filed
50,2021,2021-10-15,16406397000,2021-10-29
54,2022,2022-10-14,15908118000,2022-10-28
58,2023,2023-10-20,15552752000,2023-11-03
62,2024,2024-10-18,15115823000,2024-11-01
66,2025,2025-10-17,14776353000,2025-10-31


In [55]:
annual_shares["shares_date"] = pd.to_datetime(
    annual_shares["shares_date"]
)

results = []

for _, row in annual_shares.iterrows():
    fy = row["fy"]
    shares_date = row["shares_date"]
    shares_outstanding = row["shares_outstanding"]

    eligible = price_data[
        price_data["Date"] <= shares_date
    ]

    closest_row = eligible.iloc[-1]

    results.append({
        "fy": fy,
        "shares_date": shares_date,
        "price_date": closest_row["Date"],
        "close_price": closest_row["Close"],
        "shares_outstanding": shares_outstanding
    })

apple_market_data = pd.DataFrame(results)

apple_market_data

,fy,shares_date,price_date,close_price,shares_outstanding
0,2021,2021-10-15,2021-10-15,144.839996,16406397000
1,2022,2022-10-14,2022-10-14,138.380005,15908118000
2,2023,2023-10-20,2023-10-20,172.880005,15552752000
3,2024,2024-10-18,2024-10-18,235.000000,15115823000
4,2025,2025-10-17,2025-10-17,252.289993,14776353000


In [56]:
apple_market_data["market_cap"] = (
    apple_market_data["close_price"]
    * apple_market_data["shares_outstanding"]
)

apple_market_data[
    [
        "fy",
        "close_price",
        "shares_outstanding",
        "market_cap"
    ]
]

,fy,close_price,shares_outstanding,market_cap
0,2021,144.839996,16406397000,2.376302e+12
1,2022,138.380005,15908118000,2.201365e+12
2,2023,172.880005,15552752000,2.688760e+12
3,2024,235.000000,15115823000,3.552218e+12
4,2025,252.289993,14776353000,3.727926e+12


In [57]:
apple_core = apple_core.merge(
    apple_market_data[
        ["fy", "market_cap"]
    ],
    on="fy",
    how="left"
)

In [58]:
apple_core["enterprise_value"] = (
    apple_core["market_cap"]
    + apple_core["total_debt"]
    - apple_core["cash"]
)

apple_core["ev_sales"] = (
    apple_core["enterprise_value"]
    / apple_core["revenue"]
)

apple_core[
    [
        "fy",
        "market_cap",
        "enterprise_value",
        "ev_sales"
    ]
].round(3)

,fy,market_cap,enterprise_value,ev_sales
0,2021,2.376302e+12,2.460081e+12,6.725
1,2022,2.201365e+12,2.287806e+12,5.802
2,2023,2.688760e+12,2.763898e+12,7.211
3,2024,3.552218e+12,3.618937e+12,9.255
4,2025,3.727926e+12,3.782670e+12,9.089


In [59]:
apple_core.to_csv(
    "../data/raw/apple_prototype_2021_2025.csv",
    index=False
)

print("Saved Apple prototype.")

Saved Apple prototype.


In [60]:
test_tickers = [
    "AAPL",
    "MSFT",
    "NVDA",
    "ADBE",
    "META"
]

test_companies = company_universe[
    company_universe["ticker"].isin(test_tickers)
].copy()

test_companies = test_companies[
    [
        "ticker",
        "company",
        "cik",
        "sector",
        "industry"
    ]
].reset_index(drop=True)

test_companies

,ticker,company,cik,sector,industry
0,ADBE,Adobe Inc.,796343,Information Technology,Application Software
1,AAPL,Apple Inc.,320193,Information Technology,"Technology Hardware, Storage & Peripherals"
2,META,Meta Platforms,1326801,Communication Services,Interactive Media & Services
3,MSFT,Microsoft,789019,Information Technology,Systems Software
4,NVDA,Nvidia,1045810,Information Technology,Semiconductors


In [61]:
print("Number of test companies:", len(test_companies))
print()

print(test_companies[["ticker", "company", "cik"]])

Number of test companies: 5

  ticker         company      cik
0   ADBE      Adobe Inc.   796343
1   AAPL      Apple Inc.   320193
2   META  Meta Platforms  1326801
3   MSFT       Microsoft   789019
4   NVDA          Nvidia  1045810


In [62]:
def get_company_facts(cik):
    cik_str = str(int(cik)).zfill(10)

    url = (
        "https://data.sec.gov/api/xbrl/companyfacts/"
        f"CIK{cik_str}.json"
    )

    response = requests.get(
        url,
        headers=headers,
        timeout=30
    )

    response.raise_for_status()

    return response.json()

In [63]:
msft_row = test_companies[
    test_companies["ticker"] == "MSFT"
].iloc[0]

msft_facts = get_company_facts(
    msft_row["cik"]
)

print("Company:", msft_facts["entityName"])
print("CIK:", msft_facts["cik"])
print(
    "US-GAAP concepts:",
    len(msft_facts["facts"]["us-gaap"])
)

Company: MICROSOFT CORP
CIK: 789019
US-GAAP concepts: 562


In [64]:
msft_us_gaap = msft_facts["facts"]["us-gaap"]

msft_revenue = extract_annual_fact(
    msft_us_gaap,
    "RevenueFromContractWithCustomerExcludingAssessedTax"
)

msft_rd = extract_annual_fact(
    msft_us_gaap,
    "ResearchAndDevelopmentExpense"
)

msft_operating_income = extract_annual_fact(
    msft_us_gaap,
    "OperatingIncomeLoss"
)

print("Revenue rows:", len(msft_revenue))
print("R&D rows:", len(msft_rd))
print("Operating income rows:", len(msft_operating_income))

Revenue rows: 5
R&D rows: 5
Operating income rows: 5


In [65]:
print("Revenue:")
print(msft_revenue[["fy", "val"]])

print("\nR&D:")
print(msft_rd[["fy", "val"]])

print("\nOperating income:")
print(msft_operating_income[["fy", "val"]])

Revenue:
       fy           val
0  2021.0  168088000000
1  2022.0  198270000000
2  2023.0  211915000000
3  2024.0  245122000000
4  2025.0  281724000000

R&D:
       fy          val
0  2021.0  20716000000
1  2022.0  24512000000
2  2023.0  27195000000
3  2024.0  29510000000
4  2025.0  32488000000

Operating income:
       fy           val
0  2021.0   69916000000
1  2022.0   83383000000
2  2023.0   88523000000
3  2024.0  109433000000
4  2025.0  128528000000


In [66]:
msft_capex = extract_annual_fact(
    msft_us_gaap,
    "PaymentsToAcquirePropertyPlantAndEquipment"
)

print("Capex rows:", len(msft_capex))
print()
print(msft_capex[["fy", "val"]])

Capex rows: 5

       fy          val
0  2021.0  20622000000
1  2022.0  23886000000
2  2023.0  28107000000
3  2024.0  44477000000
4  2025.0  64551000000


In [67]:
msft_assets = extract_point_in_time_fact(
    msft_us_gaap,
    "Assets"
)

msft_cash = extract_point_in_time_fact(
    msft_us_gaap,
    "CashAndCashEquivalentsAtCarryingValue"
)

print("Assets rows:", len(msft_assets))
print("Cash rows:", len(msft_cash))

Assets rows: 5
Cash rows: 5


In [68]:
print("Assets:")
print(msft_assets[["fy", "val"]])

print("\nCash:")
print(msft_cash[["fy", "val"]])

Assets:
       fy           val
0  2021.0  333779000000
1  2022.0  364840000000
2  2023.0  411976000000
3  2024.0  512163000000
4  2025.0  619003000000

Cash:
       fy          val
0  2021.0  14224000000
1  2022.0  13931000000
2  2023.0  34704000000
3  2024.0  18315000000
4  2025.0  30242000000


In [69]:
msft_debt_candidates = [
    key for key in msft_us_gaap.keys()
    if (
        "LongTermDebt" in key
        or "ShortTermBorrowings" in key
        or "DebtCurrent" in key
    )
]

msft_debt_candidates

['ConvertibleDebtCurrent',
 'LongTermDebt',
 'LongTermDebtCurrent',
 'LongTermDebtFairValue',
 'LongTermDebtMaturitiesRepaymentsOfPrincipalAfterYearFive',
 'LongTermDebtMaturitiesRepaymentsOfPrincipalInNextTwelveMonths',
 'LongTermDebtMaturitiesRepaymentsOfPrincipalInYearFive',
 'LongTermDebtMaturitiesRepaymentsOfPrincipalInYearFour',
 'LongTermDebtMaturitiesRepaymentsOfPrincipalInYearThree',
 'LongTermDebtMaturitiesRepaymentsOfPrincipalInYearTwo',
 'LongTermDebtMaturitiesRepaymentsOfPrincipalRemainderOfFiscalYear',
 'LongTermDebtNoncurrent',
 'ShortTermBorrowings']

In [70]:
msft_debt_current = extract_point_in_time_fact(
    msft_us_gaap,
    "LongTermDebtCurrent"
)

msft_debt_noncurrent = extract_point_in_time_fact(
    msft_us_gaap,
    "LongTermDebtNoncurrent"
)

msft_short_term = extract_point_in_time_fact(
    msft_us_gaap,
    "ShortTermBorrowings"
)

print("Current debt rows:", len(msft_debt_current))
print("Noncurrent debt rows:", len(msft_debt_noncurrent))
print("Short-term borrowings rows:", len(msft_short_term))

Current debt rows: 5
Noncurrent debt rows: 5
Short-term borrowings rows: 0


In [71]:
msft_debt = (
    msft_debt_current[["fy", "val"]]
    .rename(columns={"val": "debt_current"})
    .merge(
        msft_debt_noncurrent[["fy", "val"]]
        .rename(columns={"val": "debt_noncurrent"}),
        on="fy",
        how="outer"
    )
)

msft_debt["total_debt"] = (
    msft_debt["debt_current"]
    + msft_debt["debt_noncurrent"]
)

msft_debt["fy"] = msft_debt["fy"].astype(int)

print(msft_debt)
print()
print("Missing values:")
print(msft_debt.isna().sum())

     fy  debt_current  debt_noncurrent   total_debt
0  2021    8072000000      50074000000  58146000000
1  2022    2749000000      47032000000  49781000000
2  2023    5247000000      41990000000  47237000000
3  2024    2249000000      42688000000  44937000000
4  2025    2999000000      40152000000  43151000000

Missing values:
fy                 0
debt_current       0
debt_noncurrent    0
total_debt         0
dtype: int64


In [72]:
nvda_row = test_companies[
    test_companies["ticker"] == "NVDA"
].iloc[0]

nvda_facts = get_company_facts(
    nvda_row["cik"]
)

nvda_us_gaap = nvda_facts["facts"]["us-gaap"]

print("Company:", nvda_facts["entityName"])
print("CIK:", nvda_facts["cik"])
print("US-GAAP concepts:", len(nvda_us_gaap))

Company: NVIDIA CORP
CIK: 1045810
US-GAAP concepts: 627


In [73]:
nvda_revenue = extract_annual_fact(
    nvda_us_gaap,
    "RevenueFromContractWithCustomerExcludingAssessedTax"
)

nvda_rd = extract_annual_fact(
    nvda_us_gaap,
    "ResearchAndDevelopmentExpense"
)

nvda_operating_income = extract_annual_fact(
    nvda_us_gaap,
    "OperatingIncomeLoss"
)

print("Revenue rows:", len(nvda_revenue))
print("R&D rows:", len(nvda_rd))
print("Operating income rows:", len(nvda_operating_income))

Revenue rows: 2
R&D rows: 5
Operating income rows: 5


In [74]:
nvda_revenue_candidates = [
    key for key in nvda_us_gaap.keys()
    if "Revenue" in key
]

nvda_revenue_candidates

['BusinessAcquisitionProFormaRevenue',
 'BusinessAcquisitionsProFormaRevenue',
 'ContractWithCustomerLiabilityRevenueRecognized',
 'CostOfRevenue',
 'DeferredRevenue',
 'DeferredRevenueAdditions',
 'DeferredRevenueCurrent',
 'DeferredRevenueNoncurrent',
 'DeferredRevenueRevenueRecognized',
 'DeferredRevenueRevenueRecognized1',
 'RevenueFromContractWithCustomerExcludingAssessedTax',
 'RevenueRemainingPerformanceObligation',
 'RevenueRemainingPerformanceObligationPercentage',
 'Revenues']

In [75]:
nvda_revenue_fallback = extract_annual_fact(
    nvda_us_gaap,
    "Revenues"
)

print("Fallback revenue rows:", len(nvda_revenue_fallback))
print()
print(nvda_revenue_fallback[["fy", "val"]])

Fallback revenue rows: 6

     fy           val
0  2022   10918000000
1  2022   16675000000
2  2022   26914000000
3  2023   26974000000
4  2024   60922000000
5  2025  130497000000


In [76]:
nvda_revenue_fallback[
    ["fy", "start", "end", "val", "filed"]
]

,fy,start,end,val,filed
0,2022,2019-01-28,2020-01-26,10918000000,2022-03-18
1,2022,2020-01-27,2021-01-31,16675000000,2022-03-18
2,2022,2021-02-01,2022-01-30,26914000000,2022-03-18
3,2023,2022-01-31,2023-01-29,26974000000,2023-02-24
4,2024,2023-01-30,2024-01-28,60922000000,2024-02-21
5,2025,2024-01-29,2025-01-26,130497000000,2025-02-26


In [77]:
nvda_revenue_clean = nvda_revenue_fallback.copy()

nvda_revenue_clean["fiscal_year"] = (
    pd.to_datetime(nvda_revenue_clean["end"]).dt.year
)

nvda_revenue_clean = nvda_revenue_clean[
    nvda_revenue_clean["fiscal_year"].between(2021, 2025)
].copy()

nvda_revenue_clean[
    ["fiscal_year", "start", "end", "val", "filed"]
]

,fiscal_year,start,end,val,filed
1,2021,2020-01-27,2021-01-31,16675000000,2022-03-18
2,2022,2021-02-01,2022-01-30,26914000000,2022-03-18
3,2023,2022-01-31,2023-01-29,26974000000,2023-02-24
4,2024,2023-01-30,2024-01-28,60922000000,2024-02-21
5,2025,2024-01-29,2025-01-26,130497000000,2025-02-26


In [79]:
apple_revenue_new = extract_annual_fact(
    us_gaap_facts,
    "RevenueFromContractWithCustomerExcludingAssessedTax"
)

msft_revenue_new = extract_annual_fact(
    msft_us_gaap,
    "RevenueFromContractWithCustomerExcludingAssessedTax"
)

nvda_revenue_new = extract_annual_fact(
    nvda_us_gaap,
    "Revenues"
)

print("Apple:")
print(apple_revenue_new[["fiscal_year", "val"]])

print("\nMicrosoft:")
print(msft_revenue_new[["fiscal_year", "val"]])

print("\nNVIDIA:")
print(nvda_revenue_new[["fiscal_year", "val"]])

Apple:
   fiscal_year           val
0         2021  365817000000
1         2022  394328000000
2         2023  383285000000
3         2024  391035000000
4         2025  416161000000

Microsoft:
   fiscal_year           val
0         2021  168088000000
1         2022  198270000000
2         2023  211915000000
3         2024  245122000000
4         2025  281724000000

NVIDIA:
   fiscal_year           val
0         2021   16675000000
1         2022   26914000000
2         2023   26974000000
3         2024   60922000000
4         2025  130497000000


In [80]:
nvda_rd_new = extract_annual_fact(
    nvda_us_gaap,
    "ResearchAndDevelopmentExpense"
)

nvda_operating_income_new = extract_annual_fact(
    nvda_us_gaap,
    "OperatingIncomeLoss"
)

nvda_capex_new = extract_annual_fact(
    nvda_us_gaap,
    "PaymentsToAcquirePropertyPlantAndEquipment"
)

print("R&D rows:", len(nvda_rd_new))
print("Operating income rows:", len(nvda_operating_income_new))
print("Capex rows:", len(nvda_capex_new))

R&D rows: 5
Operating income rows: 5
Capex rows: 0


In [81]:
nvda_capex_candidates = [
    key for key in nvda_us_gaap.keys()
    if (
        "PropertyPlantAndEquipment" in key
        or "CapitalExpenditure" in key
        or "PaymentsToAcquire" in key
    )
]

nvda_capex_candidates

['AccumulatedDepreciationDepletionAndAmortizationPropertyPlantAndEquipment',
 'BusinessAcquisitionPurchasePriceAllocationPropertyPlantAndEquipment',
 'PaymentsToAcquireAvailableForSaleSecuritiesDebt',
 'PaymentsToAcquireBusinessesNetOfCashAcquired',
 'PaymentsToAcquireMarketableSecurities',
 'PaymentsToAcquireOtherInvestments',
 'PaymentsToAcquireProductiveAssets',
 'PaymentsToAcquirePropertyPlantAndEquipment',
 'PropertyPlantAndEquipmentGross',
 'PropertyPlantAndEquipmentNet',
 'PropertyPlantAndEquipmentUsefulLifeMaximum',
 'PropertyPlantAndEquipmentUsefulLifeMinimum',
 'CapitalExpendituresIncurredButNotYetPaid',
 'PaymentsToAcquireEquitySecuritiesFvNi',
 'PaymentsToAcquireBusinessTwoNetOfCashAcquired']

In [82]:
nvda_capex_fallback = extract_annual_fact(
    nvda_us_gaap,
    "PaymentsToAcquireProductiveAssets"
)

print("Fallback capex rows:", len(nvda_capex_fallback))
print()
print(
    nvda_capex_fallback[
        ["fiscal_year", "val"]
    ]
)

Fallback capex rows: 4

   fiscal_year         val
0         2022   976000000
1         2023  1833000000
2         2024  1069000000
3         2025  3236000000


In [83]:
for tag in [
    "PaymentsToAcquirePropertyPlantAndEquipment",
    "PaymentsToAcquireProductiveAssets"
]:
    print("\nTAG:", tag)

    fact = nvda_us_gaap[tag]
    df = pd.DataFrame(fact["units"]["USD"])

    print(
        df[
            ["start", "end", "val", "fy", "fp", "form", "filed"]
        ].tail(20)
    )


TAG: PaymentsToAcquirePropertyPlantAndEquipment
         start         end        val    fy  fp  form       filed
35  2016-02-01  2016-10-30  125000000  2018  Q3  10-Q  2017-11-21
36  2017-01-30  2017-04-30   54000000  2018  Q1  10-Q  2017-05-23
37  2017-01-30  2017-04-30   54000000  2019  Q1  10-Q  2018-05-22
38  2017-01-30  2017-07-30  108000000  2018  Q2  10-Q  2017-08-23
39  2017-01-30  2017-07-30  108000000  2019  Q2  10-Q  2018-08-16
40  2017-01-30  2017-10-29  177000000  2018  Q3  10-Q  2017-11-21
41  2017-01-30  2017-10-29  177000000  2019  Q3  10-Q  2018-11-15
42  2018-01-29  2018-04-29  118000000  2019  Q1  10-Q  2018-05-22
43  2018-01-29  2018-04-29  118000000  2020  Q1  10-Q  2019-05-16
44  2018-01-29  2018-07-29  247000000  2019  Q2  10-Q  2018-08-16
45  2018-01-29  2018-07-29  247000000  2020  Q2  10-Q  2019-08-15
46  2018-01-29  2018-10-28  397000000  2019  Q3  10-Q  2018-11-15
47  2018-01-29  2018-10-28  397000000  2020  Q3  10-Q  2019-11-14
48  2019-01-28  2019-04-28 

In [84]:
for tag in [
    "PaymentsToAcquirePropertyPlantAndEquipment",
    "PaymentsToAcquireProductiveAssets"
]:
    print("\nTAG:", tag)

    fact = nvda_us_gaap[tag]
    df = pd.DataFrame(fact["units"]["USD"])

    df["end"] = pd.to_datetime(df["end"])

    check = df[
        (df["form"] == "10-K") &
        (df["fp"] == "FY") &
        (df["end"].dt.year.between(2020, 2022))
    ].copy()

    print(
        check[
            ["start", "end", "val", "fy", "filed"]
        ]
    )


TAG: PaymentsToAcquirePropertyPlantAndEquipment
Empty DataFrame
Columns: [start, end, val, fy, filed]
Index: []

TAG: PaymentsToAcquireProductiveAssets
        start        end        val    fy       filed
8  2021-02-01 2022-01-30  976000000  2024  2024-02-21


In [86]:
possible_nvda_capex = []

for key, fact in nvda_us_gaap.items():
    label = fact.get("label") or ""
    description = fact.get("description") or ""

    text = (
        key + " " +
        label + " " +
        description
    ).lower()

    if (
        "capital expenditure" in text
        or "property, plant" in text
        or "productive asset" in text
        or "purchase of property" in text
        or "acquire property" in text
    ):
        possible_nvda_capex.append(
            (key, label)
        )

possible_nvda_capex

[('AccumulatedDepreciationDepletionAndAmortizationPropertyPlantAndEquipment',
  'Accumulated Depreciation, Depletion and Amortization, Property, Plant, and Equipment'),
 ('BusinessAcquisitionPurchasePriceAllocationPropertyPlantAndEquipment',
  'Business Acquisition, Purchase Price Allocation, Property, Plant and Equipment (Deprecated 2013-01-31)'),
 ('NetCashProvidedByUsedInInvestingActivities',
  'Net Cash Provided by (Used in) Investing Activities'),
 ('PaymentsToAcquireProductiveAssets',
  'Payments to Acquire Productive Assets'),
 ('PaymentsToAcquirePropertyPlantAndEquipment',
  'Payments to Acquire Property, Plant, and Equipment'),
 ('PropertyPlantAndEquipmentGross', 'Property, Plant and Equipment, Gross'),
 ('PropertyPlantAndEquipmentNet', 'Property, Plant and Equipment, Net'),
 ('PropertyPlantAndEquipmentUsefulLifeMaximum',
  'Property, Plant and Equipment, Useful Life, Maximum (Deprecated 2012-01-31)'),
 ('PropertyPlantAndEquipmentUsefulLifeMinimum',
  'Property, Plant and Equi

In [87]:
nvda_capex_final = nvda_capex_fallback.copy()

nvda_capex_final = (
    nvda_capex_final
    .rename(columns={"fiscal_year": "fiscal_year"})
)

nvda_capex_final[
    ["fiscal_year", "val"]
]

,fiscal_year,val
0,2022,976000000
1,2023,1833000000
2,2024,1069000000
3,2025,3236000000


In [88]:
nvda_assets = extract_point_in_time_fact(
    nvda_us_gaap,
    "Assets"
)

nvda_cash = extract_point_in_time_fact(
    nvda_us_gaap,
    "CashAndCashEquivalentsAtCarryingValue"
)

print("Assets rows:", len(nvda_assets))
print("Cash rows:", len(nvda_cash))

Assets rows: 5
Cash rows: 5


In [89]:
print("Assets:")
print(nvda_assets[["fy", "val"]])

print("\nCash:")
print(nvda_cash[["fy", "val"]])

Assets:
     fy           val
0  2021   28791000000
1  2022   44187000000
2  2023   41182000000
3  2024   65728000000
4  2025  111601000000

Cash:
     fy         val
0  2021   847000000
1  2022  1990000000
2  2023  3389000000
3  2024  7280000000
4  2025  8589000000


In [90]:
nvda_debt_candidates = [
    key for key in nvda_us_gaap.keys()
    if (
        "LongTermDebt" in key
        or "ShortTermBorrowings" in key
        or "DebtCurrent" in key
    )
]

nvda_debt_candidates

['BusinessAcquisitionPurchasePriceAllocationNotesPayableAndLongTermDebt',
 'ConvertibleDebtCurrent',
 'DebtCurrent',
 'LongTermDebt',
 'LongTermDebtCurrent',
 'LongTermDebtNoncurrent',
 'ProceedsFromIssuanceOfLongTermDebt',
 'LongTermDebtFairValue']

In [91]:
nvda_debt_current = extract_point_in_time_fact(
    nvda_us_gaap,
    "LongTermDebtCurrent"
)

nvda_debt_noncurrent = extract_point_in_time_fact(
    nvda_us_gaap,
    "LongTermDebtNoncurrent"
)

print("Current debt rows:", len(nvda_debt_current))
print("Noncurrent debt rows:", len(nvda_debt_noncurrent))

Current debt rows: 6
Noncurrent debt rows: 6


In [92]:
print("Current debt:")
print(
    nvda_debt_current[
        ["fy", "end", "val", "filed"]
    ]
)

print("\nNoncurrent debt:")
print(
    nvda_debt_noncurrent[
        ["fy", "end", "val", "filed"]
    ]
)

Current debt:
     fy        end         val       filed
0  2021 2020-01-26           0  2021-02-26
1  2021 2021-01-31   999000000  2021-02-26
2  2022 2022-01-30           0  2022-03-18
3  2023 2023-01-29  1250000000  2023-02-24
4  2024 2024-01-28  1250000000  2024-02-21
5  2025 2025-01-26           0  2025-02-26

Noncurrent debt:
     fy        end          val       filed
0  2021 2020-01-26   1991000000  2021-02-26
1  2021 2021-01-31   5964000000  2021-02-26
2  2022 2022-01-30  10946000000  2022-03-18
3  2023 2023-01-29   9703000000  2023-02-24
4  2024 2024-01-28   8459000000  2024-02-21
5  2025 2025-01-26   8463000000  2025-02-26


In [94]:
nvda_debt_current_new = extract_point_in_time_fact(
    nvda_us_gaap,
    "LongTermDebtCurrent"
)

nvda_debt_noncurrent_new = extract_point_in_time_fact(
    nvda_us_gaap,
    "LongTermDebtNoncurrent"
)

print("Current debt:")
print(
    nvda_debt_current_new[
        ["fiscal_year", "val"]
    ]
)

print("\nNoncurrent debt:")
print(
    nvda_debt_noncurrent_new[
        ["fiscal_year", "val"]
    ]
)

Current debt:
   fiscal_year         val
0         2021   999000000
1         2022           0
2         2023  1250000000
3         2024  1250000000
4         2025           0

Noncurrent debt:
   fiscal_year          val
0         2021   5964000000
1         2022  10946000000
2         2023   9703000000
3         2024   8459000000
4         2025   8463000000


In [95]:
nvda_debt = (
    nvda_debt_current_new[
        ["fiscal_year", "val"]
    ]
    .rename(columns={"val": "debt_current"})
    .merge(
        nvda_debt_noncurrent_new[
            ["fiscal_year", "val"]
        ].rename(columns={"val": "debt_noncurrent"}),
        on="fiscal_year",
        how="outer"
    )
)

nvda_debt["total_debt"] = (
    nvda_debt["debt_current"]
    + nvda_debt["debt_noncurrent"]
)

print(nvda_debt)

print("\nMissing values:")
print(nvda_debt.isna().sum())

   fiscal_year  debt_current  debt_noncurrent   total_debt
0         2021     999000000       5964000000   6963000000
1         2022             0      10946000000  10946000000
2         2023    1250000000       9703000000  10953000000
3         2024    1250000000       8459000000   9709000000
4         2025             0       8463000000   8463000000

Missing values:
fiscal_year        0
debt_current       0
debt_noncurrent    0
total_debt         0
dtype: int64


In [96]:
adbe_row = test_companies[
    test_companies["ticker"] == "ADBE"
].iloc[0]

adbe_facts = get_company_facts(
    adbe_row["cik"]
)

adbe_us_gaap = adbe_facts["facts"]["us-gaap"]

print("Company:", adbe_facts["entityName"])
print("CIK:", adbe_facts["cik"])
print("US-GAAP concepts:", len(adbe_us_gaap))

Company: ADOBE INC.
CIK: 796343
US-GAAP concepts: 559


In [97]:
adbe_revenue = extract_annual_fact(
    adbe_us_gaap,
    "RevenueFromContractWithCustomerExcludingAssessedTax"
)

adbe_rd = extract_annual_fact(
    adbe_us_gaap,
    "ResearchAndDevelopmentExpense"
)

adbe_operating_income = extract_annual_fact(
    adbe_us_gaap,
    "OperatingIncomeLoss"
)

print("Revenue rows:", len(adbe_revenue))
print("R&D rows:", len(adbe_rd))
print("Operating income rows:", len(adbe_operating_income))

KeyError: 'RevenueFromContractWithCustomerExcludingAssessedTax'

In [98]:
adbe_revenue_candidates = [
    key for key in adbe_us_gaap.keys()
    if "Revenue" in key
    or "Sales" in key
]

adbe_revenue_candidates

['BusinessAcquisitionProFormaRevenue',
 'ContractWithCustomerLiabilityRevenueRecognized',
 'CostOfRevenue',
 'DeferredRevenue',
 'DeferredRevenueCurrent',
 'DeferredRevenueNoncurrent',
 'EntityWideDisclosureOnGeographicAreasRevenueFromExternalCustomersAttributedToIndividualForeignCountriesAmount',
 'EntityWideRevenueMajorCustomerPercentage',
 'FairValueMeasurementWithUnobservableInputsReconciliationRecurringBasisAssetPurchasesSalesIssuancesSettlements',
 'GainLossOnSalesOfAssetsAndAssetImpairmentCharges',
 'IncreaseDecreaseInDeferredRevenue',
 'OperatingLeasesIncomeStatementSubleaseRevenue',
 'RevenueRemainingPerformanceObligation',
 'RevenueRemainingPerformanceObligationPercentage',
 'Revenues',
 'SalesAndExciseTaxPayableCurrent',
 'SalesReturnsGoods',
 'SalesRevenueGoodsNet',
 'SalesRevenueServicesNet',
 'SubscriptionRevenue']

In [99]:
adbe_revenue_fallback = extract_annual_fact(
    adbe_us_gaap,
    "Revenues"
)

print("Revenue rows:", len(adbe_revenue_fallback))
print()
print(
    adbe_revenue_fallback[
        ["fiscal_year", "val"]
    ]
)

Revenue rows: 5

   fiscal_year          val
0         2021  15785000000
1         2022  17606000000
2         2023  19409000000
3         2024  21505000000
4         2025  23769000000


In [100]:
adbe_rd = extract_annual_fact(
    adbe_us_gaap,
    "ResearchAndDevelopmentExpense"
)

adbe_operating_income = extract_annual_fact(
    adbe_us_gaap,
    "OperatingIncomeLoss"
)

print("R&D rows:", len(adbe_rd))
print("Operating income rows:", len(adbe_operating_income))

KeyError: 'ResearchAndDevelopmentExpense'

In [101]:
adbe_rd_candidates = [
    key for key in adbe_us_gaap.keys()
    if (
        "Research" in key
        or "Development" in key
        or "Technology" in key
    )
]

adbe_rd_candidates

['ResearchAndDevelopmentExpenseSoftwareExcludingAcquiredInProcessCost']

In [102]:
adbe_operating_income = extract_annual_fact(
    adbe_us_gaap,
    "OperatingIncomeLoss"
)

print("Operating income rows:", len(adbe_operating_income))

print(
    adbe_operating_income[
        ["fiscal_year", "val"]
    ]
)

Operating income rows: 5
   fiscal_year         val
0         2021  5802000000
1         2022  6098000000
2         2023  6650000000
3         2024  6741000000
4         2025  8706000000


In [103]:
print("Revenue rows:", len(adbe_revenue_fallback))

print(
    adbe_revenue_fallback[
        ["fiscal_year", "val"]
    ]
)

Revenue rows: 5
   fiscal_year          val
0         2021  15785000000
1         2022  17606000000
2         2023  19409000000
3         2024  21505000000
4         2025  23769000000


In [104]:
adbe_rd_fallback = extract_annual_fact(
    adbe_us_gaap,
    "ResearchAndDevelopmentExpenseSoftwareExcludingAcquiredInProcessCost"
)

print("R&D rows:", len(adbe_rd_fallback))
print()

print(
    adbe_rd_fallback[
        ["fiscal_year", "val"]
    ]
)

R&D rows: 5

   fiscal_year         val
0         2021  2540000000
1         2022  2987000000
2         2023  3473000000
3         2024  3944000000
4         2025  4294000000


In [105]:
print("Revenue rows:", len(adbe_revenue_fallback))

adbe_operating_income = extract_annual_fact(
    adbe_us_gaap,
    "OperatingIncomeLoss"
)

print("Operating income rows:", len(adbe_operating_income))

Revenue rows: 5
Operating income rows: 5


In [106]:
adbe_capex = extract_annual_fact(
    adbe_us_gaap,
    "PaymentsToAcquirePropertyPlantAndEquipment"
)

print("Capex rows:", len(adbe_capex))
print()

print(
    adbe_capex[
        ["fiscal_year", "val"]
    ]
)

Capex rows: 5

   fiscal_year        val
0         2021  348000000
1         2022  442000000
2         2023  360000000
3         2024  183000000
4         2025  179000000


In [107]:
adbe_assets = extract_point_in_time_fact(
    adbe_us_gaap,
    "Assets"
)

adbe_cash = extract_point_in_time_fact(
    adbe_us_gaap,
    "CashAndCashEquivalentsAtCarryingValue"
)

print("Assets rows:", len(adbe_assets))
print("Cash rows:", len(adbe_cash))

Assets rows: 5
Cash rows: 5


In [108]:
print("Assets:")
print(adbe_assets[["fiscal_year", "val"]])

print("\nCash:")
print(adbe_cash[["fiscal_year", "val"]])

Assets:
   fiscal_year          val
0         2021  27241000000
1         2022  27165000000
2         2023  29779000000
3         2024  30230000000
4         2025  29496000000

Cash:
   fiscal_year         val
0         2021  3844000000
1         2022  4236000000
2         2023  7141000000
3         2024  7613000000
4         2025  5431000000


In [109]:
adbe_debt_candidates = [
    key for key in adbe_us_gaap.keys()
    if (
        "LongTermDebt" in key
        or "ShortTermBorrowings" in key
        or "DebtCurrent" in key
    )
]

adbe_debt_candidates

['DebtCurrent',
 'LongTermDebt',
 'LongTermDebtAndCapitalLeaseObligations',
 'LongTermDebtAndCapitalLeaseObligationsCurrent',
 'LongTermDebtNoncurrent',
 'ProceedsFromIssuanceOfLongTermDebt',
 'ProceedsFromIssuanceOfSeniorLongTermDebt',
 'ProceedsFromRepaymentsOfLongTermDebtAndCapitalSecurities',
 'RepaymentsOfLongTermDebt',
 'RepaymentsOfLongTermDebtAndCapitalSecurities',
 'LongTermDebtCurrent']

In [110]:
adbe_debt_current = extract_point_in_time_fact(
    adbe_us_gaap,
    "LongTermDebtCurrent"
)

adbe_debt_noncurrent = extract_point_in_time_fact(
    adbe_us_gaap,
    "LongTermDebtNoncurrent"
)

print("Current debt rows:", len(adbe_debt_current))
print("Noncurrent debt rows:", len(adbe_debt_noncurrent))

Current debt rows: 2
Noncurrent debt rows: 0


In [111]:
for tag in [
    "DebtCurrent",
    "LongTermDebt",
    "LongTermDebtAndCapitalLeaseObligations",
    "LongTermDebtAndCapitalLeaseObligationsCurrent",
    "LongTermDebtCurrent"
]:
    print("\nTAG:", tag)

    fact = adbe_us_gaap[tag]
    df = pd.DataFrame(fact["units"]["USD"])

    df["end"] = pd.to_datetime(df["end"])

    check = df[
        (df["form"] == "10-K") &
        (df["fp"] == "FY") &
        (df["end"].dt.year.between(2021, 2025))
    ].copy()

    print(
        check[
            ["end", "val", "fy", "filed"]
        ]
    )


TAG: DebtCurrent
          end         val    fy       filed
29 2021-12-03           0  2022  2023-01-17
33 2022-12-02   500000000  2022  2023-01-17
37 2022-12-02   500000000  2023  2024-01-17
41 2023-12-01           0  2023  2024-01-17
45 2023-12-01           0  2024  2025-01-13
49 2024-11-29  1499000000  2024  2025-01-13
53 2024-11-29  1499000000  2025  2026-01-15
57 2025-11-28           0  2025  2026-01-15

TAG: LongTermDebt
          end         val    fy       filed
41 2021-12-03  4123000000  2021  2022-01-21
45 2021-12-03  4123000000  2022  2023-01-17
49 2022-12-02  3629000000  2022  2023-01-17
53 2022-12-02  3629000000  2023  2024-01-17
57 2023-12-01  3634000000  2023  2024-01-17
61 2023-12-01  3634000000  2024  2025-01-13
65 2024-11-29  4129000000  2024  2025-01-13
69 2024-11-29  4129000000  2025  2026-01-15
73 2025-11-28  6210000000  2025  2026-01-15

TAG: LongTermDebtAndCapitalLeaseObligations
          end         val    fy       filed
49 2021-12-03  4123000000  2021  2022-

In [112]:
adbe_long_term_debt = extract_point_in_time_fact(
    adbe_us_gaap,
    "LongTermDebt"
)

adbe_debt_current = extract_point_in_time_fact(
    adbe_us_gaap,
    "DebtCurrent"
)

adbe_debt = (
    adbe_long_term_debt[
        ["fiscal_year", "val"]
    ]
    .rename(columns={"val": "long_term_debt"})
    .merge(
        adbe_debt_current[
            ["fiscal_year", "val"]
        ].rename(columns={"val": "debt_current"}),
        on="fiscal_year",
        how="outer"
    )
)

adbe_debt["debt_current"] = (
    adbe_debt["debt_current"].fillna(0)
)

adbe_debt["total_debt"] = (
    adbe_debt["long_term_debt"]
    + adbe_debt["debt_current"]
)

print(adbe_debt)

print("\nMissing values:")
print(adbe_debt.isna().sum())

   fiscal_year  long_term_debt  debt_current  total_debt
0         2021      4123000000             0  4123000000
1         2022      3629000000     500000000  4129000000
2         2023      3634000000             0  3634000000
3         2024      4129000000    1499000000  5628000000
4         2025      6210000000             0  6210000000

Missing values:
fiscal_year       0
long_term_debt    0
debt_current      0
total_debt        0
dtype: int64


In [113]:
meta_row = test_companies[
    test_companies["ticker"] == "META"
].iloc[0]

meta_facts = get_company_facts(
    meta_row["cik"]
)

meta_us_gaap = meta_facts["facts"]["us-gaap"]

print("Company:", meta_facts["entityName"])
print("CIK:", meta_facts["cik"])
print("US-GAAP concepts:", len(meta_us_gaap))

Company: Meta Platforms, Inc.
CIK: 1326801
US-GAAP concepts: 462


In [114]:
meta_revenue = extract_annual_fact(
    meta_us_gaap,
    "RevenueFromContractWithCustomerExcludingAssessedTax"
)

meta_rd = extract_annual_fact(
    meta_us_gaap,
    "ResearchAndDevelopmentExpense"
)

meta_operating_income = extract_annual_fact(
    meta_us_gaap,
    "OperatingIncomeLoss"
)

print("Revenue rows:", len(meta_revenue))
print("R&D rows:", len(meta_rd))
print("Operating income rows:", len(meta_operating_income))

Revenue rows: 5
R&D rows: 5
Operating income rows: 5


In [115]:
meta_capex = extract_annual_fact(
    meta_us_gaap,
    "PaymentsToAcquirePropertyPlantAndEquipment"
)

print("Capex rows:", len(meta_capex))
print()

print(
    meta_capex[
        ["fiscal_year", "val"]
    ]
)

Capex rows: 8

   fiscal_year          val
0         2021  18567000000
1         2021  18690000000
2         2022  31431000000
3         2023  27266000000
4         2022  31186000000
5         2023  27045000000
6         2024  37256000000
7         2025  69691000000


In [116]:
meta_capex_detail = extract_annual_fact(
    meta_us_gaap,
    "PaymentsToAcquirePropertyPlantAndEquipment"
)

meta_capex_detail[
    [
        "fiscal_year",
        "start",
        "end",
        "val",
        "filed"
    ]
]

,fiscal_year,start,end,val,filed
0,2021,2021-01-01,2021-12-31,18567000000,2022-02-03
1,2021,2021-01-01,2021-12-31,18690000000,2023-02-02
2,2022,2022-01-01,2022-12-31,31431000000,2023-02-02
3,2023,2023-01-01,2023-12-31,27266000000,2024-02-02
4,2022,2022-01-01,2022-12-31,31186000000,2025-01-30
5,2023,2023-01-01,2023-12-31,27045000000,2025-01-30
6,2024,2024-01-01,2024-12-31,37256000000,2025-01-30
7,2025,2025-01-01,2025-12-31,69691000000,2026-01-29


In [118]:
meta_capex = extract_annual_fact(
    meta_us_gaap,
    "PaymentsToAcquirePropertyPlantAndEquipment"
)

print("Capex rows:", len(meta_capex))
print()

print(
    meta_capex[
        ["fiscal_year", "val"]
    ]
)

Capex rows: 5

   fiscal_year          val
0         2021  18567000000
1         2022  31431000000
2         2023  27266000000
3         2024  37256000000
4         2025  69691000000


In [119]:
meta_assets = extract_point_in_time_fact(
    meta_us_gaap,
    "Assets"
)

meta_cash = extract_point_in_time_fact(
    meta_us_gaap,
    "CashAndCashEquivalentsAtCarryingValue"
)

print("Assets rows:", len(meta_assets))
print("Cash rows:", len(meta_cash))

Assets rows: 5
Cash rows: 5


In [120]:
print("Assets:")
print(meta_assets[["fiscal_year", "val"]])

print("\nCash:")
print(meta_cash[["fiscal_year", "val"]])

Assets:
   fiscal_year           val
0         2021  165987000000
1         2022  185727000000
2         2023  229623000000
3         2024  276054000000
4         2025  366021000000

Cash:
   fiscal_year          val
0         2021  16601000000
1         2022  14681000000
2         2023  41862000000
3         2024  43889000000
4         2025  35873000000


In [121]:
meta_debt_candidates = [
    key for key in meta_us_gaap.keys()
    if (
        "LongTermDebt" in key
        or "ShortTermBorrowings" in key
        or "DebtCurrent" in key
    )
]

meta_debt_candidates

['LongTermDebtNoncurrent',
 'ProceedsFromRepaymentsOfOtherLongTermDebt',
 'RepaymentsOfLongTermDebt',
 'LongTermDebtMaturitiesRepaymentsOfPrincipalAfterYearFive',
 'LongTermDebtMaturitiesRepaymentsOfPrincipalInYearFive',
 'ProceedsFromIssuanceOfLongTermDebt',
 'InterestExpenseLongTermDebt',
 'LongTermDebt',
 'LongTermDebtMaturitiesRepaymentsOfPrincipalInYearFour',
 'LongTermDebtMaturitiesRepaymentsOfPrincipalInYearThree',
 'LongTermDebtMaturitiesRepaymentsOfPrincipalInNextTwelveMonths',
 'LongTermDebtMaturitiesRepaymentsOfPrincipalInYearTwo',
 'LongTermDebtMaturitiesRepaymentsOfPrincipalRemainderOfFiscalYear',
 'LongTermDebtCurrent']

In [122]:
meta_debt_current = extract_point_in_time_fact(
    meta_us_gaap,
    "LongTermDebtCurrent"
)

meta_debt_noncurrent = extract_point_in_time_fact(
    meta_us_gaap,
    "LongTermDebtNoncurrent"
)

print("Current debt rows:", len(meta_debt_current))
print("Noncurrent debt rows:", len(meta_debt_noncurrent))

Current debt rows: 0
Noncurrent debt rows: 5


In [123]:
meta_long_term_debt = extract_point_in_time_fact(
    meta_us_gaap,
    "LongTermDebt"
)

print("LongTermDebt rows:", len(meta_long_term_debt))
print()

print(
    meta_long_term_debt[
        ["fiscal_year", "val"]
    ]
)

LongTermDebt rows: 4

   fiscal_year          val
0         2022   9923000000
1         2023  18385000000
2         2024  28826000000
3         2025  58744000000


In [124]:
print("LongTermDebt:")
print(
    meta_long_term_debt[
        ["fiscal_year", "val"]
    ]
)

print("\nLongTermDebtNoncurrent:")
print(
    meta_debt_noncurrent[
        ["fiscal_year", "val"]
    ]
)

LongTermDebt:
   fiscal_year          val
0         2022   9923000000
1         2023  18385000000
2         2024  28826000000
3         2025  58744000000

LongTermDebtNoncurrent:
   fiscal_year          val
0         2021            0
1         2022   9923000000
2         2023  18385000000
3         2024  28826000000
4         2025  58744000000


In [125]:
meta_debt = (
    meta_debt_noncurrent[
        ["fiscal_year", "val"]
    ]
    .rename(columns={"val": "total_debt"})
)

print(meta_debt)

print("\nMissing values:")
print(meta_debt.isna().sum())

   fiscal_year   total_debt
0         2021            0
1         2022   9923000000
2         2023  18385000000
3         2024  28826000000
4         2025  58744000000

Missing values:
fiscal_year    0
total_debt     0
dtype: int64


## 5-Company SEC Stress Test Complete

Test companies:
- Apple
- Microsoft
- NVIDIA
- Adobe
- Meta

Key findings:
- Flow variables should be identified by actual reporting period (`start`, `end`) rather than SEC `fy`.
- When the same reporting period appears in multiple 10-K filings with different values, keep the earliest-filed observation.
- Revenue, R&D, capex, and debt require fallback tag logic.
- Missing observations are retained as missing rather than manually invented.
- Debt requires company-specific fallback logic because SEC tagging is inconsistent across issuers.

In [277]:
REVENUE_TAGS = [
    "RevenueFromContractWithCustomerExcludingAssessedTax",
    "Revenues",
]

RD_TAGS = [
    "ResearchAndDevelopmentExpense",
    "ResearchAndDevelopmentExpenseExcludingAcquiredInProcessCost",
    "ResearchAndDevelopmentExpenseSoftwareExcludingAcquiredInProcessCost",
]

CAPEX_TAGS = [
    "PaymentsToAcquirePropertyPlantAndEquipment",
    "PaymentsToAcquireProductiveAssets",
]

print("Fallback tag lists created.")

Fallback tag lists created.


In [278]:
def extract_with_fallback(
    us_gaap_facts,
    candidate_tags,
    extractor_func
):
    best_df = pd.DataFrame()
    best_tag = None

    for tag in candidate_tags:
        try:
            df = extractor_func(
                us_gaap_facts,
                tag
            )

            if len(df) > len(best_df):
                best_df = df
                best_tag = tag

        except KeyError:
            continue

    return best_df, best_tag

In [279]:
test_df, test_tag = extract_with_fallback(
    adbe_us_gaap,
    REVENUE_TAGS,
    extract_annual_fact
)

print("Tag used:", test_tag)
print("Rows:", len(test_df))
print(test_df[["fiscal_year", "val"]])

Tag used: Revenues
Rows: 5
   fiscal_year          val
0         2021  15785000000
1         2022  17606000000
2         2023  19409000000
3         2024  21505000000
4         2025  23769000000


In [280]:
nvda_capex_test, nvda_capex_tag = extract_with_fallback(
    nvda_us_gaap,
    CAPEX_TAGS,
    extract_annual_fact
)

print("Tag used:", nvda_capex_tag)
print("Rows:", len(nvda_capex_test))
print(nvda_capex_test[["fiscal_year", "val"]])

Tag used: PaymentsToAcquireProductiveAssets
Rows: 4
   fiscal_year         val
0         2022   976000000
1         2023  1833000000
2         2024  1069000000
3         2025  3236000000


In [281]:
OPERATING_INCOME_TAGS = [
    "OperatingIncomeLoss",
]

ASSETS_TAGS = [
    "Assets",
]

CASH_TAGS = [
    "CashAndCashEquivalentsAtCarryingValue",
]

In [282]:
meta_assets_test, meta_assets_tag = extract_with_fallback(
    meta_us_gaap,
    ASSETS_TAGS,
    extract_point_in_time_fact
)

print("Tag used:", meta_assets_tag)
print("Rows:", len(meta_assets_test))

Tag used: Assets
Rows: 5


In [283]:
def extract_total_debt(us_gaap_facts):

    # Helper: safely extract a point-in-time tag
    def safe_extract(tag):
        try:
            return extract_point_in_time_fact(
                us_gaap_facts,
                tag
            )
        except KeyError:
            return pd.DataFrame()

    # Rule 1:
    # LongTermDebtCurrent + LongTermDebtNoncurrent
    current = safe_extract("LongTermDebtCurrent")
    noncurrent = safe_extract("LongTermDebtNoncurrent")

    if len(current) > 0 and len(noncurrent) > 0:
        debt = (
            current[["fiscal_year", "val"]]
            .rename(columns={"val": "debt_current"})
            .merge(
                noncurrent[["fiscal_year", "val"]]
                .rename(columns={"val": "debt_noncurrent"}),
                on="fiscal_year",
                how="outer"
            )
        )

        debt["total_debt"] = (
            debt["debt_current"].fillna(0)
            + debt["debt_noncurrent"].fillna(0)
        )

        return (
            debt[["fiscal_year", "total_debt"]],
            "LongTermDebtCurrent + LongTermDebtNoncurrent"
        )

    # Rule 2:
    # LongTermDebt + DebtCurrent if available
    long_term = safe_extract("LongTermDebt")
    debt_current = safe_extract("DebtCurrent")

    if len(long_term) > 0:
        debt = (
            long_term[["fiscal_year", "val"]]
            .rename(columns={"val": "long_term_debt"})
        )

        if len(debt_current) > 0:
            debt = debt.merge(
                debt_current[["fiscal_year", "val"]]
                .rename(columns={"val": "debt_current"}),
                on="fiscal_year",
                how="outer"
            )

            debt["debt_current"] = (
                debt["debt_current"].fillna(0)
            )

            debt["total_debt"] = (
                debt["long_term_debt"]
                + debt["debt_current"]
            )

            rule = "LongTermDebt + DebtCurrent"

        else:
            debt["total_debt"] = debt["long_term_debt"]
            rule = "LongTermDebt"

        return (
            debt[["fiscal_year", "total_debt"]],
            rule
        )

    # Rule 3:
    # LongTermDebtNoncurrent alone
    if len(noncurrent) > 0:
        debt = (
            noncurrent[["fiscal_year", "val"]]
            .rename(columns={"val": "total_debt"})
        )

        return debt, "LongTermDebtNoncurrent"

    # Rule 4:
    # UnsecuredLongTermDebt
    unsecured = safe_extract("UnsecuredLongTermDebt")

    if len(unsecured) > 0:
        debt = (
            unsecured[["fiscal_year", "val"]]
            .rename(columns={"val": "total_debt"})
        )

        return debt, "UnsecuredLongTermDebt"

    # Rule 5:
    # DebtInstrumentCarryingAmount
    carrying = safe_extract("DebtInstrumentCarryingAmount")

    if len(carrying) > 0:
        debt = (
            carrying[["fiscal_year", "val"]]
            .rename(columns={"val": "total_debt"})
        )

        return debt, "DebtInstrumentCarryingAmount"

    return pd.DataFrame(), None

In [284]:
adbe_debt_test, adbe_debt_rule = extract_total_debt(
    adbe_us_gaap
)

print("Rule used:", adbe_debt_rule)
print("Rows:", len(adbe_debt_test))
print(adbe_debt_test)

Rule used: LongTermDebt + DebtCurrent
Rows: 5
   fiscal_year  total_debt
0         2021  4123000000
1         2022  4129000000
2         2023  3634000000
3         2024  5628000000
4         2025  6210000000


In [285]:
meta_debt_test, meta_debt_rule = extract_total_debt(
    meta_us_gaap
)

print("Rule used:", meta_debt_rule)
print("Rows:", len(meta_debt_test))
print(meta_debt_test)

Rule used: LongTermDebt
Rows: 4
   fiscal_year   total_debt
0         2022   9923000000
1         2023  18385000000
2         2024  28826000000
3         2025  58744000000


In [286]:
msft_debt_test, msft_debt_rule = extract_total_debt(
    msft_us_gaap
)

print("Rule used:", msft_debt_rule)
print("Rows:", len(msft_debt_test))
print(msft_debt_test)

Rule used: LongTermDebtCurrent + LongTermDebtNoncurrent
Rows: 5
   fiscal_year   total_debt
0         2021  58146000000
1         2022  49781000000
2         2023  47237000000
3         2024  44937000000
4         2025  43151000000


In [287]:
def build_company_sec_table(
    us_gaap_facts,
    start_year=2021,
    end_year=2025
):
    years = pd.DataFrame({
        "fiscal_year": range(start_year, end_year + 1)
    })

    revenue, revenue_tag = extract_with_fallback(
        us_gaap_facts,
        REVENUE_TAGS,
        extract_annual_fact
    )

    rd, rd_tag = extract_with_fallback(
        us_gaap_facts,
        RD_TAGS,
        extract_annual_fact
    )

    capex, capex_tag = extract_with_fallback(
        us_gaap_facts,
        CAPEX_TAGS,
        extract_annual_fact
    )

    operating_income, operating_income_tag = extract_with_fallback(
        us_gaap_facts,
        OPERATING_INCOME_TAGS,
        extract_annual_fact
    )

    assets, assets_tag = extract_with_fallback(
        us_gaap_facts,
        ASSETS_TAGS,
        extract_point_in_time_fact
    )

    cash, cash_tag = extract_with_fallback(
        us_gaap_facts,
        CASH_TAGS,
        extract_point_in_time_fact
    )

    debt, debt_rule = extract_total_debt(
        us_gaap_facts
    )

    series = [
        (revenue, "revenue"),
        (rd, "rd"),
        (capex, "capex"),
        (operating_income, "operating_income"),
        (assets, "assets"),
        (cash, "cash"),
        (debt, "total_debt"),
    ]

    result = years.copy()

    for df, column_name in series:
        if len(df) == 0:
            result[column_name] = pd.NA
            continue

        if column_name == "total_debt":
            temp = df[
                ["fiscal_year", "total_debt"]
            ].copy()
        else:
            temp = (
                df[["fiscal_year", "val"]]
                .rename(columns={"val": column_name})
            )

        result = result.merge(
            temp,
            on="fiscal_year",
            how="left"
        )

    tags_used = {
        "revenue": revenue_tag,
        "rd": rd_tag,
        "capex": capex_tag,
        "operating_income": operating_income_tag,
        "assets": assets_tag,
        "cash": cash_tag,
        "debt": debt_rule,
    }

    return result, tags_used

In [288]:
nvda_sec_table, nvda_tags = build_company_sec_table(
    nvda_us_gaap
)

print(nvda_sec_table)

print("\nTags/rules used:")
print(nvda_tags)

print("\nMissing values:")
print(nvda_sec_table.isna().sum())

   fiscal_year       revenue           rd         capex  operating_income  \
0         2021   16675000000   3924000000           NaN        4532000000   
1         2022   26914000000   5268000000  9.760000e+08       10041000000   
2         2023   26974000000   7339000000  1.833000e+09        4224000000   
3         2024   60922000000   8675000000  1.069000e+09       32972000000   
4         2025  130497000000  12914000000  3.236000e+09       81453000000   

         assets        cash   total_debt  
0   28791000000   847000000   6963000000  
1   44187000000  1990000000  10946000000  
2   41182000000  3389000000  10953000000  
3   65728000000  7280000000   9709000000  
4  111601000000  8589000000   8463000000  

Tags/rules used:
{'revenue': 'Revenues', 'rd': 'ResearchAndDevelopmentExpense', 'capex': 'PaymentsToAcquireProductiveAssets', 'operating_income': 'OperatingIncomeLoss', 'assets': 'Assets', 'cash': 'CashAndCashEquivalentsAtCarryingValue', 'debt': 'LongTermDebtCurrent + LongTermD

In [289]:
nvda_sec_table, nvda_tags = build_company_sec_table(
    nvda_us_gaap
)

print(nvda_sec_table)

print("\nTags/rules used:")
print(nvda_tags)

print("\nMissing values:")
print(nvda_sec_table.isna().sum())

   fiscal_year       revenue           rd         capex  operating_income  \
0         2021   16675000000   3924000000           NaN        4532000000   
1         2022   26914000000   5268000000  9.760000e+08       10041000000   
2         2023   26974000000   7339000000  1.833000e+09        4224000000   
3         2024   60922000000   8675000000  1.069000e+09       32972000000   
4         2025  130497000000  12914000000  3.236000e+09       81453000000   

         assets        cash   total_debt  
0   28791000000   847000000   6963000000  
1   44187000000  1990000000  10946000000  
2   41182000000  3389000000  10953000000  
3   65728000000  7280000000   9709000000  
4  111601000000  8589000000   8463000000  

Tags/rules used:
{'revenue': 'Revenues', 'rd': 'ResearchAndDevelopmentExpense', 'capex': 'PaymentsToAcquireProductiveAssets', 'operating_income': 'OperatingIncomeLoss', 'assets': 'Assets', 'cash': 'CashAndCashEquivalentsAtCarryingValue', 'debt': 'LongTermDebtCurrent + LongTermD

In [290]:
test_us_gaap = {
    "AAPL": us_gaap_facts,
    "MSFT": msft_us_gaap,
    "NVDA": nvda_us_gaap,
    "ADBE": adbe_us_gaap,
    "META": meta_us_gaap,
}

test_tables = []
test_tag_log = {}

for ticker, us_gaap in test_us_gaap.items():
    table, tags = build_company_sec_table(us_gaap)

    table["ticker"] = ticker

    test_tables.append(table)
    test_tag_log[ticker] = tags

five_company_sec = pd.concat(
    test_tables,
    ignore_index=True
)

five_company_sec = five_company_sec[
    [
        "ticker",
        "fiscal_year",
        "revenue",
        "rd",
        "capex",
        "operating_income",
        "assets",
        "cash",
        "total_debt",
    ]
]

print("Rows:", len(five_company_sec))
print()
print(five_company_sec)

Rows: 25

   ticker  fiscal_year       revenue           rd         capex  \
0    AAPL         2021  365817000000  21914000000  1.108500e+10   
1    AAPL         2022  394328000000  26251000000  1.070800e+10   
2    AAPL         2023  383285000000  29915000000  1.095900e+10   
3    AAPL         2024  391035000000  31370000000  9.447000e+09   
4    AAPL         2025  416161000000  34550000000  1.271500e+10   
5    MSFT         2021  168088000000  20716000000  2.062200e+10   
6    MSFT         2022  198270000000  24512000000  2.388600e+10   
7    MSFT         2023  211915000000  27195000000  2.810700e+10   
8    MSFT         2024  245122000000  29510000000  4.447700e+10   
9    MSFT         2025  281724000000  32488000000  6.455100e+10   
10   NVDA         2021   16675000000   3924000000           NaN   
11   NVDA         2022   26914000000   5268000000  9.760000e+08   
12   NVDA         2023   26974000000   7339000000  1.833000e+09   
13   NVDA         2024   60922000000   8675000000  1

In [291]:
[name for name in globals() if "gaap" in name.lower()]

['us_gaap_facts',
 'msft_us_gaap',
 'nvda_us_gaap',
 'adbe_us_gaap',
 'meta_us_gaap',
 'test_us_gaap',
 'us_gaap',
 'cdns_us_gaap',
 'trmb_us_gaap',
 'ntap_us_gaap',
 'adsk_us_gaap']

In [292]:
print("\nMissing values:")
print(five_company_sec.isna().sum())

print("\nTag/rule log:")
for ticker, tags in test_tag_log.items():
    print(ticker, tags)


Missing values:
ticker              0
fiscal_year         0
revenue             0
rd                  0
capex               1
operating_income    0
assets              0
cash                0
total_debt          1
dtype: int64

Tag/rule log:
AAPL {'revenue': 'RevenueFromContractWithCustomerExcludingAssessedTax', 'rd': 'ResearchAndDevelopmentExpense', 'capex': 'PaymentsToAcquirePropertyPlantAndEquipment', 'operating_income': 'OperatingIncomeLoss', 'assets': 'Assets', 'cash': 'CashAndCashEquivalentsAtCarryingValue', 'debt': 'LongTermDebtCurrent + LongTermDebtNoncurrent'}
MSFT {'revenue': 'RevenueFromContractWithCustomerExcludingAssessedTax', 'rd': 'ResearchAndDevelopmentExpense', 'capex': 'PaymentsToAcquirePropertyPlantAndEquipment', 'operating_income': 'OperatingIncomeLoss', 'assets': 'Assets', 'cash': 'CashAndCashEquivalentsAtCarryingValue', 'debt': 'LongTermDebtCurrent + LongTermDebtNoncurrent'}
NVDA {'revenue': 'Revenues', 'rd': 'ResearchAndDevelopmentExpense', 'capex': 'PaymentsToA

In [293]:
five_company_sec.to_csv(
    "../data/raw/five_company_sec_test.csv",
    index=False
)

print("Saved five_company_sec_test.csv")

Saved five_company_sec_test.csv


In [294]:
tag_log_rows = []

for ticker, tags in test_tag_log.items():
    row = {"ticker": ticker}
    row.update(tags)
    tag_log_rows.append(row)

tag_log_df = pd.DataFrame(tag_log_rows)

tag_log_df.to_csv(
    "../data/raw/five_company_tag_log.csv",
    index=False
)

print("Saved five_company_tag_log.csv")

Saved five_company_tag_log.csv


In [295]:
def build_company_from_cik(cik):
    company_facts = get_company_facts(cik)
    us_gaap = company_facts["facts"]["us-gaap"]

    table, tags = build_company_sec_table(us_gaap)

    return table, tags, company_facts["entityName"]

In [296]:
msft_cik = test_companies.loc[
    test_companies["ticker"] == "MSFT",
    "cik"
].iloc[0]

msft_table_from_cik, msft_tags_from_cik, msft_name = build_company_from_cik(
    msft_cik
)

print("Company:", msft_name)
print(msft_table_from_cik)
print()
print(msft_tags_from_cik)

Company: MICROSOFT CORP
   fiscal_year       revenue           rd        capex  operating_income  \
0         2021  168088000000  20716000000  20622000000       69916000000   
1         2022  198270000000  24512000000  23886000000       83383000000   
2         2023  211915000000  27195000000  28107000000       88523000000   
3         2024  245122000000  29510000000  44477000000      109433000000   
4         2025  281724000000  32488000000  64551000000      128528000000   

         assets         cash   total_debt  
0  333779000000  14224000000  58146000000  
1  364840000000  13931000000  49781000000  
2  411976000000  34704000000  47237000000  
3  512163000000  18315000000  44937000000  
4  619003000000  30242000000  43151000000  

{'revenue': 'RevenueFromContractWithCustomerExcludingAssessedTax', 'rd': 'ResearchAndDevelopmentExpense', 'capex': 'PaymentsToAcquirePropertyPlantAndEquipment', 'operating_income': 'OperatingIncomeLoss', 'assets': 'Assets', 'cash': 'CashAndCashEquivalent

In [ ]:
full_tables = []
full_tag_log = []
failures = []

for i, row in company_universe.iterrows():
    ticker = row["ticker"]
    cik = row["cik"]

    try:
        table, tags, company_name = build_company_from_cik(cik)

        table["ticker"] = ticker
        table["company"] = row["company"]
        table["sector"] = row["sector"]
        table["industry"] = row["industry"]

        full_tables.append(table)

        tag_row = {
            "ticker": ticker,
            "company": row["company"],
            **tags
        }
        full_tag_log.append(tag_row)

        print(f"{i + 1}/{len(company_universe)} OK: {ticker}")

    except Exception as e:
        failures.append({
            "ticker": ticker,
            "cik": cik,
            "error": str(e)
        })

        print(f"{i + 1}/{len(company_universe)} FAILED: {ticker} - {e}")

1/97 OK: ACN
2/97 OK: ADBE
3/97 OK: AMD
4/97 OK: AKAM
5/97 OK: GOOGL
6/97 OK: GOOG
7/97 OK: APH
8/97 OK: ADI
9/97 OK: AAPL
10/97 OK: AMAT
11/97 OK: APP
12/97 OK: ANET
13/97 OK: T
14/97 OK: ADSK
15/97 OK: AVGO
16/97 OK: CDNS
17/97 OK: CDW
18/97 OK: CHTR
19/97 OK: CIEN
20/97 OK: CSCO
21/97 OK: CTSH
22/97 OK: COHR
23/97 OK: CMCSA
24/97 OK: GLW


In [ ]:
full_sec_data = pd.concat(
    full_tables,
    ignore_index=True
)

full_tag_log_df = pd.DataFrame(full_tag_log)
failures_df = pd.DataFrame(failures)

print("\nSuccessful companies:", full_sec_data["ticker"].nunique())
print("Total rows:", len(full_sec_data))
print("Failures:", len(failures_df))


Successful companies: 96
Total rows: 1195
Failures: 1


In [ ]:
row_counts = (
    full_sec_data
    .groupby("ticker")
    .size()
    .sort_values(ascending=False)
)

print(row_counts.head(20))

ticker
TRMB    260
TDY     260
CDNS     68
MPWR     21
MRVL     20
NTAP     20
TKO      20
DELL     13
NWS      13
NWSA     13
WDAY     13
FLEX     12
T        12
PANW      9
ANET      9
AKAM      8
TTWO      8
ECHO      8
TYL       8
VRSN      8
dtype: int64


In [ ]:
duplicate_years = (
    full_sec_data
    .groupby(["ticker", "fiscal_year"])
    .size()
    .reset_index(name="count")
)

duplicate_years = duplicate_years[
    duplicate_years["count"] > 1
]

print("Duplicate ticker-years:", len(duplicate_years))
print(duplicate_years.head(20))

Duplicate ticker-years: 37
    ticker  fiscal_year  count
16    ADSK         2022      4
21    AKAM         2022      4
37    ANET         2023      2
38    ANET         2024      4
48     APP         2024      2
50    AVGO         2021      2
56    CDNS         2022     64
110   DELL         2021      8
113   DELL         2024      2
121   ECHO         2022      4
137   FLEX         2023      8
168    GEN         2024      2
191    HPQ         2022      2
242    LYV         2023      2
258   MPWR         2024      2
259   MPWR         2025     16
261   MRVL         2022     16
290   NTAP         2021     16
302    NWS         2023      2
303    NWS         2024      8


In [ ]:
print(failures_df)

  ticker      cik                                              error
0    ACN  1467373  HTTPSConnectionPool(host='data.sec.gov', port=...


In [ ]:
cdns_row = company_universe[
    company_universe["ticker"] == "CDNS"
].iloc[0]

cdns_facts = get_company_facts(
    cdns_row["cik"]
)

cdns_us_gaap = cdns_facts["facts"]["us-gaap"]

In [ ]:
cdns_revenue, _ = extract_with_fallback(
    cdns_us_gaap,
    REVENUE_TAGS,
    extract_annual_fact
)

cdns_rd, _ = extract_with_fallback(
    cdns_us_gaap,
    RD_TAGS,
    extract_annual_fact
)

cdns_capex, _ = extract_with_fallback(
    cdns_us_gaap,
    CAPEX_TAGS,
    extract_annual_fact
)

cdns_oi, _ = extract_with_fallback(
    cdns_us_gaap,
    OPERATING_INCOME_TAGS,
    extract_annual_fact
)

cdns_assets, _ = extract_with_fallback(
    cdns_us_gaap,
    ASSETS_TAGS,
    extract_point_in_time_fact
)

cdns_cash, _ = extract_with_fallback(
    cdns_us_gaap,
    CASH_TAGS,
    extract_point_in_time_fact
)

cdns_debt, _ = extract_total_debt(
    cdns_us_gaap
)

In [ ]:
for name, df in {
    "revenue": cdns_revenue,
    "rd": cdns_rd,
    "capex": cdns_capex,
    "operating_income": cdns_oi,
    "assets": cdns_assets,
    "cash": cdns_cash,
    "debt": cdns_debt,
}.items():

    print("\n", name.upper())

    if df.empty or "fiscal_year" not in df.columns:
        print("No usable rows")
    else:
        print(df.groupby("fiscal_year").size())


 REVENUE
fiscal_year
2021    1
2022    2
2023    1
2024    1
2025    1
dtype: int64

 RD
fiscal_year
2021    1
2022    2
2023    1
2024    1
2025    1
dtype: int64

 CAPEX
fiscal_year
2021    1
2022    2
2023    1
2024    1
2025    1
dtype: int64

 OPERATING_INCOME
fiscal_year
2021    1
2022    2
2023    1
2024    1
2025    1
dtype: int64

 ASSETS
fiscal_year
2021    1
2022    2
2023    1
2024    1
2025    1
dtype: int64

 CASH
fiscal_year
2021    1
2022    2
2023    1
2024    1
2025    1
dtype: int64

 DEBT
No usable rows


In [ ]:
for name, df in {
    "revenue": cdns_revenue,
    "rd": cdns_rd,
    "capex": cdns_capex,
    "operating_income": cdns_oi,
}.items():

    dup = df[df["fiscal_year"] == 2022]

    print("\n", name.upper())
    print(
        dup[
            ["fiscal_year", "start", "end", "val", "filed"]
        ]
    )


 REVENUE
   fiscal_year      start        end         val       filed
1         2022 2021-01-03 2022-01-01  2988244000  2022-02-22
2         2022 2022-01-02 2022-12-31  3561718000  2023-02-13

 RD
   fiscal_year      start        end         val       filed
1         2022 2021-01-03 2022-01-01  1134277000  2022-02-22
2         2022 2022-01-02 2022-12-31  1251544000  2023-02-13

 CAPEX
   fiscal_year      start        end        val       filed
1         2022 2021-01-03 2022-01-01   65298000  2022-02-22
2         2022 2022-01-02 2022-12-31  123215000  2023-02-13

 OPERATING_INCOME
   fiscal_year      start        end         val       filed
1         2022 2021-01-03 2022-01-01   779089000  2022-02-22
2         2022 2022-01-02 2022-12-31  1073686000  2023-02-13


In [ ]:
cdns_raw = pd.DataFrame(
    cdns_us_gaap[
        "RevenueFromContractWithCustomerExcludingAssessedTax"
    ]["units"]["USD"]
)

cdns_raw["start"] = pd.to_datetime(cdns_raw["start"])
cdns_raw["end"] = pd.to_datetime(cdns_raw["end"])

cdns_raw[
    (cdns_raw["form"] == "10-K") &
    (cdns_raw["fp"] == "FY")
][
    ["fy", "start", "end", "val", "filed"]
].tail(12)

,fy,start,end,val,filed
54,2021,2021-01-03,2022-01-01,2988244000,2022-02-22
55,2022,2021-01-03,2022-01-01,2988244000,2023-02-13
56,2023,2021-01-03,2022-01-01,2988244000,2024-02-14
67,2022,2022-01-02,2022-12-31,3561718000,2023-02-13
68,2023,2022-01-02,2022-12-31,3561718000,2024-02-14
69,2024,2022-01-02,2022-12-31,3561718000,2025-02-21
80,2023,2023-01-01,2023-12-31,4089986000,2024-02-14
81,2024,2023-01-01,2023-12-31,4089986000,2025-02-21
82,2025,2023-01-01,2023-12-31,4089986000,2026-02-19
93,2024,2024-01-01,2024-12-31,4641264000,2025-02-21


In [ ]:
cdns_revenue_test = extract_annual_fact(
    cdns_us_gaap,
    "RevenueFromContractWithCustomerExcludingAssessedTax"
)

print(
    cdns_revenue_test[
        ["fiscal_year", "start", "end", "val"]
    ]
)

   fiscal_year      start        end         val
0         2021 2021-01-03 2022-01-01  2988244000
1         2022 2022-01-02 2022-12-31  3561718000
2         2023 2023-01-01 2023-12-31  4089986000
3         2024 2024-01-01 2024-12-31  4641264000
4         2025 2025-01-01 2025-12-31  5296759000


In [ ]:
nvda_revenue_test = extract_annual_fact(
    nvda_us_gaap,
    "Revenues"
)

print(
    nvda_revenue_test[
        ["fiscal_year", "start", "end", "val"]
    ]
)

   fiscal_year      start        end           val
0         2022 2019-01-28 2020-01-26   10918000000
1         2022 2020-01-27 2021-01-31   16675000000
2         2022 2021-02-01 2022-01-30   26914000000
3         2023 2022-01-31 2023-01-29   26974000000
4         2024 2023-01-30 2024-01-28   60922000000
5         2025 2024-01-29 2025-01-26  130497000000


In [ ]:
cdns_revenue_test = extract_annual_fact(
    cdns_us_gaap,
    "RevenueFromContractWithCustomerExcludingAssessedTax"
)

print("CADENCE")
print(
    cdns_revenue_test[
        ["fiscal_year", "start", "end", "val"]
    ]
)

CADENCE
   fiscal_year      start        end         val
0         2021 2021-01-03 2022-01-01  2988244000
1         2022 2022-01-02 2022-12-31  3561718000
2         2023 2023-01-01 2023-12-31  4089986000
3         2024 2024-01-01 2024-12-31  4641264000
4         2025 2025-01-01 2025-12-31  5296759000


In [ ]:
nvda_revenue_test = extract_annual_fact(
    nvda_us_gaap,
    "Revenues"
)

print("NVIDIA")
print(
    nvda_revenue_test[
        ["fiscal_year", "start", "end", "val"]
    ]
)

NVIDIA
   fiscal_year      start        end           val
0         2021 2020-01-27 2021-01-31   16675000000
1         2022 2021-02-01 2022-01-30   26914000000
2         2023 2022-01-31 2023-01-29   26974000000
3         2024 2023-01-30 2024-01-28   60922000000
4         2025 2024-01-29 2025-01-26  130497000000


In [ ]:
full_tables = []
full_tag_log = []
failures = []

In [ ]:
for i, row in company_universe.iterrows():
    ticker = row["ticker"]
    cik = row["cik"]

    try:
        table, tags, company_name = build_company_from_cik(cik)

        table["ticker"] = ticker
        table["company"] = row["company"]
        table["sector"] = row["sector"]
        table["industry"] = row["industry"]

        full_tables.append(table)

        tag_row = {
            "ticker": ticker,
            "company": row["company"],
            **tags
        }
        full_tag_log.append(tag_row)

        print(f"{i + 1}/{len(company_universe)} OK: {ticker}")

    except Exception as e:
        failures.append({
            "ticker": ticker,
            "cik": cik,
            "error": str(e)
        })

        print(f"{i + 1}/{len(company_universe)} FAILED: {ticker} - {e}")

1/97 OK: ACN
2/97 OK: ADBE
3/97 OK: AMD
4/97 OK: AKAM
5/97 OK: GOOGL
6/97 OK: GOOG
7/97 OK: APH
8/97 OK: ADI
9/97 OK: AAPL
10/97 OK: AMAT
11/97 OK: APP
12/97 OK: ANET
13/97 OK: T
14/97 OK: ADSK
15/97 OK: AVGO
16/97 OK: CDNS
17/97 OK: CDW
18/97 OK: CHTR
19/97 OK: CIEN
20/97 OK: CSCO
21/97 OK: CTSH
22/97 OK: COHR
23/97 OK: CMCSA
24/97 OK: GLW
25/97 OK: CRWD
26/97 OK: DDOG
27/97 OK: DELL
28/97 OK: ECHO
29/97 OK: P
30/97 OK: FFIV
31/97 OK: FICO
32/97 OK: FSLR
33/97 OK: FLEX
34/97 OK: FTNT
35/97 OK: FOXA
36/97 OK: FOX
37/97 OK: IT
38/97 OK: GEN
39/97 OK: GDDY
40/97 OK: HPE
41/97 OK: HPQ
42/97 OK: IBM
43/97 OK: INTC
44/97 OK: INTU
45/97 OK: JBL
46/97 OK: KEYS
47/97 OK: KLAC
48/97 OK: LRCX
49/97 OK: LYV
50/97 OK: LITE
51/97 OK: MRVL
52/97 OK: META
53/97 OK: MCHP
54/97 OK: MU
55/97 OK: MSFT
56/97 OK: MPWR
57/97 OK: MSI
58/97 OK: NTAP
59/97 OK: NFLX
60/97 OK: NWSA
61/97 OK: NWS
62/97 OK: NVDA
63/97 OK: NXPI
64/97 OK: OMC
65/97 OK: ON
66/97 OK: ORCL
67/97 OK: PLTR
68/97 OK: PANW
69/97 OK: PTC
70

In [ ]:
full_sec_data = pd.concat(
    full_tables,
    ignore_index=True
)

full_tag_log_df = pd.DataFrame(full_tag_log)
failures_df = pd.DataFrame(failures)

print("Successful companies:", full_sec_data["ticker"].nunique())
print("Total rows:", len(full_sec_data))
print("Failures:", len(failures_df))

Successful companies: 97
Total rows: 660
Failures: 0


In [ ]:
duplicate_years = (
    full_sec_data
    .groupby(["ticker", "fiscal_year"])
    .size()
    .reset_index(name="count")
)

duplicate_years = duplicate_years[
    duplicate_years["count"] > 1
]

print("Duplicate ticker-years:", len(duplicate_years))

Duplicate ticker-years: 37


In [ ]:
trmb_row = company_universe[
    company_universe["ticker"] == "TRMB"
].iloc[0]

trmb_facts = get_company_facts(
    trmb_row["cik"]
)

trmb_us_gaap = trmb_facts["facts"]["us-gaap"]

trmb_revenue, _ = extract_with_fallback(
    trmb_us_gaap,
    REVENUE_TAGS,
    extract_annual_fact
)

trmb_rd, _ = extract_with_fallback(
    trmb_us_gaap,
    RD_TAGS,
    extract_annual_fact
)

trmb_capex, _ = extract_with_fallback(
    trmb_us_gaap,
    CAPEX_TAGS,
    extract_annual_fact
)

trmb_oi, _ = extract_with_fallback(
    trmb_us_gaap,
    OPERATING_INCOME_TAGS,
    extract_annual_fact
)

trmb_assets, _ = extract_with_fallback(
    trmb_us_gaap,
    ASSETS_TAGS,
    extract_point_in_time_fact
)

trmb_cash, _ = extract_with_fallback(
    trmb_us_gaap,
    CASH_TAGS,
    extract_point_in_time_fact
)

trmb_debt, _ = extract_total_debt(
    trmb_us_gaap
)

In [ ]:
for name, df in {
    "revenue": trmb_revenue,
    "rd": trmb_rd,
    "capex": trmb_capex,
    "operating_income": trmb_oi,
    "assets": trmb_assets,
    "cash": trmb_cash,
    "debt": trmb_debt,
}.items():

    print("\n" + name.upper())

    if df.empty or "fiscal_year" not in df.columns:
        print("No usable rows")
    else:
        print(df.groupby("fiscal_year").size())


REVENUE
fiscal_year
2021    1
2022    1
2023    1
2024    1
2025    1
dtype: int64

RD
fiscal_year
2021    1
2022    1
2023    1
2024    1
2025    1
dtype: int64

CAPEX
fiscal_year
2021    1
2022    1
2023    1
2024    1
2025    1
dtype: int64

OPERATING_INCOME
fiscal_year
2021    1
2022    1
2023    1
2024    1
2025    1
dtype: int64

ASSETS
fiscal_year
2021    2
2022    1
2023    1
2025    1
dtype: int64

CASH
fiscal_year
2021    2
2022    1
2023    1
2025    1
dtype: int64

DEBT
fiscal_year
2021    4
2022    1
2023    1
2025    1
dtype: int64


In [ ]:
for name, df in {
    "revenue": trmb_revenue,
    "rd": trmb_rd,
    "capex": trmb_capex,
    "operating_income": trmb_oi,
    "assets": trmb_assets,
    "cash": trmb_cash,
    "debt": trmb_debt,
}.items():

    if df.empty or "fiscal_year" not in df.columns:
        print(name, "-> no usable rows")
    else:
        counts = df.groupby("fiscal_year").size()
        duplicates = counts[counts > 1]

        print(
            name,
            "rows =", len(df),
            "| duplicate years =",
            duplicates.to_dict()
        )

revenue rows = 5 | duplicate years = {}
rd rows = 5 | duplicate years = {}
capex rows = 5 | duplicate years = {}
operating_income rows = 5 | duplicate years = {}
assets rows = 5 | duplicate years = {2021: 2}
cash rows = 5 | duplicate years = {2021: 2}
debt rows = 7 | duplicate years = {2021: 4}


In [ ]:
print("ASSETS")
print(
    trmb_assets[
        ["fiscal_year", "end", "val", "filed"]
    ]
)

print("\nCASH")
print(
    trmb_cash[
        ["fiscal_year", "end", "val", "filed"]
    ]
)

print("\nDEBT")
print(trmb_debt)

ASSETS
   fiscal_year        end         val       filed
0         2021 2021-01-01  6876900000  2021-02-26
1         2021 2021-12-31  7099600000  2022-02-23
2         2022 2022-12-30  7269000000  2023-02-17
3         2023 2023-12-29  9539300000  2024-02-26
4         2025 2025-01-03  9488300000  2025-04-25

CASH
   fiscal_year        end        val       filed
0         2021 2021-01-01  237700000  2021-02-26
1         2021 2021-12-31  325700000  2022-02-23
2         2022 2022-12-30  271000000  2023-02-17
3         2023 2023-12-29  229800000  2024-02-26
4         2025 2025-01-03  738800000  2025-04-25

DEBT
   fiscal_year    total_debt
0         2021  1.547200e+09
1         2021  1.549000e+09
2         2021  1.291400e+09
3         2021  1.293200e+09
4         2022  1.520000e+09
5         2023  3.066600e+09
6         2025  1.390600e+09


In [ ]:
trmb_assets_test = extract_point_in_time_fact(
    trmb_us_gaap,
    "Assets"
)

trmb_cash_test = extract_point_in_time_fact(
    trmb_us_gaap,
    "CashAndCashEquivalentsAtCarryingValue"
)

trmb_debt_test, trmb_debt_rule = extract_total_debt(
    trmb_us_gaap
)

print("ASSETS")
print(trmb_assets_test)

print("\nCASH")
print(trmb_cash_test)

print("\nDEBT")
print(trmb_debt_test)

print("\nDebt rule:", trmb_debt_rule)

ASSETS
   fiscal_year        end         val       filed
0         2021 2021-12-31  7099600000  2022-02-23
1         2022 2022-12-30  7269000000  2023-02-17
2         2023 2023-12-29  9539300000  2024-02-26
3         2024 2025-01-03  9488300000  2025-04-25
4         2025 2026-01-02  9312000000  2026-02-25

CASH
   fiscal_year        end        val       filed
0         2021 2021-12-31  325700000  2022-02-23
1         2022 2022-12-30  271000000  2023-02-17
2         2023 2023-12-29  229800000  2024-02-26
3         2024 2025-01-03  738800000  2025-04-25
4         2025 2026-01-02  253400000  2026-02-25

DEBT
   fiscal_year    total_debt
0         2021  1.293200e+09
1         2022  1.520000e+09
2         2023  3.066600e+09
3         2024  1.390600e+09
4         2025  1.392200e+09

Debt rule: LongTermDebtCurrent + LongTermDebtNoncurrent


In [ ]:
full_tables = []
full_tag_log = []
failures = []

In [ ]:
for i, row in company_universe.iterrows():
    ticker = row["ticker"]
    cik = row["cik"]

    try:
        table, tags, company_name = build_company_from_cik(cik)

        table["ticker"] = ticker
        table["company"] = row["company"]
        table["sector"] = row["sector"]
        table["industry"] = row["industry"]

        full_tables.append(table)

        tag_row = {
            "ticker": ticker,
            "company": row["company"],
            **tags
        }
        full_tag_log.append(tag_row)

        print(f"{i + 1}/{len(company_universe)} OK: {ticker}")

    except Exception as e:
        failures.append({
            "ticker": ticker,
            "cik": cik,
            "error": str(e)
        })

        print(f"{i + 1}/{len(company_universe)} FAILED: {ticker} - {e}")

1/97 OK: ACN
2/97 OK: ADBE
3/97 OK: AMD
4/97 OK: AKAM
5/97 OK: GOOGL
6/97 OK: GOOG
7/97 OK: APH
8/97 OK: ADI
9/97 OK: AAPL
10/97 OK: AMAT
11/97 OK: APP
12/97 OK: ANET
13/97 OK: T
14/97 OK: ADSK
15/97 OK: AVGO
16/97 OK: CDNS
17/97 OK: CDW
18/97 OK: CHTR
19/97 OK: CIEN
20/97 OK: CSCO
21/97 OK: CTSH
22/97 OK: COHR
23/97 OK: CMCSA
24/97 OK: GLW
25/97 OK: CRWD
26/97 OK: DDOG
27/97 OK: DELL
28/97 OK: ECHO
29/97 OK: P
30/97 OK: FFIV
31/97 OK: FICO
32/97 OK: FSLR
33/97 OK: FLEX
34/97 OK: FTNT
35/97 OK: FOXA
36/97 OK: FOX
37/97 OK: IT
38/97 OK: GEN
39/97 OK: GDDY
40/97 OK: HPE
41/97 OK: HPQ
42/97 OK: IBM
43/97 OK: INTC
44/97 OK: INTU
45/97 OK: JBL
46/97 OK: KEYS
47/97 OK: KLAC
48/97 OK: LRCX
49/97 OK: LYV
50/97 OK: LITE
51/97 OK: MRVL
52/97 OK: META
53/97 OK: MCHP
54/97 OK: MU
55/97 OK: MSFT
56/97 OK: MPWR
57/97 OK: MSI
58/97 OK: NTAP
59/97 OK: NFLX
60/97 OK: NWSA
61/97 OK: NWS
62/97 OK: NVDA
63/97 OK: NXPI
64/97 OK: OMC
65/97 OK: ON
66/97 OK: ORCL
67/97 OK: PLTR
68/97 OK: PANW
69/97 OK: PTC
70

In [ ]:
full_sec_data = pd.concat(
    full_tables,
    ignore_index=True
)

full_tag_log_df = pd.DataFrame(full_tag_log)
failures_df = pd.DataFrame(failures)

print("Successful companies:", full_sec_data["ticker"].nunique())
print("Total rows:", len(full_sec_data))
print("Failures:", len(failures_df))

Successful companies: 97
Total rows: 519
Failures: 0


In [ ]:
duplicate_years = (
    full_sec_data
    .groupby(["ticker", "fiscal_year"])
    .size()
    .reset_index(name="count")
)

duplicate_years = duplicate_years[
    duplicate_years["count"] > 1
]

print("Duplicate ticker-years:", len(duplicate_years))

Duplicate ticker-years: 4


In [ ]:
print(duplicate_years)

    ticker  fiscal_year  count
26    AKAM         2022      4
264   MPWR         2025     16
295   NTAP         2021     16
455    TYL         2021      2


In [ ]:
problem_tickers = duplicate_years["ticker"].unique()

print("Problem tickers:")
print(problem_tickers)

print("\nRow counts:")
print(
    full_sec_data[
        full_sec_data["ticker"].isin(problem_tickers)
    ]
    .groupby("ticker")
    .size()
)

Problem tickers:
<StringArray>
['AKAM', 'MPWR', 'NTAP', 'TYL']
Length: 4, dtype: str

Row counts:
ticker
AKAM     8
MPWR    20
NTAP    20
TYL      6
dtype: int64


In [ ]:
problem_tickers = ["AKAM", "MPWR", "NTAP", "TYL"]

for ticker in problem_tickers:
    row = company_universe[
        company_universe["ticker"] == ticker
    ].iloc[0]

    facts = get_company_facts(row["cik"])
    us_gaap = facts["facts"]["us-gaap"]

    revenue, _ = extract_with_fallback(
        us_gaap, REVENUE_TAGS, extract_annual_fact
    )

    rd, _ = extract_with_fallback(
        us_gaap, RD_TAGS, extract_annual_fact
    )

    capex, _ = extract_with_fallback(
        us_gaap, CAPEX_TAGS, extract_annual_fact
    )

    oi, _ = extract_with_fallback(
        us_gaap, OPERATING_INCOME_TAGS, extract_annual_fact
    )

    assets, _ = extract_with_fallback(
        us_gaap, ASSETS_TAGS, extract_point_in_time_fact
    )

    cash, _ = extract_with_fallback(
        us_gaap, CASH_TAGS, extract_point_in_time_fact
    )

    debt, _ = extract_total_debt(us_gaap)

    print("\n---", ticker, "---")

    for name, df in {
        "revenue": revenue,
        "rd": rd,
        "capex": capex,
        "operating_income": oi,
        "assets": assets,
        "cash": cash,
        "debt": debt,
    }.items():

        if df.empty or "fiscal_year" not in df.columns:
            print(name, "-> no usable rows")
        else:
            counts = df.groupby("fiscal_year").size()
            duplicates = counts[counts > 1]

            print(
                name,
                "rows =", len(df),
                "| duplicates =",
                duplicates.to_dict()
            )


--- AKAM ---
revenue -> no usable rows
rd rows = 5 | duplicates = {}
capex rows = 5 | duplicates = {}
operating_income rows = 5 | duplicates = {}
assets rows = 8 | duplicates = {2022: 4}
cash rows = 5 | duplicates = {}
debt -> no usable rows

--- MPWR ---
revenue rows = 5 | duplicates = {}
rd -> no usable rows
capex rows = 5 | duplicates = {}
operating_income rows = 5 | duplicates = {}
assets rows = 8 | duplicates = {2025: 4}
cash rows = 8 | duplicates = {2025: 4}
debt -> no usable rows

--- NTAP ---
revenue rows = 6 | duplicates = {2021: 2}
rd rows = 6 | duplicates = {2021: 2}
capex rows = 6 | duplicates = {2021: 2}
operating_income rows = 6 | duplicates = {2021: 2}
assets rows = 5 | duplicates = {}
cash rows = 5 | duplicates = {}
debt rows = 5 | duplicates = {}

--- TYL ---
revenue rows = 5 | duplicates = {}
rd rows = 5 | duplicates = {}
capex rows = 5 | duplicates = {}
operating_income rows = 5 | duplicates = {}
assets rows = 5 | duplicates = {}
cash rows = 6 | duplicates = {2021: 

In [ ]:
problem_cases = {
    "AKAM": {
        "assets": 2022,
    },
    "MPWR": {
        "assets": 2025,
        "cash": 2025,
    },
    "NTAP": {
        "revenue": 2021,
        "rd": 2021,
        "capex": 2021,
        "operating_income": 2021,
    },
    "TYL": {
        "cash": 2021,
    },
}

In [ ]:
for ticker, cases in problem_cases.items():
    row = company_universe[
        company_universe["ticker"] == ticker
    ].iloc[0]

    facts = get_company_facts(row["cik"])
    us_gaap = facts["facts"]["us-gaap"]

    variable_map = {}

    revenue, _ = extract_with_fallback(
        us_gaap, REVENUE_TAGS, extract_annual_fact
    )
    rd, _ = extract_with_fallback(
        us_gaap, RD_TAGS, extract_annual_fact
    )
    capex, _ = extract_with_fallback(
        us_gaap, CAPEX_TAGS, extract_annual_fact
    )
    oi, _ = extract_with_fallback(
        us_gaap, OPERATING_INCOME_TAGS, extract_annual_fact
    )
    assets, _ = extract_with_fallback(
        us_gaap, ASSETS_TAGS, extract_point_in_time_fact
    )
    cash, _ = extract_with_fallback(
        us_gaap, CASH_TAGS, extract_point_in_time_fact
    )

    variable_map = {
        "revenue": revenue,
        "rd": rd,
        "capex": capex,
        "operating_income": oi,
        "assets": assets,
        "cash": cash,
    }

    print("\n###", ticker)

    for variable, year in cases.items():
        df = variable_map[variable]

        print("\n", variable.upper(), year)

        if "start" in df.columns:
            print(
                df[df["fiscal_year"] == year][
                    ["fiscal_year", "start", "end", "val", "filed"]
                ]
            )
        else:
            print(
                df[df["fiscal_year"] == year][
                    ["fiscal_year", "end", "val", "filed"]
                ]
            )


### AKAM

 ASSETS 2022
   fiscal_year        end         val       filed
1         2022 2022-12-31  8303400000  2023-02-28
2         2022 2022-09-30  8136481000  2023-02-28
3         2022 2022-06-30  8230752000  2023-02-28
4         2022 2022-03-31  8312389000  2023-02-28

### MPWR

 ASSETS 2025
   fiscal_year        end         val       filed
4         2025 2025-03-31  3705143000  2026-02-27
5         2025 2025-06-30  3872262000  2026-02-27
6         2025 2025-09-30  4107222000  2026-02-27
7         2025 2025-12-31  4194199000  2026-02-27

 CASH 2025
   fiscal_year        end         val       filed
4         2025 2025-03-31   637354000  2026-02-27
5         2025 2025-06-30   787382000  2026-02-27
6         2025 2025-09-30  1081251000  2026-02-27
7         2025 2025-12-31  1099302000  2026-02-27

### NTAP

 REVENUE 2021
   fiscal_year      start        end         val       filed
0         2021 2020-04-25 2021-04-30  5744000000  2021-06-21
3         2021 2020-05-01 2021-04-30  57440

In [ ]:
ntap_row = company_universe[
    company_universe["ticker"] == "NTAP"
].iloc[0]

ntap_facts = get_company_facts(ntap_row["cik"])
ntap_us_gaap = ntap_facts["facts"]["us-gaap"]

ntap_revenue, _ = extract_with_fallback(
    ntap_us_gaap, REVENUE_TAGS, extract_annual_fact
)

ntap_rd, _ = extract_with_fallback(
    ntap_us_gaap, RD_TAGS, extract_annual_fact
)

ntap_capex, _ = extract_with_fallback(
    ntap_us_gaap, CAPEX_TAGS, extract_annual_fact
)

ntap_oi, _ = extract_with_fallback(
    ntap_us_gaap, OPERATING_INCOME_TAGS, extract_annual_fact
)

In [ ]:
for name, df in {
    "revenue": ntap_revenue,
    "rd": ntap_rd,
    "capex": ntap_capex,
    "operating_income": ntap_oi,
}.items():

    print("\n", name.upper())

    print(
        df[df["fiscal_year"] == 2021][
            ["fiscal_year", "start", "end", "val", "filed"]
        ]
    )


 REVENUE
   fiscal_year      start        end         val       filed
0         2021 2020-04-25 2021-04-30  5744000000  2021-06-21
3         2021 2020-05-01 2021-04-30  5744000000  2023-06-14

 RD
   fiscal_year      start        end        val       filed
0         2021 2020-04-25 2021-04-30  881000000  2021-06-21
2         2021 2020-05-01 2021-04-30  881000000  2023-06-14

 CAPEX
   fiscal_year      start        end        val       filed
0         2021 2020-04-25 2021-04-30  162000000  2021-06-21
2         2021 2020-05-01 2021-04-30  162000000  2023-06-14

 OPERATING_INCOME
   fiscal_year      start        end         val       filed
0         2021 2020-04-25 2021-04-30  1031000000  2021-06-21
2         2021 2020-05-01 2021-04-30  1031000000  2023-06-14


In [ ]:
ntap_revenue, _ = extract_with_fallback(
    ntap_us_gaap,
    REVENUE_TAGS,
    extract_annual_fact
)

ntap_rd, _ = extract_with_fallback(
    ntap_us_gaap,
    RD_TAGS,
    extract_annual_fact
)

ntap_capex, _ = extract_with_fallback(
    ntap_us_gaap,
    CAPEX_TAGS,
    extract_annual_fact
)

ntap_oi, _ = extract_with_fallback(
    ntap_us_gaap,
    OPERATING_INCOME_TAGS,
    extract_annual_fact
)

for name, df in {
    "revenue": ntap_revenue,
    "rd": ntap_rd,
    "capex": ntap_capex,
    "operating_income": ntap_oi,
}.items():
    print(
        name,
        "rows =", len(df),
        "| duplicate years =",
        df.groupby("fiscal_year").size()
          .loc[lambda x: x > 1]
          .to_dict()
    )

revenue rows = 5 | duplicate years = {}
rd rows = 5 | duplicate years = {}
capex rows = 5 | duplicate years = {}
operating_income rows = 5 | duplicate years = {}


In [ ]:
full_tables = []
full_tag_log = []
failures = []

In [ ]:
full_sec_data = pd.concat(
    full_tables,
    ignore_index=True
)

full_tag_log_df = pd.DataFrame(full_tag_log)
failures_df = pd.DataFrame(failures)

print("Successful companies:", full_sec_data["ticker"].nunique())
print("Total rows:", len(full_sec_data))
print("Failures:", len(failures_df))

ValueError: No objects to concatenate

In [ ]:
print("full_tables type:", type(full_tables))
print("number of items:", len(full_tables))

for i, item in enumerate(full_tables[:10]):
    print(i, type(item))

full_tables type: <class 'list'>
number of items: 0


In [ ]:
bad_items = []

for i, item in enumerate(full_tables):
    if not isinstance(item, pd.DataFrame):
        bad_items.append((i, type(item)))

print("Non-DataFrame items:")
print(bad_items[:20])
print("Count:", len(bad_items))

Non-DataFrame items:
[]
Count: 0


In [ ]:
full_tables = []
full_tag_log = []
failures = []

In [ ]:
for i, row in company_universe.iterrows():
    ticker = row["ticker"]
    cik = row["cik"]

    try:
        table, tags, company_name = build_company_from_cik(cik)

        table["ticker"] = ticker
        table["company"] = row["company"]
        table["sector"] = row["sector"]
        table["industry"] = row["industry"]

        full_tables.append(table)

        tag_row = {
            "ticker": ticker,
            "company": row["company"],
            **tags
        }
        full_tag_log.append(tag_row)

        print(f"{i + 1}/{len(company_universe)} OK: {ticker}")

    except Exception as e:
        failures.append({
            "ticker": ticker,
            "cik": cik,
            "error": str(e)
        })

        print(f"{i + 1}/{len(company_universe)} FAILED: {ticker} - {e}")

1/97 OK: ACN
2/97 OK: ADBE
3/97 OK: AMD
4/97 OK: AKAM
5/97 OK: GOOGL
6/97 OK: GOOG
7/97 OK: APH
8/97 OK: ADI
9/97 OK: AAPL
10/97 OK: AMAT
11/97 OK: APP
12/97 OK: ANET
13/97 OK: T
14/97 OK: ADSK
15/97 OK: AVGO
16/97 OK: CDNS
17/97 OK: CDW
18/97 OK: CHTR
19/97 OK: CIEN
20/97 OK: CSCO
21/97 OK: CTSH
22/97 OK: COHR
23/97 OK: CMCSA
24/97 OK: GLW
25/97 OK: CRWD
26/97 OK: DDOG
27/97 OK: DELL
28/97 OK: ECHO
29/97 OK: P
30/97 OK: FFIV
31/97 OK: FICO
32/97 OK: FSLR
33/97 OK: FLEX
34/97 OK: FTNT
35/97 OK: FOXA
36/97 OK: FOX
37/97 OK: IT
38/97 OK: GEN
39/97 OK: GDDY
40/97 OK: HPE
41/97 OK: HPQ
42/97 OK: IBM
43/97 OK: INTC
44/97 OK: INTU
45/97 OK: JBL
46/97 OK: KEYS
47/97 OK: KLAC
48/97 OK: LRCX
49/97 OK: LYV
50/97 OK: LITE
51/97 OK: MRVL
52/97 OK: META
53/97 OK: MCHP
54/97 OK: MU
55/97 OK: MSFT
56/97 OK: MPWR
57/97 OK: MSI
58/97 OK: NTAP
59/97 OK: NFLX
60/97 OK: NWSA
61/97 OK: NWS
62/97 OK: NVDA
63/97 OK: NXPI
64/97 OK: OMC
65/97 OK: ON
66/97 OK: ORCL
67/97 OK: PLTR
68/97 OK: PANW
69/97 OK: PTC
70

In [ ]:
print(len(full_tables))

97


In [ ]:
full_sec_data = pd.concat(
    full_tables,
    ignore_index=True
)

full_tag_log_df = pd.DataFrame(full_tag_log)
failures_df = pd.DataFrame(failures)

print("Successful companies:", full_sec_data["ticker"].nunique())
print("Total rows:", len(full_sec_data))
print("Failures:", len(failures_df))

Successful companies: 97
Total rows: 485
Failures: 0


In [ ]:
duplicate_years = (
    full_sec_data
    .groupby(["ticker", "fiscal_year"])
    .size()
    .reset_index(name="count")
)

duplicate_years = duplicate_years[
    duplicate_years["count"] > 1
]

print("Duplicate ticker-years:", len(duplicate_years))

Duplicate ticker-years: 0


In [ ]:
missing_summary = full_sec_data[
    [
        "revenue",
        "rd",
        "capex",
        "operating_income",
        "assets",
        "cash",
        "total_debt",
    ]
].isna().sum()

print(missing_summary)

revenue              22
rd                  121
capex                41
operating_income     42
assets               12
cash                 35
total_debt          142
dtype: int64


In [ ]:
missing_by_company = (
    full_sec_data[
        [
            "ticker",
            "revenue",
            "rd",
            "capex",
            "operating_income",
            "assets",
            "cash",
            "total_debt",
        ]
    ]
    .set_index("ticker")
    .isna()
    .groupby(level=0)
    .sum()
)

print(
    missing_by_company[
        missing_by_company.sum(axis=1) > 0
    ].sort_values(
        by=list(missing_by_company.columns),
        ascending=False
    )
)

        revenue  rd  capex  operating_income  assets  cash  total_debt
ticker                                                                
AKAM          5   0      0                 0       0     0           5
CRWD          5   0      0                 0       0     0           0
SKYD          3   5      3                 3       3     3           5
Q             2   2      2                 5       3     3           4
SNDK          2   2      2                 2       3     3           3
...         ...  ..    ...               ...     ...   ...         ...
NXPI          0   0      0                 0       0     0           2
SMCI          0   0      0                 0       0     0           2
ADI           0   0      0                 0       0     0           1
TTWO          0   0      0                 0       0     0           1
TYL           0   0      0                 0       0     0           1

[61 rows x 7 columns]


In [ ]:
variables = [
    "revenue",
    "rd",
    "capex",
    "operating_income",
    "assets",
    "cash",
    "total_debt",
]

for variable in variables:
    missing_per_company = (
        full_sec_data
        .groupby("ticker")[variable]
        .apply(lambda x: x.isna().sum())
    )

    fully_missing = (missing_per_company == 5).sum()
    partially_missing = (
        (missing_per_company > 0) &
        (missing_per_company < 5)
    ).sum()

    print(
        variable,
        "| fully missing companies:", fully_missing,
        "| partially missing companies:", partially_missing
    )

revenue | fully missing companies: 2 | partially missing companies: 8
rd | fully missing companies: 23 | partially missing companies: 4
capex | fully missing companies: 5 | partially missing companies: 8
operating_income | fully missing companies: 7 | partially missing companies: 4
assets | fully missing companies: 0 | partially missing companies: 5
cash | fully missing companies: 3 | partially missing companies: 8
total_debt | fully missing companies: 22 | partially missing companies: 13


In [ ]:
rd_missing = (
    full_sec_data
    .groupby("ticker")["rd"]
    .apply(lambda x: x.isna().sum())
)

debt_missing = (
    full_sec_data
    .groupby("ticker")["total_debt"]
    .apply(lambda x: x.isna().sum())
)

print("Completely missing R&D:")
print(rd_missing[rd_missing == 5].index.tolist())

print("\nCompletely missing debt:")
print(debt_missing[debt_missing == 5].index.tolist())

Completely missing R&D:
['CDW', 'CHTR', 'CMCSA', 'CTSH', 'DIS', 'FLEX', 'FOX', 'FOXA', 'GDDY', 'IT', 'LYV', 'MPWR', 'NWS', 'NWSA', 'OMC', 'ON', 'SKYD', 'T', 'TER', 'TKO', 'TMUS', 'VZ', 'WDC']

Completely missing debt:
['ACN', 'ADSK', 'AKAM', 'ANET', 'APH', 'CDNS', 'CMCSA', 'DDOG', 'GLW', 'KLAC', 'LYV', 'MPWR', 'NOW', 'OMC', 'PANW', 'PTC', 'RDDT', 'SKYD', 'TER', 'TWLO', 'VRSN', 'VZ']


In [ ]:
rd_check_tickers = [
    "MPWR",
    "ON",
    "TER",
    "WDC",
    "FLEX",
]

for ticker in rd_check_tickers:
    row = company_universe[
        company_universe["ticker"] == ticker
    ].iloc[0]

    facts = get_company_facts(row["cik"])
    us_gaap = facts["facts"]["us-gaap"]

    candidates = [
        tag for tag in us_gaap.keys()
        if (
            "Research" in tag
            or "Development" in tag
        )
    ]

    print("\n---", ticker, "---")
    print(candidates)


--- MPWR ---
['DeferredTaxAssetsTaxCreditCarryforwardsResearch', 'EffectiveIncomeTaxRateReconciliationNondeductibleExpenseResearchAndDevelopment', 'EffectiveIncomeTaxRateReconciliationTaxCreditsResearch', 'ResearchAndDevelopmentExpense', 'ResearchAndDevelopmentExpenseExcludingAcquiredInProcessCost', 'IncomeTaxReconciliationTaxCreditsResearch']

--- ON ---
['DeferredTaxAssetsInProcessResearchAndDevelopment', 'EffectiveIncomeTaxRateReconciliationNondeductibleExpenseResearchAndDevelopment', 'ResearchAndDevelopmentExpenseExcludingAcquiredInProcessCost', 'ResearchAndDevelopmentInProcess', 'EffectiveIncomeTaxRateReconciliationTaxCreditsResearch', 'IncomeTaxReconciliationTaxCreditsResearch']

--- TER ---
['DeferredTaxAssetsInProcessResearchAndDevelopment', 'EffectiveIncomeTaxRateReconciliationTaxCreditsResearch', 'IncomeTaxReconciliationTaxCreditsResearch', 'ResearchAndDevelopmentExpenseExcludingAcquiredInProcessCost', 'ResearchAndDevelopmentInProcess']

--- WDC ---
['EffectiveIncomeTaxRateR

In [ ]:
rd_check_tickers = [
    "MPWR",
    "ON",
    "TER",
    "WDC",
    "FLEX",
]

rd_candidate_tags = [
    "ResearchAndDevelopmentExpense",
    "ResearchAndDevelopmentExpenseExcludingAcquiredInProcessCost",
    "ResearchAndDevelopmentExpenseSoftwareExcludingAcquiredInProcessCost",
]

for ticker in rd_check_tickers:
    row = company_universe[
        company_universe["ticker"] == ticker
    ].iloc[0]

    facts = get_company_facts(row["cik"])
    us_gaap = facts["facts"]["us-gaap"]

    print("\n---", ticker, "---")

    for tag in rd_candidate_tags:
        try:
            df = extract_annual_fact(
                us_gaap,
                tag
            )

            print(tag, "->", len(df), "rows")

        except KeyError:
            print(tag, "-> tag not present")


--- MPWR ---
ResearchAndDevelopmentExpense -> 0 rows
ResearchAndDevelopmentExpenseExcludingAcquiredInProcessCost -> 5 rows
ResearchAndDevelopmentExpenseSoftwareExcludingAcquiredInProcessCost -> tag not present

--- ON ---
ResearchAndDevelopmentExpense -> tag not present
ResearchAndDevelopmentExpenseExcludingAcquiredInProcessCost -> 5 rows
ResearchAndDevelopmentExpenseSoftwareExcludingAcquiredInProcessCost -> tag not present

--- TER ---
ResearchAndDevelopmentExpense -> tag not present
ResearchAndDevelopmentExpenseExcludingAcquiredInProcessCost -> 5 rows
ResearchAndDevelopmentExpenseSoftwareExcludingAcquiredInProcessCost -> tag not present

--- WDC ---
ResearchAndDevelopmentExpense -> tag not present
ResearchAndDevelopmentExpenseExcludingAcquiredInProcessCost -> 5 rows
ResearchAndDevelopmentExpenseSoftwareExcludingAcquiredInProcessCost -> tag not present

--- FLEX ---
ResearchAndDevelopmentExpense -> 0 rows
ResearchAndDevelopmentExpenseExcludingAcquiredInProcessCost -> tag not present


In [ ]:
rd_check_tickers = [
    "MPWR",
    "ON",
    "TER",
    "WDC",
    "FLEX",
]

rd_candidate_tags = [
    "ResearchAndDevelopmentExpense",
    "ResearchAndDevelopmentExpenseExcludingAcquiredInProcessCost",
    "ResearchAndDevelopmentExpenseSoftwareExcludingAcquiredInProcessCost",
]

for ticker in rd_check_tickers:
    row = company_universe[
        company_universe["ticker"] == ticker
    ].iloc[0]

    facts = get_company_facts(row["cik"])
    us_gaap = facts["facts"]["us-gaap"]

    print("\n---", ticker, "---")

    for tag in rd_candidate_tags:
        try:
            df = extract_annual_fact(
                us_gaap,
                tag
            )

            print(tag, "->", len(df), "rows")

        except KeyError:
            print(tag, "-> tag not present")


--- MPWR ---
ResearchAndDevelopmentExpense -> 0 rows
ResearchAndDevelopmentExpenseExcludingAcquiredInProcessCost -> 5 rows
ResearchAndDevelopmentExpenseSoftwareExcludingAcquiredInProcessCost -> tag not present

--- ON ---
ResearchAndDevelopmentExpense -> tag not present
ResearchAndDevelopmentExpenseExcludingAcquiredInProcessCost -> 5 rows
ResearchAndDevelopmentExpenseSoftwareExcludingAcquiredInProcessCost -> tag not present

--- TER ---
ResearchAndDevelopmentExpense -> tag not present
ResearchAndDevelopmentExpenseExcludingAcquiredInProcessCost -> 5 rows
ResearchAndDevelopmentExpenseSoftwareExcludingAcquiredInProcessCost -> tag not present

--- WDC ---
ResearchAndDevelopmentExpense -> tag not present
ResearchAndDevelopmentExpenseExcludingAcquiredInProcessCost -> 5 rows
ResearchAndDevelopmentExpenseSoftwareExcludingAcquiredInProcessCost -> tag not present

--- FLEX ---
ResearchAndDevelopmentExpense -> 0 rows
ResearchAndDevelopmentExpenseExcludingAcquiredInProcessCost -> tag not present


In [ ]:
full_tables = []
full_tag_log = []
failures = []

In [ ]:
for i, row in company_universe.iterrows():
    ticker = row["ticker"]
    cik = row["cik"]

    try:
        table, tags, company_name = build_company_from_cik(cik)

        table["ticker"] = ticker
        table["company"] = row["company"]
        table["sector"] = row["sector"]
        table["industry"] = row["industry"]

        full_tables.append(table)

        tag_row = {
            "ticker": ticker,
            "company": row["company"],
            **tags
        }
        full_tag_log.append(tag_row)

        print(f"{i + 1}/{len(company_universe)} OK: {ticker}")

    except Exception as e:
        failures.append({
            "ticker": ticker,
            "cik": cik,
            "error": str(e)
        })

        print(f"{i + 1}/{len(company_universe)} FAILED: {ticker} - {e}")

1/97 OK: ACN
2/97 OK: ADBE
3/97 OK: AMD
4/97 OK: AKAM
5/97 OK: GOOGL
6/97 OK: GOOG
7/97 OK: APH
8/97 OK: ADI
9/97 OK: AAPL
10/97 OK: AMAT
11/97 OK: APP
12/97 OK: ANET
13/97 OK: T
14/97 OK: ADSK
15/97 OK: AVGO
16/97 OK: CDNS
17/97 OK: CDW
18/97 OK: CHTR
19/97 OK: CIEN
20/97 OK: CSCO
21/97 OK: CTSH
22/97 OK: COHR
23/97 OK: CMCSA
24/97 OK: GLW
25/97 OK: CRWD
26/97 OK: DDOG
27/97 OK: DELL
28/97 OK: ECHO
29/97 OK: P
30/97 OK: FFIV
31/97 OK: FICO
32/97 OK: FSLR
33/97 OK: FLEX
34/97 OK: FTNT
35/97 OK: FOXA
36/97 OK: FOX
37/97 OK: IT
38/97 OK: GEN
39/97 OK: GDDY
40/97 OK: HPE
41/97 OK: HPQ
42/97 OK: IBM
43/97 OK: INTC
44/97 OK: INTU
45/97 OK: JBL
46/97 OK: KEYS
47/97 OK: KLAC
48/97 OK: LRCX
49/97 OK: LYV
50/97 OK: LITE
51/97 OK: MRVL
52/97 OK: META
53/97 OK: MCHP
54/97 OK: MU
55/97 OK: MSFT
56/97 OK: MPWR
57/97 OK: MSI
58/97 OK: NTAP
59/97 OK: NFLX
60/97 OK: NWSA
61/97 OK: NWS
62/97 OK: NVDA
63/97 OK: NXPI
64/97 OK: OMC
65/97 OK: ON
66/97 OK: ORCL
67/97 OK: PLTR
68/97 OK: PANW
69/97 OK: PTC
70

In [ ]:
full_sec_data = pd.concat(
    full_tables,
    ignore_index=True
)

full_tag_log_df = pd.DataFrame(full_tag_log)
failures_df = pd.DataFrame(failures)

print("Successful companies:", full_sec_data["ticker"].nunique())
print("Total rows:", len(full_sec_data))
print("Failures:", len(failures_df))

missing_summary = full_sec_data[
    [
        "revenue",
        "rd",
        "capex",
        "operating_income",
        "assets",
        "cash",
        "total_debt",
    ]
].isna().sum()

print("\nMissing values:")
print(missing_summary)

Successful companies: 97
Total rows: 485
Failures: 0

Missing values:
revenue              22
rd                  101
capex                41
operating_income     42
assets               12
cash                 35
total_debt          142
dtype: int64


In [ ]:
full_sec_data.to_csv(
    "../data/raw/full_sec_data_2021_2025.csv",
    index=False
)

full_tag_log_df.to_csv(
    "../data/raw/full_sec_tag_log.csv",
    index=False
)

print("Saved full SEC dataset and tag log.")

Saved full SEC dataset and tag log.


In [ ]:
debt_check_tickers = [
    "ADSK",
    "ANET",
    "CDNS",
    "PANW",
    "VRSN",
]

for ticker in debt_check_tickers:
    row = company_universe[
        company_universe["ticker"] == ticker
    ].iloc[0]

    facts = get_company_facts(row["cik"])
    us_gaap = facts["facts"]["us-gaap"]

    candidates = [
        tag for tag in us_gaap.keys()
        if (
            "Debt" in tag
            or "Borrow" in tag
            or "NotesPayable" in tag
            or "CreditFacility" in tag
        )
    ]

    print("\n---", ticker, "---")
    print(candidates)


--- ADSK ---
['AvailableForSaleDebtSecuritiesAmortizedCostBasis', 'AvailableForSaleSecuritiesDebtMaturitiesAfterFiveThroughTenYearsAmortizedCost', 'AvailableForSaleSecuritiesDebtMaturitiesAfterFiveThroughTenYearsFairValue', 'AvailableForSaleSecuritiesDebtMaturitiesAfterOneThroughFiveYearsAmortizedCost', 'AvailableForSaleSecuritiesDebtMaturitiesAfterOneThroughFiveYearsFairValue', 'AvailableForSaleSecuritiesDebtMaturitiesAfterTenYearsAmortizedCost', 'AvailableForSaleSecuritiesDebtMaturitiesAfterTenYearsFairValue', 'AvailableForSaleSecuritiesDebtMaturitiesAmortizedCost', 'AvailableForSaleSecuritiesDebtMaturitiesFairValue', 'AvailableForSaleSecuritiesDebtMaturitiesWithinOneYearAmortizedCost', 'AvailableForSaleSecuritiesDebtMaturitiesWithinOneYearFairValue', 'AvailableForSaleSecuritiesDebtMaturitiesWithoutSingleMaturityDateFairValue', 'AvailableForSaleSecuritiesDebtSecurities', 'DebtInstrumentConvertibleThresholdConsecutiveTradingDays1', 'DebtSecuritiesAvailableForSaleAllowanceForCreditLos

In [ ]:
DEBT_TEST_TAGS = [
    "LongTermDebt",
    "LongTermDebtNoncurrent",
    "LongTermDebtCurrent",
    "ConvertibleDebt",
    "ConvertibleDebtNoncurrent",
    "ConvertibleDebtCurrent",
    "UnsecuredLongTermDebt",
    "UnsecuredDebtCurrent",
    "DebtInstrumentCarryingAmount",
]

In [ ]:
debt_check_tickers = [
    "ADSK",
    "ANET",
    "CDNS",
    "PANW",
    "VRSN",
]

for ticker in debt_check_tickers:
    row = company_universe[
        company_universe["ticker"] == ticker
    ].iloc[0]

    facts = get_company_facts(row["cik"])
    us_gaap = facts["facts"]["us-gaap"]

    print("\n---", ticker, "---")

    for tag in DEBT_TEST_TAGS:
        if tag not in us_gaap:
            continue

        try:
            df = extract_point_in_time_fact(
                us_gaap,
                tag
            )

            print(
                tag,
                "->",
                len(df),
                "rows"
            )

        except Exception as e:
            print(tag, "-> error:", e)


--- ADSK ---
LongTermDebt -> 5 rows

--- ANET ---
LongTermDebtNoncurrent -> 0 rows
LongTermDebtCurrent -> 0 rows
DebtInstrumentCarryingAmount -> 0 rows

--- CDNS ---
LongTermDebt -> 0 rows
LongTermDebtNoncurrent -> 0 rows
LongTermDebtCurrent -> 0 rows
UnsecuredLongTermDebt -> 5 rows
UnsecuredDebtCurrent -> 3 rows
DebtInstrumentCarryingAmount -> 0 rows

--- PANW ---
LongTermDebt -> 3 rows
ConvertibleDebt -> 0 rows
ConvertibleDebtNoncurrent -> 3 rows
ConvertibleDebtCurrent -> 5 rows
DebtInstrumentCarryingAmount -> 3 rows

--- VRSN ---
LongTermDebt -> 0 rows
ConvertibleDebt -> 0 rows
ConvertibleDebtNoncurrent -> 0 rows
ConvertibleDebtCurrent -> 0 rows


In [ ]:
adsk_row = company_universe[
    company_universe["ticker"] == "ADSK"
].iloc[0]

adsk_facts = get_company_facts(adsk_row["cik"])
adsk_us_gaap = adsk_facts["facts"]["us-gaap"]

adsk_debt, adsk_debt_rule = extract_total_debt(
    adsk_us_gaap
)

print("Rule:", adsk_debt_rule)
print("Rows:", len(adsk_debt))
print(adsk_debt)

Rule: LongTermDebt
Rows: 5
   fiscal_year  total_debt
0         2021  1650000000
1         2022  2650000000
2         2023  2300000000
3         2024  2300000000
4         2025  2300000000


In [ ]:
full_tables = []
full_tag_log = []
failures = []

In [ ]:
full_sec_data = pd.concat(
    full_tables,
    ignore_index=True
)

full_tag_log_df = pd.DataFrame(full_tag_log)
failures_df = pd.DataFrame(failures)

print("Successful companies:", full_sec_data["ticker"].nunique())
print("Total rows:", len(full_sec_data))
print("Failures:", len(failures_df))

missing_summary = full_sec_data[
    [
        "revenue",
        "rd",
        "capex",
        "operating_income",
        "assets",
        "cash",
        "total_debt",
    ]
].isna().sum()

print("\nMissing values:")
print(missing_summary)

ValueError: No objects to concatenate

In [ ]:
full_tables = []
full_tag_log = []
failures = []

In [ ]:
for i, row in company_universe.iterrows():
    ticker = row["ticker"]
    cik = row["cik"]

    try:
        table, tags, company_name = build_company_from_cik(cik)

        table["ticker"] = ticker
        table["company"] = row["company"]
        table["sector"] = row["sector"]
        table["industry"] = row["industry"]

        full_tables.append(table)

        tag_row = {
            "ticker": ticker,
            "company": row["company"],
            **tags
        }
        full_tag_log.append(tag_row)

        print(f"{i + 1}/{len(company_universe)} OK: {ticker}")

    except Exception as e:
        failures.append({
            "ticker": ticker,
            "cik": cik,
            "error": str(e)
        })

        print(f"{i + 1}/{len(company_universe)} FAILED: {ticker} - {e}")

1/97 OK: ACN
2/97 OK: ADBE
3/97 OK: AMD
4/97 OK: AKAM
5/97 OK: GOOGL
6/97 OK: GOOG
7/97 OK: APH
8/97 OK: ADI
9/97 OK: AAPL
10/97 OK: AMAT
11/97 OK: APP
12/97 OK: ANET
13/97 OK: T
14/97 OK: ADSK
15/97 OK: AVGO
16/97 OK: CDNS
17/97 OK: CDW
18/97 OK: CHTR
19/97 OK: CIEN
20/97 OK: CSCO
21/97 OK: CTSH
22/97 OK: COHR
23/97 OK: CMCSA
24/97 OK: GLW
25/97 OK: CRWD
26/97 OK: DDOG
27/97 OK: DELL
28/97 OK: ECHO
29/97 OK: P
30/97 OK: FFIV
31/97 OK: FICO
32/97 OK: FSLR
33/97 OK: FLEX
34/97 OK: FTNT
35/97 OK: FOXA
36/97 OK: FOX
37/97 OK: IT
38/97 OK: GEN
39/97 OK: GDDY
40/97 OK: HPE
41/97 OK: HPQ
42/97 OK: IBM
43/97 OK: INTC
44/97 OK: INTU
45/97 OK: JBL
46/97 OK: KEYS
47/97 OK: KLAC
48/97 OK: LRCX
49/97 OK: LYV
50/97 OK: LITE
51/97 OK: MRVL
52/97 OK: META
53/97 OK: MCHP
54/97 OK: MU
55/97 OK: MSFT
56/97 OK: MPWR
57/97 OK: MSI
58/97 OK: NTAP
59/97 OK: NFLX
60/97 OK: NWSA
61/97 OK: NWS
62/97 OK: NVDA
63/97 OK: NXPI
64/97 OK: OMC
65/97 OK: ON
66/97 OK: ORCL
67/97 OK: PLTR
68/97 OK: PANW
69/97 OK: PTC
70

In [ ]:
print(len(full_tables))

97


In [ ]:
full_sec_data = pd.concat(
    full_tables,
    ignore_index=True
)

full_tag_log_df = pd.DataFrame(full_tag_log)
failures_df = pd.DataFrame(failures)

print("Successful companies:", full_sec_data["ticker"].nunique())
print("Total rows:", len(full_sec_data))
print("Failures:", len(failures_df))

missing_summary = full_sec_data[
    [
        "revenue",
        "rd",
        "capex",
        "operating_income",
        "assets",
        "cash",
        "total_debt",
    ]
].isna().sum()

print("\nMissing values:")
print(missing_summary)

Successful companies: 97
Total rows: 485
Failures: 0

Missing values:
revenue              22
rd                  101
capex                41
operating_income     42
assets               12
cash                 35
total_debt          118
dtype: int64


In [ ]:
debt_missing = (
    full_sec_data
    .groupby("ticker")["total_debt"]
    .apply(lambda x: x.isna().sum())
)

print("Fully missing debt companies:")
print(debt_missing[debt_missing == 5].index.tolist())

print("\nPartially missing debt companies:")
print(debt_missing[(debt_missing > 0) & (debt_missing < 5)].sort_values(ascending=False))

Fully missing debt companies:
['ACN', 'AKAM', 'ANET', 'APH', 'CDNS', 'CMCSA', 'DDOG', 'GLW', 'LYV', 'MPWR', 'NOW', 'RDDT', 'SKYD', 'TER', 'VRSN', 'VZ']

Partially missing debt companies:
ticker
ECHO    4
ORCL    4
PLTR    4
Q       4
TTWO    4
SNDK    3
LITE    3
SMCI    2
PANW    2
FFIV    2
NXPI    2
META    1
ADI     1
TKO     1
TYL     1
Name: total_debt, dtype: int64


In [ ]:
DEBT_EXTRA_TAGS = [
    "UnsecuredLongTermDebt",
    "UnsecuredDebtCurrent",
    "ConvertibleDebt",
    "ConvertibleDebtCurrent",
    "ConvertibleDebtNoncurrent",
    "LineOfCreditFacilityAmountOutstanding",
    "DebtInstrumentCarryingAmount",
    "LongTermNotesPayable",
    "NotesPayableCurrent",
]

fully_missing_debt = [
    "ACN", "AKAM", "ANET", "APH", "CDNS", "CMCSA",
    "DDOG", "GLW", "LYV", "MPWR", "NOW", "RDDT",
    "SKYD", "TER", "VRSN", "VZ"
]

for ticker in fully_missing_debt:
    row = company_universe[
        company_universe["ticker"] == ticker
    ].iloc[0]

    facts = get_company_facts(row["cik"])
    us_gaap = facts["facts"]["us-gaap"]

    usable = []

    for tag in DEBT_EXTRA_TAGS:
        if tag not in us_gaap:
            continue

        try:
            df = extract_point_in_time_fact(
                us_gaap,
                tag
            )

            if len(df) > 0:
                usable.append(
                    (tag, len(df))
                )

        except Exception:
            pass

    print(ticker, "->", usable)

ACN -> []
AKAM -> []
ANET -> []
APH -> []
CDNS -> [('UnsecuredLongTermDebt', 5), ('UnsecuredDebtCurrent', 3)]
CMCSA -> []
DDOG -> []
GLW -> []
LYV -> [('DebtInstrumentCarryingAmount', 5)]
MPWR -> []
NOW -> []
RDDT -> []
SKYD -> []
TER -> [('ConvertibleDebt', 2), ('ConvertibleDebtCurrent', 3), ('ConvertibleDebtNoncurrent', 2)]
VRSN -> []
VZ -> []


In [ ]:
for ticker in ["CDNS", "LYV"]:
    row = company_universe[
        company_universe["ticker"] == ticker
    ].iloc[0]

    facts = get_company_facts(row["cik"])
    us_gaap = facts["facts"]["us-gaap"]

    debt, rule = extract_total_debt(us_gaap)

    print("\n", ticker)
    print("Rule:", rule)
    print("Rows:", len(debt))
    print(debt)


 CDNS
Rule: UnsecuredLongTermDebt
Rows: 5
   fiscal_year  total_debt
0         2021   347588000
1         2022   648078000
2         2023   299771000
3         2024  2476183000
4         2025  2480150000

 LYV
Rule: DebtInstrumentCarryingAmount
Rows: 5
   fiscal_year  total_debt
0         2021  5830275000
1         2022  5955343000
2         2023  6643113000
3         2024  6491377000
4         2025  8268659000


In [ ]:
full_tables = []
full_tag_log = []
failures = []

In [ ]:
for i, row in company_universe.iterrows():
    ticker = row["ticker"]
    cik = row["cik"]

    try:
        table, tags, company_name = build_company_from_cik(cik)

        table["ticker"] = ticker
        table["company"] = row["company"]
        table["sector"] = row["sector"]
        table["industry"] = row["industry"]

        full_tables.append(table)

        tag_row = {
            "ticker": ticker,
            "company": row["company"],
            **tags
        }

        full_tag_log.append(tag_row)

        print(
            f"{i + 1}/{len(company_universe)} "
            f"OK: {ticker}"
        )

    except Exception as e:
        failures.append({
            "ticker": ticker,
            "cik": cik,
            "error": str(e)
        })

        print(
            f"{i + 1}/{len(company_universe)} "
            f"FAILED: {ticker} - {e}"
        )

1/97 OK: ACN
2/97 OK: ADBE
3/97 OK: AMD
4/97 OK: AKAM
5/97 OK: GOOGL
6/97 OK: GOOG
7/97 OK: APH
8/97 OK: ADI
9/97 OK: AAPL
10/97 OK: AMAT
11/97 OK: APP
12/97 OK: ANET
13/97 OK: T
14/97 OK: ADSK
15/97 OK: AVGO
16/97 OK: CDNS
17/97 OK: CDW
18/97 OK: CHTR
19/97 OK: CIEN
20/97 OK: CSCO
21/97 OK: CTSH
22/97 OK: COHR
23/97 OK: CMCSA
24/97 OK: GLW
25/97 OK: CRWD
26/97 OK: DDOG
27/97 OK: DELL
28/97 OK: ECHO
29/97 OK: P
30/97 OK: FFIV
31/97 OK: FICO
32/97 OK: FSLR
33/97 OK: FLEX
34/97 OK: FTNT
35/97 OK: FOXA
36/97 OK: FOX
37/97 OK: IT
38/97 OK: GEN
39/97 OK: GDDY
40/97 OK: HPE
41/97 OK: HPQ
42/97 OK: IBM
43/97 OK: INTC
44/97 OK: INTU
45/97 OK: JBL
46/97 OK: KEYS
47/97 OK: KLAC
48/97 OK: LRCX
49/97 OK: LYV
50/97 OK: LITE
51/97 OK: MRVL
52/97 OK: META
53/97 OK: MCHP
54/97 OK: MU
55/97 OK: MSFT
56/97 OK: MPWR
57/97 OK: MSI
58/97 OK: NTAP
59/97 OK: NFLX
60/97 OK: NWSA
61/97 OK: NWS
62/97 OK: NVDA
63/97 OK: NXPI
64/97 OK: OMC
65/97 OK: ON
66/97 OK: ORCL
67/97 OK: PLTR
68/97 OK: PANW
69/97 OK: PTC
70

In [ ]:
full_sec_data = pd.concat(
    full_tables,
    ignore_index=True
)

full_tag_log_df = pd.DataFrame(full_tag_log)
failures_df = pd.DataFrame(failures)

print("Successful companies:", full_sec_data["ticker"].nunique())
print("Total rows:", len(full_sec_data))
print("Failures:", len(failures_df))

missing_summary = full_sec_data[
    [
        "revenue",
        "rd",
        "capex",
        "operating_income",
        "assets",
        "cash",
        "total_debt",
    ]
].isna().sum()

print("\nMissing values:")
print(missing_summary)

Successful companies: 97
Total rows: 485
Failures: 0

Missing values:
revenue              22
rd                  101
capex                41
operating_income     42
assets               12
cash                 35
total_debt          108
dtype: int64


In [ ]:
full_sec_data.to_csv(
    "../data/raw/full_sec_data_2021_2025_final.csv",
    index=False
)

full_tag_log_df.to_csv(
    "../data/raw/full_sec_tag_log_final.csv",
    index=False
)

print("Saved finalized SEC dataset and tag log.")

Saved finalized SEC dataset and tag log.


In [ ]:
analysis_data = full_sec_data.copy()

analysis_data["rd_intensity"] = (
    analysis_data["rd"] / analysis_data["revenue"]
)

analysis_data["capex_intensity"] = (
    analysis_data["capex"] / analysis_data["revenue"]
)

analysis_data["operating_margin"] = (
    analysis_data["operating_income"] / analysis_data["revenue"]
)

analysis_data["leverage"] = (
    analysis_data["total_debt"] / analysis_data["assets"]
)

analysis_data["log_assets"] = np.log(
    analysis_data["assets"]
)

In [ ]:
analysis_data = analysis_data.sort_values(
    ["ticker", "fiscal_year"]
).copy()

analysis_data["revenue_growth"] = (
    analysis_data
    .groupby("ticker")["revenue"]
    .pct_change(fill_method=None)
)

In [ ]:
print(
    analysis_data[
        [
            "ticker",
            "fiscal_year",
            "rd_intensity",
            "capex_intensity",
            "operating_margin",
            "leverage",
            "log_assets",
            "revenue_growth",
        ]
    ].head(15)
)

print("\nMissing values:")
print(
    analysis_data[
        [
            "rd_intensity",
            "capex_intensity",
            "operating_margin",
            "leverage",
            "log_assets",
            "revenue_growth",
        ]
    ].isna().sum()
)

   ticker  fiscal_year rd_intensity capex_intensity operating_margin  \
40   AAPL         2021     0.059904        0.030302         0.297824   
41   AAPL         2022     0.066571        0.027155         0.302887   
42   AAPL         2023     0.078049        0.028592         0.298214   
43   AAPL         2024     0.080223        0.024159         0.315102   
44   AAPL         2025     0.083021        0.030553         0.319708   
0     ACN         2021      0.02213         0.01148         0.150822   
1     ACN         2022     0.018237        0.011657         0.152079   
2     ACN         2023     0.020256        0.008238         0.137415   
3     ACN         2024     0.017727        0.007959         0.147864   
4     ACN         2025     0.011731        0.008612         0.146767   
5    ADBE         2021     0.160912        0.022046         0.367564   
6    ADBE         2022     0.169658        0.025105         0.346359   
7    ADBE         2023     0.178938        0.018548         0.34

In [ ]:
print(
    analysis_data[
        [
            "rd_intensity",
            "capex_intensity",
            "operating_margin",
            "leverage",
            "log_assets",
            "revenue_growth",
        ]
    ].isna().sum()
)

rd_intensity        114
capex_intensity      55
operating_margin     56
leverage            108
log_assets           12
revenue_growth      117
dtype: int64


In [ ]:
analysis_data.to_csv(
    "../data/raw/accounting_analysis_checkpoint.csv",
    index=False
)

print("Saved accounting analysis checkpoint.")

Saved accounting analysis checkpoint.


In [ ]:
import yfinance as yf

In [ ]:
def get_price_history(ticker):
    return yf.download(
        ticker,
        start="2021-01-01",
        end="2026-12-31",
        auto_adjust=False,
        progress=False
    )

In [ ]:
aapl_prices = get_price_history("AAPL")

print(aapl_prices.head())
print()
print(aapl_prices.tail())

Price        Adj Close       Close        High         Low        Open  \
Ticker            AAPL        AAPL        AAPL        AAPL        AAPL   
Date                                                                     
2021-01-04  125.632492  129.410004  133.610001  126.760002  133.520004   
2021-01-05  127.185791  131.009995  131.740005  128.429993  128.889999   
2021-01-06  122.904518  126.599998  131.050003  126.379997  127.720001   
2021-01-07  127.098412  130.919998  131.630005  127.860001  128.360001   
2021-01-08  128.195419  132.050003  132.630005  130.229996  132.429993   

Price          Volume  
Ticker           AAPL  
Date                   
2021-01-04  143301900  
2021-01-05   97664900  
2021-01-06  155088000  
2021-01-07  109578200  
2021-01-08  105158200  

Price        Adj Close       Close        High         Low        Open  \
Ticker            AAPL        AAPL        AAPL        AAPL        AAPL   
Date                                                              

In [ ]:
def extract_sec_shares_outstanding(
    company_facts,
    start_year=2021,
    end_year=2025
):
    dei = company_facts["facts"]["dei"]

    fact = dei["EntityCommonStockSharesOutstanding"]

    unit_key = list(fact["units"].keys())[0]
    df = pd.DataFrame(fact["units"][unit_key])

    df = df[
        (df["form"] == "10-K")
    ].copy()

    df["end"] = pd.to_datetime(df["end"])

    df = (
        df
        .sort_values("filed")
        .drop_duplicates(
            subset=["end"],
            keep="first"
        )
    )

    df["fiscal_year"] = df["fy"]

    df = df[
        df["fiscal_year"].between(
            start_year,
            end_year
        )
    ].copy()

    return df[
        ["fiscal_year", "end", "val", "filed"]
    ].rename(
        columns={
            "end": "shares_date",
            "val": "shares_outstanding"
        }
    ).reset_index(drop=True)

In [ ]:
aapl_shares = extract_shares_outstanding(
    company_facts
)

print(aapl_shares)

NameError: name 'company_facts' is not defined

In [ ]:
aapl_row = company_universe[
    company_universe["ticker"] == "AAPL"
].iloc[0]

aapl_company_facts = get_company_facts(
    aapl_row["cik"]
)

aapl_shares = extract_shares_outstanding(
    aapl_company_facts
)

print(aapl_shares)

   fiscal_year shares_date  shares_outstanding       filed
0         2021  2021-10-15         16406397000  2021-10-29
1         2022  2022-10-14         15908118000  2022-10-28
2         2023  2023-10-20         15552752000  2023-11-03
3         2024  2024-10-18         15115823000  2024-11-01
4         2025  2025-10-17         14776353000  2025-10-31


In [ ]:
def match_prices_to_shares_dates(
    shares_df,
    price_history,
    ticker
):
    prices = price_history.copy()

    # yfinance may return MultiIndex columns
    if isinstance(prices.columns, pd.MultiIndex):
        close = prices["Close"][ticker].copy()
    else:
        close = prices["Close"].copy()

    close.index = pd.to_datetime(close.index)

    rows = []

    for _, row in shares_df.iterrows():
        shares_date = pd.to_datetime(row["shares_date"])

        available = close[
            close.index <= shares_date
        ]

        if len(available) == 0:
            matched_date = pd.NaT
            close_price = np.nan
        else:
            matched_date = available.index[-1]
            close_price = available.iloc[-1]

        rows.append({
            "fiscal_year": row["fiscal_year"],
            "shares_date": shares_date,
            "shares_outstanding": row["shares_outstanding"],
            "price_date": matched_date,
            "close_price": close_price,
        })

    return pd.DataFrame(rows)

In [ ]:
aapl_market = match_prices_to_shares_dates(
    aapl_shares,
    aapl_prices,
    "AAPL"
)

print(aapl_market)

   fiscal_year shares_date  shares_outstanding price_date  close_price
0         2021  2021-10-15         16406397000 2021-10-15   144.839996
1         2022  2022-10-14         15908118000 2022-10-14   138.380005
2         2023  2023-10-20         15552752000 2023-10-20   172.880005
3         2024  2024-10-18         15115823000 2024-10-18   235.000000
4         2025  2025-10-17         14776353000 2025-10-17   252.289993


In [ ]:
aapl_valuation = aapl_market.merge(
    analysis_data[
        analysis_data["ticker"] == "AAPL"
    ][
        [
            "fiscal_year",
            "revenue",
            "cash",
            "total_debt",
        ]
    ],
    on="fiscal_year",
    how="left"
)

aapl_valuation["market_cap"] = (
    aapl_valuation["close_price"]
    * aapl_valuation["shares_outstanding"]
)

aapl_valuation["enterprise_value"] = (
    aapl_valuation["market_cap"]
    + aapl_valuation["total_debt"]
    - aapl_valuation["cash"]
)

aapl_valuation["ev_sales"] = (
    aapl_valuation["enterprise_value"]
    / aapl_valuation["revenue"]
)

aapl_valuation["price_sales"] = (
    aapl_valuation["market_cap"]
    / aapl_valuation["revenue"]
)

print(
    aapl_valuation[
        [
            "fiscal_year",
            "market_cap",
            "enterprise_value",
            "ev_sales",
            "price_sales",
        ]
    ]
)

   fiscal_year    market_cap      enterprise_value  ev_sales price_sales
0         2021  2.376302e+12  2460081481397.979492  6.724897    6.495878
1         2022  2.201365e+12  2287806446516.357422  5.801785    5.582575
2         2023  2.688760e+12  2763897841701.171875  7.211078    7.015041
3         2024  3.552218e+12       3618937405000.0  9.254766    9.084144
4         2025  3.727926e+12   3782669999163.52832  9.089439    8.957894


In [ ]:
def build_company_valuation_table(
    ticker,
    cik,
    accounting_data
):
    company_facts = get_company_facts(cik)

    shares = extract_shares_outstanding(
        company_facts,
        ticker
    )

    shares = adjust_shares_for_splits(
        shares,
        ticker
    )

    prices = get_price_history(ticker)

    market = match_prices_to_shares_dates(
        shares,
        prices,
        ticker
    )

    market = market.merge(
        shares[
            [
                "fiscal_year",
                "split_factor",
                "adjusted_shares",
                "shares_source",
            ]
        ],
        on="fiscal_year",
        how="left"
    )

    accounting = accounting_data[
        accounting_data["ticker"] == ticker
    ][
        [
            "fiscal_year",
            "revenue",
            "cash",
            "total_debt",
        ]
    ].copy()

    result = market.merge(
        accounting,
        on="fiscal_year",
        how="left"
    )

    result["market_cap"] = (
        result["close_price"]
        * result["adjusted_shares"]
    )

    result["enterprise_value"] = (
        result["market_cap"]
        + result["total_debt"]
        - result["cash"]
    )

    result["ev_sales"] = (
        result["enterprise_value"]
        / result["revenue"]
    )

    result["price_sales"] = (
        result["market_cap"]
        / result["revenue"]
    )

    result["ticker"] = ticker

    return result

In [ ]:
msft_row = company_universe[
    company_universe["ticker"] == "MSFT"
].iloc[0]

msft_valuation = build_company_valuation_table(
    "MSFT",
    msft_row["cik"],
    analysis_data
)

print(
    msft_valuation[
        [
            "ticker",
            "fiscal_year",
            "shares_date",
            "close_price",
            "market_cap",
            "enterprise_value",
            "ev_sales",
            "price_sales",
        ]
    ]
)

  ticker  fiscal_year shares_date  close_price    market_cap  \
0   MSFT         2021  2021-07-26   289.049988  2.172179e+12   
1   MSFT         2022  2022-07-25   258.829987  1.930326e+12   
2   MSFT         2023  2023-07-24   345.109985  2.564086e+12   
3   MSFT         2024  2024-07-25   418.399994  3.109983e+12   
4   MSFT         2025  2025-07-24   510.880005  3.797456e+12   

       enterprise_value   ev_sales price_sales  
0  2216101223499.887695  13.184173   12.922869  
1  1966176053087.168945   9.916659    9.735845  
2  2576618649264.990234  12.158737   12.099595  
3  3136605213242.733887  12.796098   12.687491  
4  3810365075998.277832   13.52517   13.479349  


In [ ]:
nvda_row = company_universe[
    company_universe["ticker"] == "NVDA"
].iloc[0]

nvda_valuation = build_company_valuation_table(
    "NVDA",
    nvda_row["cik"],
    analysis_data
)

print(
    nvda_valuation[
        [
            "ticker",
            "fiscal_year",
            "shares_date",
            "close_price",
            "market_cap",
            "enterprise_value",
            "ev_sales",
            "price_sales",
        ]
    ]
)

  ticker  fiscal_year shares_date  close_price    market_cap  \
0   NVDA         2021  2021-02-19    14.926500  9.254430e+09   
1   NVDA         2022  2022-03-11    22.100000  5.547100e+10   
2   NVDA         2023  2023-02-17    21.388000  5.282836e+10   
3   NVDA         2024  2024-02-16    72.612999  1.815325e+11   
4   NVDA         2025  2025-02-21   134.429993  3.280092e+12   

      enterprise_value   ev_sales price_sales  
0   15370430198.669434   0.921765    0.554988  
1   64427000957.489014    2.39381    2.061046  
2   60392361206.054688    2.23891    1.958492  
3  183961497406.005859   3.019623    2.979753  
4   3279965821289.0625  25.134416   25.135381  


In [ ]:
nvda_ticker = yf.Ticker("NVDA")

nvda_splits = nvda_ticker.splits

print(nvda_splits)

Date
2000-06-27 09:30:00-04:00     2.0
2001-09-12 09:30:00-04:00     2.0
2006-04-07 09:30:00-04:00     2.0
2007-09-11 09:30:00-04:00     1.5
2021-07-20 09:30:00-04:00     4.0
2024-06-10 09:30:00-04:00    10.0
Name: Stock Splits, dtype: float64


In [ ]:
def adjust_shares_for_splits(
    shares_df,
    ticker
):
    result = shares_df.copy()

    splits = yf.Ticker(ticker).splits.copy()

    if len(splits) == 0:
        result["split_factor"] = 1.0
        result["adjusted_shares"] = (
            result["shares_outstanding"]
        )
        return result

    # Remove timezone so dates can be compared cleanly
    splits.index = (
        pd.to_datetime(splits.index)
        .tz_localize(None)
    )

    split_factors = []

    for shares_date in result["shares_date"]:
        shares_date = pd.to_datetime(shares_date)

        # Yahoo historical prices reflect splits occurring
        # after the historical shares date
        later_splits = splits[
            splits.index > shares_date
        ]

        if len(later_splits) == 0:
            factor = 1.0
        else:
            factor = later_splits.prod()

        split_factors.append(factor)

    result["split_factor"] = split_factors

    result["adjusted_shares"] = (
        result["shares_outstanding"]
        * result["split_factor"]
    )

    return result

In [ ]:
nvda_shares = extract_shares_outstanding(
    nvda_facts
)

nvda_adjusted_shares = adjust_shares_for_splits(
    nvda_shares,
    "NVDA"
)

print(
    nvda_adjusted_shares[
        [
            "fiscal_year",
            "shares_date",
            "shares_outstanding",
            "split_factor",
            "adjusted_shares",
        ]
    ]
)

   fiscal_year shares_date  shares_outstanding  split_factor  adjusted_shares
0         2021  2021-02-19           620000000          40.0     2.480000e+10
1         2022  2022-03-11          2510000000          10.0     2.510000e+10
2         2023  2023-02-17          2470000000          10.0     2.470000e+10
3         2024  2024-02-16          2500000000          10.0     2.500000e+10
4         2025  2025-02-21         24400000000           1.0     2.440000e+10


In [ ]:
nvda_valuation = build_company_valuation_table(
    "NVDA",
    nvda_row["cik"],
    analysis_data
)

print(
    nvda_valuation[
        [
            "ticker",
            "fiscal_year",
            "split_factor",
            "adjusted_shares",
            "close_price",
            "market_cap",
            "enterprise_value",
            "ev_sales",
            "price_sales",
        ]
    ]
)

  ticker  fiscal_year  split_factor  adjusted_shares  close_price  \
0   NVDA         2021          40.0     2.480000e+10    14.926500   
1   NVDA         2022          10.0     2.510000e+10    22.100000   
2   NVDA         2023          10.0     2.470000e+10    21.388000   
3   NVDA         2024          10.0     2.500000e+10    72.612999   
4   NVDA         2025           1.0     2.440000e+10   134.429993   

     market_cap      enterprise_value   ev_sales price_sales  
0  3.701772e+11   376293207946.777344  22.566309   22.199533  
1  5.547100e+11   563666009574.890137  20.943227   20.610463  
2  5.282836e+11   535847612060.546875  19.865337   19.584919  
3  1.815325e+12  1817753974060.058594  29.837398   29.797528  
4  3.280092e+12    3279965821289.0625  25.134416   25.135381  


In [ ]:
test_tickers = [
    "AAPL",
    "MSFT",
    "NVDA",
    "ADBE",
    "META",
]

valuation_test_tables = []
valuation_test_failures = []

for ticker in test_tickers:

    row = company_universe[
        company_universe["ticker"] == ticker
    ].iloc[0]

    try:
        table = build_company_valuation_table(
            ticker,
            row["cik"],
            analysis_data
        )

        valuation_test_tables.append(table)

        print(
            f"OK: {ticker} - {len(table)} rows"
        )

    except Exception as e:
        valuation_test_failures.append({
            "ticker": ticker,
            "error": str(e)
        })

        print(
            f"FAILED: {ticker} - {e}"
        )

OK: AAPL - 5 rows
OK: MSFT - 5 rows
OK: NVDA - 5 rows
OK: ADBE - 5 rows
FAILED: META - 'EntityCommonStockSharesOutstanding'


In [ ]:
meta_row = company_universe[
    company_universe["ticker"] == "META"
].iloc[0]

meta_facts = get_company_facts(
    meta_row["cik"]
)

dei_keys = list(
    meta_facts["facts"]["dei"].keys()
)

for key in dei_keys:
    if "Share" in key or "Stock" in key:
        print(key)

In [ ]:
for namespace, facts in meta_facts["facts"].items():
    for key in facts.keys():
        key_lower = key.lower()

        if (
            "share" in key_lower
            or "stock" in key_lower
            or "common" in key_lower
        ):
            print(namespace, "->", key)

us-gaap -> AdjustmentsRelatedToTaxWithholdingForShareBasedCompensation
us-gaap -> AdjustmentsToAdditionalPaidInCapitalSharebasedCompensationRequisiteServicePeriodRecognitionValue
us-gaap -> AdjustmentsToAdditionalPaidInCapitalTaxEffectFromShareBasedCompensation
us-gaap -> AdjustmentToAdditionalPaidInCapitalIncomeTaxEffectFromShareBasedCompensationNet
us-gaap -> AllocatedShareBasedCompensationExpense
us-gaap -> BusinessAcquisitionEquityInterestsIssuedOrIssuableNumberOfSharesIssued
us-gaap -> CommonStockParOrStatedValuePerShare
us-gaap -> CommonStocksIncludingAdditionalPaidInCapital
us-gaap -> CommonStockValue
us-gaap -> ConversionOfStockSharesConverted1
us-gaap -> DeferredTaxAssetsTaxDeferredExpenseCompensationAndBenefitsShareBasedCompensationCost
us-gaap -> EarningsPerShareBasic
us-gaap -> EarningsPerShareDiluted
us-gaap -> EffectiveIncomeTaxRateReconciliationNondeductibleExpenseShareBasedCompensationCost
us-gaap -> EmployeeServiceShareBasedCompensationNonvestedAwardsTotalCompensationC

In [ ]:
for namespace, facts in meta_facts["facts"].items():
    for key in facts.keys():
        key_lower = key.lower()

        if (
            "outstanding" in key_lower
            or "sharesoutstanding" in key_lower
        ):
            print(namespace, "->", key)

us-gaap -> LineOfCreditFacilityAmountOutstanding
us-gaap -> PreferredStockSharesOutstanding
us-gaap -> ShareBasedCompensationArrangementByShareBasedPaymentAwardOptionsOutstandingIntrinsicValue
us-gaap -> ShareBasedCompensationArrangementByShareBasedPaymentAwardOptionsOutstandingNumber
us-gaap -> ShareBasedCompensationArrangementByShareBasedPaymentAwardOptionsOutstandingWeightedAverageExercisePrice
us-gaap -> ShareBasedCompensationArrangementByShareBasedPaymentAwardOptionsVestedAndExpectedToVestOutstandingAggregateIntrinsicValue
us-gaap -> ShareBasedCompensationArrangementByShareBasedPaymentAwardOptionsVestedAndExpectedToVestOutstandingNumber
us-gaap -> ShareBasedCompensationArrangementByShareBasedPaymentAwardOptionsVestedAndExpectedToVestOutstandingWeightedAverageExercisePrice
us-gaap -> SharebasedCompensationSharesAuthorizedUnderStockOptionPlansExercisePriceRangeOutstandingOptionsWeightedAverageExercisePriceBeginningBalance1
us-gaap -> WeightedAverageNumberOfDilutedSharesOutstanding
u

In [ ]:
meta_yf = yf.Ticker("META")

meta_share_history = meta_yf.get_shares_full(
    start="2021-01-01",
    end="2026-01-01"
)

print(meta_share_history)

Failed to get ticker 'META' reason: Failed to perform, curl: (28) Resolving timed out after 10014 milliseconds. See https://curl.se/libcurl/c/libcurl-errors.html first for more details.


2022-06-09 00:00:00-04:00    2706319872
2022-07-30 00:00:00-04:00    2687549952
2022-08-01 00:00:00-04:00    2687549952
2022-08-03 00:00:00-04:00    2683190016
2022-08-05 00:00:00-04:00    2659660032
                                ...    
2025-11-11 00:00:00-05:00    2520527597
2025-11-12 00:00:00-05:00    2520527597
2025-11-14 00:00:00-05:00    2520527597
2025-11-15 00:00:00-05:00    2522305937
2025-11-17 00:00:00-05:00    2520527597
Length: 535, dtype: int64


In [ ]:
def get_yfinance_historical_shares(
    ticker,
    fiscal_years
):
    history = yf.Ticker(ticker).get_shares_full(
        start="2021-01-01",
        end="2026-12-31"
    )

    if history is None or len(history) == 0:
        return pd.DataFrame()

    history = history.copy()
    history.index = pd.to_datetime(
        history.index
    ).tz_localize(None)

    rows = []

    for year in fiscal_years:
        year_history = history[
            history.index.year == year
        ]

        if len(year_history) == 0:
            continue

        # Use latest available observation in the year
        shares_date = year_history.index[-1]
        shares = year_history.iloc[-1]

        rows.append({
            "fiscal_year": year,
            "shares_date": shares_date,
            "shares_outstanding": shares,
            "shares_source": "yfinance"
        })

    return pd.DataFrame(rows)

In [ ]:
meta_yf_shares = get_yfinance_historical_shares(
    "META",
    [2021, 2022, 2023, 2024, 2025]
)

print(meta_yf_shares)

   fiscal_year shares_date  shares_outstanding shares_source
0         2022  2022-12-16          2658200064      yfinance
1         2023  2023-12-30          2569860096      yfinance
2         2024  2024-12-25          2524489984      yfinance
3         2025  2025-11-17          2520527597      yfinance


In [ ]:
def get_yfinance_historical_shares(
    ticker,
    fiscal_years
):
    history = yf.Ticker(ticker).get_shares_full(
        start="2021-01-01",
        end="2026-12-31"
    )

    if history is None or len(history) == 0:
        return pd.DataFrame()

    history = history.copy()

    history.index = pd.to_datetime(
        history.index
    )

    if history.index.tz is not None:
        history.index = history.index.tz_localize(None)

    rows = []

    for year in fiscal_years:
        year_history = history[
            history.index.year == year
        ]

        if len(year_history) == 0:
            continue

        shares_date = year_history.index[-1]
        shares = year_history.iloc[-1]

        rows.append({
            "fiscal_year": year,
            "shares_date": shares_date,
            "shares_outstanding": shares,
            "shares_source": "yfinance"
        })

    return pd.DataFrame(rows)

In [ ]:
def extract_shares_outstanding(
    company_facts,
    ticker,
    start_year=2021,
    end_year=2025
):
    try:
        shares = extract_sec_shares_outstanding(
            company_facts,
            start_year,
            end_year
        )

        if len(shares) > 0:
            shares["shares_source"] = "SEC"
            return shares

    except KeyError:
        pass

    shares = get_yfinance_historical_shares(
        ticker,
        range(start_year, end_year + 1)
    )

    return shares

In [ ]:
def adjust_shares_for_splits(
    shares_df,
    ticker
):
    result = shares_df.copy()

    splits = yf.Ticker(ticker).splits.copy()

    if len(splits) == 0:
        result["split_factor"] = 1.0
        result["adjusted_shares"] = (
            result["shares_outstanding"]
        )
        return result

    splits.index = pd.to_datetime(
        splits.index
    )

    if splits.index.tz is not None:
        splits.index = splits.index.tz_localize(None)

    split_factors = []

    for shares_date in result["shares_date"]:
        shares_date = pd.to_datetime(shares_date)

        later_splits = splits[
            splits.index > shares_date
        ]

        if len(later_splits) == 0:
            factor = 1.0
        else:
            factor = later_splits.prod()

        split_factors.append(factor)

    result["split_factor"] = split_factors

    result["adjusted_shares"] = (
        result["shares_outstanding"]
        * result["split_factor"]
    )

    return result

In [ ]:
meta_valuation = build_company_valuation_table(
    "META",
    meta_row["cik"],
    analysis_data
)

print(
    meta_valuation[
        [
            "ticker",
            "fiscal_year",
            "shares_date",
            "shares_source",
            "close_price",
            "market_cap",
            "ev_sales",
            "price_sales",
        ]
    ]
)

  ticker  fiscal_year shares_date shares_source  close_price    market_cap  \
0   META         2022  2022-12-16      yfinance   119.430000  3.174688e+11   
1   META         2023  2023-12-30      yfinance   353.959991  9.096277e+11   
2   META         2024  2024-12-25      yfinance   607.750000  1.534259e+12   
3   META         2025  2025-11-17      yfinance   602.010010  1.517383e+12   

   ev_sales price_sales  
0  2.681704    2.722507  
1  6.568847    6.742877  
2  9.235177    9.326744  
3  7.664251    7.550446  


In [ ]:
test_tickers = [
    "AAPL",
    "MSFT",
    "NVDA",
    "ADBE",
    "META",
]

valuation_test_tables = []
valuation_test_failures = []

for ticker in test_tickers:

    row = company_universe[
        company_universe["ticker"] == ticker
    ].iloc[0]

    try:
        table = build_company_valuation_table(
            ticker,
            row["cik"],
            analysis_data
        )

        valuation_test_tables.append(table)

        print(
            f"OK: {ticker} - {len(table)} rows"
        )

    except Exception as e:
        valuation_test_failures.append({
            "ticker": ticker,
            "error": str(e)
        })

        print(
            f"FAILED: {ticker} - {e}"
        )

OK: AAPL - 5 rows
OK: MSFT - 5 rows
OK: NVDA - 5 rows
OK: ADBE - 5 rows
OK: META - 4 rows


In [ ]:
valuation_test = pd.concat(
    valuation_test_tables,
    ignore_index=True
)

print("\nTotal rows:", len(valuation_test))

print("\nRows by ticker:")
print(
    valuation_test
    .groupby("ticker")
    .size()
)

print("\nShares sources:")
print(
    valuation_test["shares_source"]
    .value_counts()
)

print("\nMissing values:")
print(
    valuation_test[
        [
            "adjusted_shares",
            "close_price",
            "market_cap",
            "enterprise_value",
            "ev_sales",
            "price_sales",
        ]
    ].isna().sum()
)

print("\nFailures:")
print(valuation_test_failures)


Total rows: 24

Rows by ticker:
ticker
AAPL    5
ADBE    5
META    4
MSFT    5
NVDA    5
dtype: int64

Shares sources:
shares_source
SEC         20
yfinance     4
Name: count, dtype: int64

Missing values:
adjusted_shares     0
close_price         0
market_cap          0
enterprise_value    0
ev_sales            0
price_sales         0
dtype: int64

Failures:
[]


In [ ]:
full_valuation_tables = []
full_valuation_failures = []

for i, row in company_universe.iterrows():

    ticker = row["ticker"]
    cik = row["cik"]

    try:
        table = build_company_valuation_table(
            ticker,
            cik,
            analysis_data
        )

        table["company"] = row["company"]
        table["sector"] = row["sector"]
        table["industry"] = row["industry"]

        full_valuation_tables.append(table)

        print(
            f"{i + 1}/{len(company_universe)} "
            f"OK: {ticker} - {len(table)} rows"
        )

    except Exception as e:
        full_valuation_failures.append({
            "ticker": ticker,
            "cik": cik,
            "error": str(e)
        })

        print(
            f"{i + 1}/{len(company_universe)} "
            f"FAILED: {ticker} - {e}"
        )

1/97 OK: ACN - 5 rows
2/97 OK: ADBE - 5 rows
3/97 OK: AMD - 5 rows
4/97 OK: AKAM - 5 rows
5/97 OK: GOOGL - 5 rows
6/97 OK: GOOG - 5 rows
7/97 OK: APH - 5 rows
8/97 OK: ADI - 5 rows
9/97 OK: AAPL - 5 rows
10/97 OK: AMAT - 5 rows
11/97 OK: APP - 5 rows
12/97 OK: ANET - 5 rows
13/97 OK: T - 5 rows
14/97 OK: ADSK - 5 rows
15/97 OK: AVGO - 5 rows
16/97 OK: CDNS - 5 rows
17/97 OK: CDW - 5 rows
18/97 OK: CHTR - 5 rows
19/97 OK: CIEN - 5 rows
20/97 OK: CSCO - 5 rows
21/97 OK: CTSH - 5 rows


$COHR: possibly delisted; no price data found  (1d 1927-12-04 -> 2026-11-09)


22/97 OK: COHR - 4 rows
23/97 OK: CMCSA - 5 rows
24/97 OK: GLW - 5 rows
25/97 OK: CRWD - 5 rows
26/97 OK: DDOG - 5 rows
27/97 OK: DELL - 5 rows
28/97 OK: ECHO - 1 rows
29/97 OK: P - 5 rows
30/97 OK: FFIV - 5 rows
31/97 OK: FICO - 5 rows
32/97 OK: FSLR - 5 rows
33/97 OK: FLEX - 5 rows
34/97 OK: FTNT - 5 rows
35/97 OK: FOXA - 5 rows
36/97 OK: FOX - 5 rows
37/97 OK: IT - 5 rows
38/97 OK: GEN - 5 rows
39/97 OK: GDDY - 2 rows
40/97 OK: HPE - 5 rows
41/97 OK: HPQ - 5 rows
42/97 OK: IBM - 5 rows
43/97 OK: INTC - 5 rows
44/97 OK: INTU - 5 rows
45/97 OK: JBL - 5 rows
46/97 OK: KEYS - 5 rows
47/97 OK: KLAC - 5 rows
48/97 OK: LRCX - 5 rows
49/97 OK: LYV - 5 rows
50/97 OK: LITE - 5 rows
51/97 OK: MRVL - 4 rows
52/97 OK: META - 4 rows
53/97 OK: MCHP - 5 rows
54/97 OK: MU - 5 rows
55/97 OK: MSFT - 5 rows
56/97 OK: MPWR - 5 rows
57/97 OK: MSI - 5 rows
58/97 OK: NTAP - 5 rows
59/97 OK: NFLX - 5 rows
60/97 OK: NWSA - 5 rows
61/97 OK: NWS - 5 rows
62/97 OK: NVDA - 5 rows
63/97 OK: NXPI - 5 rows
64/97 OK

In [ ]:
full_valuation_data = pd.concat(
    full_valuation_tables,
    ignore_index=True
)

full_valuation_failures_df = pd.DataFrame(
    full_valuation_failures
)

In [ ]:
print(
    "Successful companies:",
    full_valuation_data["ticker"].nunique()
)

print(
    "Total valuation rows:",
    len(full_valuation_data)
)

print(
    "Failures:",
    len(full_valuation_failures_df)
)

print("\nRows by shares source:")
print(
    full_valuation_data[
        "shares_source"
    ].value_counts()
)

print("\nMissing values:")
print(
    full_valuation_data[
        [
            "adjusted_shares",
            "close_price",
            "market_cap",
            "enterprise_value",
            "ev_sales",
            "price_sales",
        ]
    ].isna().sum()
)

print("\nFailures:")
print(full_valuation_failures_df)

Successful companies: 96
Total valuation rows: 454
Failures: 1

Rows by shares source:
shares_source
SEC         369
yfinance     85
Name: count, dtype: int64

Missing values:
adjusted_shares       0
close_price           0
market_cap            0
enterprise_value    106
ev_sales            113
price_sales          14
dtype: int64

Failures:
  ticker      cik                 error
0   SKYD  2041610  'shares_outstanding'


In [ ]:
skyd_row = company_universe[
    company_universe["ticker"] == "SKYD"
].iloc[0]

skyd_facts = get_company_facts(
    skyd_row["cik"]
)

print("DEI tags:")
print(
    list(
        skyd_facts["facts"]
        .get("dei", {})
        .keys()
    )
)

print("\nYahoo shares:")
skyd_yf_shares = get_yfinance_historical_shares(
    "SKYD",
    range(2021, 2026)
)

print(skyd_yf_shares)

DEI tags:
['EntityCommonStockSharesOutstanding']

Yahoo shares:
Empty DataFrame
Columns: []
Index: []


In [ ]:
skyd_share_fact = (
    skyd_facts["facts"]["dei"]
    ["EntityCommonStockSharesOutstanding"]
)

print("Units:")
print(skyd_share_fact["units"].keys())

for unit_name, observations in skyd_share_fact["units"].items():
    print("\nUNIT:", unit_name)

    temp = pd.DataFrame(observations)

    cols = [
        c for c in
        [
            "end",
            "val",
            "form",
            "fy",
            "fp",
            "filed",
            "accn",
        ]
        if c in temp.columns
    ]

    print(temp[cols].tail(20))

Units:
dict_keys(['shares'])

UNIT: shares
          end         val  form    fy  fp       filed                  accn
0  2025-05-13        1000  10-Q  2025  Q1  2025-05-14  0002041610-25-000010
1  2025-07-31        1000  10-Q  2025  Q2  2025-08-01  0002041610-25-000013
2  2025-11-05  1071666977  10-Q  2025  Q3  2025-11-10  0002041610-25-000044


In [ ]:
full_valuation_data.to_csv(
    "../data/raw/full_valuation_data_2021_2025.csv",
    index=False
)

full_valuation_failures_df.to_csv(
    "../data/raw/full_valuation_failures.csv",
    index=False
)

print("Saved valuation checkpoint.")

Saved valuation checkpoint.


Valuation data checkpoint

- 96 of 97 companies produced valuation observations.
- SKYD was excluded because SEC Company Facts did not provide usable 10-K
  period-end common shares outstanding for 2021–2025, and yfinance did not
  provide historical shares data.
- Primary shares source: SEC EntityCommonStockSharesOutstanding.
- Fallback shares source: yfinance historical shares.
- Historical shares were adjusted for subsequent stock splits before
  multiplying by split-adjusted historical closing prices.
- Missing EV/EV-Sales values were retained as missing rather than filled
  because debt, cash, or revenue data may be unavailable.